# IoMT Anomaly Detection — Hybrid Supervised-Unsupervised Framework

**Author**: Amro — M.Sc. AI & ML in Cybersecurity, Sakarya University of Applied Sciences
**Dataset**: CICIoMT2024 (WiFi + MQTT combined, ≈8.7 M network flows, 18 classes, 45 features)

---

## Framework Overview

This notebook implements a full 6-phase pipeline for detecting anomalies and
zero-day attacks in Internet of Medical Things (IoMT) networks.

| Phase | Script | Description |
|-------|--------|-------------|
| 2 | `ciciomt2024_eda.py` | Exploratory Data Analysis |
| 3 | `preprocessing_pipeline.py` | Preprocessing & Feature Engineering |
| 4 | `supervised_training.py` | Supervised Layer (RF + XGBoost) |
| 5 | `unsupervised_training.py` | Unsupervised Layer (Autoencoder + Isolation Forest) |
| 6 | `fusion_engine.py` | Fusion Engine & Zero-Day Simulation |

## Hypotheses

| | Hypothesis | Test |
|-|-----------|------|
| **H1** | Fusion improves 19-class macro-F1 over E7 alone | Paired bootstrap (200 iters), 95% CI |
| **H2** | AE catches ≥70% of attacks E7 misclassifies as benign, on ≥50% of targets | Recall on E7-missed samples |

## Dataset: CICIoMT2024

| Property | Value |
|----------|-------|
| Total rows | ≈8.7 M (train 7.16 M + test 1.61 M) |
| Feature space | 45 network-flow features |
| Attack classes | 17 distinct types + 1 benign |
| Max imbalance ratio | ≈2 211 : 1 (Benign vs Recon_Ping_Sweep) |

Attack families: DDoS (4), DoS (4), Reconnaissance (4), MQTT (5), ARP Spoofing (1)


In [ ]:
# ── Installation ────────────────────────────────────────────────────────────
# Uncomment the lines below if packages are not already installed.

# !pip install numpy pandas matplotlib seaborn scikit-learn imbalanced-learn
# !pip install xgboost joblib pyarrow fastparquet
# !pip install tensorflow               # Autoencoder (Phase 5)
# !pip install tensorflow-metal         # Apple Silicon GPU (Phase 5)
# !pip install tabulate                 # Markdown tables in Phase 6

import sys
print(f"Python : {sys.version}")
try:
    import numpy as np;   print(f"NumPy  : {np.__version__}")
    import pandas as pd;  print(f"Pandas : {pd.__version__}")
    import sklearn;       print(f"sklearn: {sklearn.__version__}")
    import xgboost;       print(f"XGBoost: {xgboost.__version__}")
except ImportError as e:
    print(f"Missing: {e}")


## Phase 2 — Exploratory Data Analysis

Performs a full, publication-quality EDA on the CICIoMT2024 dataset
(≈7.16 M train rows, ≈1.61 M test rows, 18 classes, 45 features).

**Key outputs** (written to `eda_output/`):
- `findings.md` — consolidated preprocessing recommendations
- `figures/` — 15+ charts (distributions, correlation, PCA, t-SNE)
- `quality_*.csv` — per-column missing/inf/unique audit
- `imbalance_table.csv` — class counts and ratios
- `high_correlation_pairs.csv` — |r| > 0.85 feature pairs
- `feature_target_cohens_d.csv` — |Cohen's d| ranking (attack vs benign)
- `benign_profile.csv` — Autoencoder reference statistics
- `train_cleaned.csv`, `test_cleaned.csv` — deduped, NaN-filled data

**Prerequisite**: CSV files in `./data/train/` and `./data/test/`.

In [ ]:
# -*- coding: utf-8 -*-
"""
CICIoMT2024 — Phase 2 Exploratory Data Analysis
================================================
Thesis: A Hybrid Supervised-Unsupervised Framework for Anomaly Detection
        and Zero-Day Attack Identification in IoMT Networks.

This script performs a complete, publication-quality EDA on the real
CICIoMT2024 WiFi_and_MQTT/attacks/csv data (train: 7.16 M rows,
test: 1.61 M rows).

Structure (run top-to-bottom or cell-by-cell):
  SECTION 1  — Configuration & Data Loading
  SECTION 2  — Data Quality Checks
  SECTION 3  — Class Distribution Analysis
  SECTION 4  — Feature Statistics
  SECTION 5  — Feature Distribution Analysis
  SECTION 6  — Correlation Analysis
  SECTION 7  — Attack-Specific Analysis
  SECTION 8  — Dimensionality Reduction
  SECTION 9  — Outlier Detection Preview
  SECTION 10 — Summary & Export

Compatible with Jupyter, VS Code Interactive, and plain `python` execution.
"""

In [ ]:
# %% [markdown]
# # CICIoMT2024 EDA — Full Pipeline
# *Amro — M.Sc. AI & ML in Cybersecurity — Sakarya University*

In [ ]:
# %% SECTION 1.0 — Imports & environment
from __future__ import annotations

import glob
import os
import re
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

print("=" * 70)
print("CICIoMT2024 EDA — environment")
print("=" * 70)
print(f"Python       : {sys.version.split()[0]}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Matplotlib   : {plt.matplotlib.__version__}")
print(f"Seaborn      : {sns.__version__}")

# Plot defaults — publication quality
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 8),
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "savefig.bbox": "tight",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "axes.titleweight": "bold",
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

In [ ]:
# %% SECTION 1.1 — Configuration (EDIT THESE PATHS FOR YOUR MACHINE)
# =============================================================================
TRAIN_DIR  = "./data/train/"
TEST_DIR   = "./data/test/"
OUTPUT_DIR = "./eda_output/"
SAMPLE_SIZE  = 300_000       # subsample for heavy visualisations
RANDOM_STATE = 42
# =============================================================================

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
FIG_DIR = Path(OUTPUT_DIR) / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = Path(OUTPUT_DIR) / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CACHE = CACHE_DIR / "train_deduped.parquet"
TEST_CACHE  = CACHE_DIR / "test_deduped.parquet"

# Category color palette (used everywhere)
CATEGORY_COLORS: Dict[str, str] = {
    "DDoS":     "#E24B4A",
    "DoS":      "#D85A30",
    "Recon":    "#EF9F27",
    "MQTT":     "#7F77DD",
    "Spoofing": "#D4537E",
    "Benign":   "#1D9E75",
}

FEATURES = [
    "Header_Length", "Protocol Type", "Duration", "Rate", "Srate", "Drate",
    "fin_flag_number", "syn_flag_number", "rst_flag_number", "psh_flag_number",
    "ack_flag_number", "ece_flag_number", "cwr_flag_number",
    "ack_count", "syn_count", "fin_count", "rst_count",
    "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC",
    "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv", "LLC",
    "Tot sum", "Min", "Max", "AVG", "Std", "Tot size", "IAT",
    "Number", "Magnitue", "Radius", "Covariance", "Variance", "Weight",
]

TOP10_FEATURES = [
    "IAT", "Rate", "Srate", "Header_Length",
    "syn_flag_number", "rst_count", "ack_flag_number",
    "psh_flag_number", "Tot sum", "UDP",
]

def save_fig(fig: plt.Figure, name: str) -> None:
    out = FIG_DIR / f"{name}.png"
    fig.savefig(out, dpi=160, bbox_inches="tight")
    print(f"  saved → {out}")

In [ ]:
# %% SECTION 1.2 — Filename → label mapping
def filename_to_label(filename: str) -> str:
    """Derive the attack label from a CICIoMT2024 CSV filename."""
    name = os.path.basename(filename)
    name = name.replace("_train.pcap.csv", "").replace("_test.pcap.csv", "")
    # Strip trailing digits (ICMP3 → ICMP, UDP8 → UDP, …)
    name = re.sub(r"(\d+)$", "", name)

    mapping = {
        "ARP_Spoofing":              "ARP_Spoofing",
        "Benign":                    "Benign",
        "MQTT-DDoS-Connect_Flood":   "MQTT_DDoS_Connect_Flood",
        "MQTT-DDoS-Publish_Flood":   "MQTT_DDoS_Publish_Flood",
        "MQTT-DoS-Connect_Flood":    "MQTT_DoS_Connect_Flood",
        "MQTT-DoS-Publish_Flood":    "MQTT_DoS_Publish_Flood",
        "MQTT-Malformed_Data":       "MQTT_Malformed_Data",
        "Recon-OS_Scan":             "Recon_OS_Scan",
        "Recon-Ping_Sweep":          "Recon_Ping_Sweep",
        "Recon-Port_Scan":           "Recon_Port_Scan",
        "Recon-VulScan":             "Recon_VulScan",
        "TCP_IP-DDoS-ICMP":          "DDoS_ICMP",
        "TCP_IP-DDoS-SYN":           "DDoS_SYN",
        "TCP_IP-DDoS-TCP":           "DDoS_TCP",
        "TCP_IP-DDoS-UDP":           "DDoS_UDP",
        "TCP_IP-DoS-ICMP":           "DoS_ICMP",
        "TCP_IP-DoS-SYN":            "DoS_SYN",
        "TCP_IP-DoS-TCP":            "DoS_TCP",
        "TCP_IP-DoS-UDP":            "DoS_UDP",
    }
    return mapping.get(name, name)


def label_to_category(label: str) -> str:
    if label == "Benign":            return "Benign"
    if label.startswith("DDoS"):     return "DDoS"
    if label.startswith("DoS"):      return "DoS"
    if label.startswith("Recon"):    return "Recon"
    if label.startswith("MQTT"):     return "MQTT"
    if label.startswith("ARP"):      return "Spoofing"
    return "Unknown"

In [ ]:
# %% SECTION 1.3 — Load all CSVs with memory-efficient dtype
def load_split(directory: str, split_name: str) -> pd.DataFrame:
    """Load every CSV in `directory`, tag with label/category/split."""
    csv_files = sorted(glob.glob(os.path.join(directory, "*.csv")))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found under {directory}")
    print(f"\n[{split_name}] found {len(csv_files)} CSV files in {directory}")

    dtype_map = {f: np.float32 for f in FEATURES}

    frames: List[pd.DataFrame] = []
    for i, path in enumerate(csv_files, 1):
        label = filename_to_label(path)
        try:
            df = pd.read_csv(path, dtype=dtype_map, engine="c",
                             low_memory=False)
        except Exception as e:
            print(f"  [{i:02d}/{len(csv_files)}] FAILED {os.path.basename(path)}: {e}")
            continue

        # Keep only known feature columns (schema guard)
        df = df[[c for c in FEATURES if c in df.columns]].copy()
        df["label"]    = label
        df["category"] = label_to_category(label)
        df["split"]    = split_name
        frames.append(df)
        print(f"  [{i:02d}/{len(csv_files)}] {os.path.basename(path):<50s} "
              f"→ {label:<26s} rows={len(df):>9,}")

    full = pd.concat(frames, ignore_index=True, copy=False)
    print(f"[{split_name}] TOTAL shape={full.shape}, "
          f"memory={full.memory_usage(deep=True).sum() / 1e6:,.1f} MB")
    return full


print("\n" + "=" * 70 + "\nSECTION 1 — LOADING DATA\n" + "=" * 70)
CACHE_HIT = False
if TRAIN_CACHE.exists() and TEST_CACHE.exists():
    print(f"Parquet cache hit — skipping CSV reload.")
    print(f"  train ← {TRAIN_CACHE}")
    print(f"  test  ← {TEST_CACHE}")
    try:
        df_train = pd.read_parquet(TRAIN_CACHE)
        df_test  = pd.read_parquet(TEST_CACHE)
        DATA_LOADED = True
        CACHE_HIT   = True
        print(f"  train rows={len(df_train):,}   test rows={len(df_test):,}")
    except Exception as e:
        print(f"  cache read failed ({e}) — falling back to CSV load")
        CACHE_HIT = False

if not CACHE_HIT:
    try:
        df_train = load_split(TRAIN_DIR, "train")
        df_test  = load_split(TEST_DIR,  "test")
        DATA_LOADED = True
    except Exception as e:
        print(f"!! Data loading failed — continuing with empty frames. Error: {e}")
        df_train = pd.DataFrame(columns=FEATURES + ["label", "category", "split"])
        df_test  = pd.DataFrame(columns=FEATURES + ["label", "category", "split"])
        DATA_LOADED = False

In [ ]:
# %% SECTION 1.4 — Quick-look
if DATA_LOADED:
    print("\n--- df_train.info() ---")
    df_train.info(memory_usage="deep")
    print("\n--- df_train.head() ---")
    print(df_train.head())

In [ ]:
# %% SECTION 2 — Data Quality Checks
print("\n" + "=" * 70 + "\nSECTION 2 — DATA QUALITY\n" + "=" * 70)

def quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Return and print a per-column quality summary for numeric features."""
    feats = [c for c in FEATURES if c in df.columns]
    numeric = df[feats]

    rows = []
    for col in feats:
        s = numeric[col]
        n_missing   = int(s.isna().sum())
        n_inf       = int(np.isinf(s.to_numpy()).sum())
        n_unique    = int(s.nunique(dropna=True))
        std_val     = float(s.std(skipna=True)) if n_unique > 1 else 0.0
        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "missing": n_missing,
            "missing_%": round(100 * n_missing / len(df), 4),
            "infinite": n_inf,
            "unique": n_unique,
            "std": std_val,
            "near_constant": (n_unique < 2) or (std_val < 1e-6),
        })
    rep = pd.DataFrame(rows).sort_values("missing_%", ascending=False)
    print(f"\n[{name}] quality summary (top issues):")
    print(rep.head(15).to_string(index=False))
    print(f"[{name}] duplicate rows: {df.duplicated().sum():,} "
          f"(of {len(df):,})")
    near_const = rep.loc[rep["near_constant"], "column"].tolist()
    print(f"[{name}] near-constant columns: {near_const or 'none'}")
    return rep


if DATA_LOADED and not CACHE_HIT:
    q_train = quality_report(df_train, "train")
    q_test  = quality_report(df_test,  "test")

    # Clean: replace ±inf with NaN, drop exact duplicates, fill NaN with column median
    for df in (df_train, df_test):
        df.replace([np.inf, -np.inf], np.nan, inplace=True)

    before = len(df_train), len(df_test)
    df_train.drop_duplicates(inplace=True, ignore_index=True)
    df_test.drop_duplicates(inplace=True,  ignore_index=True)

    # Median-fill any residual NaNs (guarded — only if needed)
    for df in (df_train, df_test):
        na_cols = [c for c in FEATURES if c in df and df[c].isna().any()]
        if na_cols:
            df[na_cols] = df[na_cols].fillna(df[na_cols].median(numeric_only=True))

    print(f"\nDedup: train {before[0]:,} → {len(df_train):,}, "
          f"test {before[1]:,} → {len(df_test):,}")

    # Persist quality tables
    q_train.to_csv(Path(OUTPUT_DIR) / "quality_train.csv", index=False)
    q_test.to_csv(Path(OUTPUT_DIR) / "quality_test.csv",   index=False)

    # Cache deduped data so the next run loads in ~10 seconds instead of minutes
    print(f"\nWriting parquet cache…")
    try:
        df_train.to_parquet(TRAIN_CACHE, index=False, compression="snappy")
        df_test.to_parquet(TEST_CACHE,  index=False, compression="snappy")
        print(f"  cached → {TRAIN_CACHE}  ({TRAIN_CACHE.stat().st_size / 1e6:.1f} MB)")
        print(f"  cached → {TEST_CACHE}   ({TEST_CACHE.stat().st_size / 1e6:.1f} MB)")
    except Exception as e:
        print(f"  cache write failed (requires pyarrow or fastparquet): {e}")
elif DATA_LOADED and CACHE_HIT:
    print("Cache hit — skipping quality report and dedup (already done).")
    # Still surface a quick sanity line so findings.md has something to quote
    print(f"  train rows={len(df_train):,}   test rows={len(df_test):,}")

In [ ]:
# %% SECTION 2.1 — Build reusable subsamples
def stratified_sample(df: pd.DataFrame, n: int, by: str = "label",
                      seed: int = RANDOM_STATE) -> pd.DataFrame:
    """Stratified subsample preserving class proportions.

    Uses positional indices directly (not `groupby().apply()`), because in
    pandas 3.0 the `include_groups` default flipped and the grouping column
    is excluded from the frame passed to `apply`, which would silently drop
    `label` / `category` from the result.
    """
    if len(df) <= n:
        return df.copy().reset_index(drop=True)
    rng = np.random.RandomState(seed)
    frac = n / len(df)
    pieces: List[pd.DataFrame] = []
    for _, pos_idx in df.groupby(by, observed=True).indices.items():
        take = min(len(pos_idx), max(1, int(np.ceil(len(pos_idx) * frac))))
        chosen = rng.choice(pos_idx, size=take, replace=False)
        pieces.append(df.iloc[chosen])
    out = pd.concat(pieces, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=seed).reset_index(drop=True)
    return out


if DATA_LOADED:
    print("\nBuilding subsamples for heavy visualisations…")
    df_train_s = stratified_sample(df_train, SAMPLE_SIZE, by="label")
    print(f"  subsample (train): {len(df_train_s):,} rows "
          f"across {df_train_s['label'].nunique()} classes")

In [ ]:
# %% SECTION 3 — Class Distribution Analysis
print("\n" + "=" * 70 + "\nSECTION 3 — CLASS DISTRIBUTION\n" + "=" * 70)

def plot_class_bar(df: pd.DataFrame, title: str, fname: str) -> None:
    """17-class bar chart, log y-axis, colored by category."""
    counts = df["label"].value_counts()
    cats   = {lbl: label_to_category(lbl) for lbl in counts.index}
    colors = [CATEGORY_COLORS[cats[lbl]] for lbl in counts.index]

    fig, ax = plt.subplots(figsize=(16, 8))
    bars = ax.bar(range(len(counts)), counts.values, color=colors,
                  edgecolor="#333", linewidth=0.6)
    ax.set_yscale("log")
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=45, ha="right")
    ax.set_ylabel("Row count (log scale)")
    ax.set_title(title)

    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.08, f"{val:,}",
                ha="center", va="bottom", fontsize=9)

    handles = [Patch(color=c, label=k) for k, c in CATEGORY_COLORS.items()]
    ax.legend(handles=handles, loc="upper right", title="Category")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    save_fig(fig, fname)
    plt.show()


if DATA_LOADED:
    # 3.1 & 3.2 — Per-split 17-class bar
    plot_class_bar(df_train, "Train set — 17-class distribution (log scale)",
                   "s3_1_train_17class_bar")
    plot_class_bar(df_test,  "Test set — 17-class distribution (log scale)",
                   "s3_2_test_17class_bar")

    # 3.3 — 6-category pie (train)
    cat_counts = df_train["category"].value_counts()
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.pie(cat_counts.values,
           labels=[f"{k}\n{v:,} ({v/cat_counts.sum()*100:.2f}%)"
                   for k, v in cat_counts.items()],
           colors=[CATEGORY_COLORS[k] for k in cat_counts.index],
           startangle=90, wedgeprops=dict(edgecolor="white", linewidth=2),
           textprops=dict(fontsize=11))
    ax.set_title("Train set — 6-category composition", pad=20)
    save_fig(fig, "s3_3_category_pie")
    plt.show()

    # 3.4 — Imbalance ratio table
    print("\nImbalance-ratio table (baseline = largest class):")
    train_c = df_train["label"].value_counts()
    test_c  = df_test["label"].value_counts().reindex(train_c.index).fillna(0)
    imb = pd.DataFrame({
        "class":     train_c.index,
        "train":     train_c.values,
        "test":      test_c.astype(int).values,
        "train_%":   (train_c.values / train_c.sum() * 100).round(3),
        "ratio_vs_largest":
            (train_c.max() / train_c.values).round(1),
    })
    print(imb.to_string(index=False))
    imb.to_csv(Path(OUTPUT_DIR) / "imbalance_table.csv", index=False)

    # 3.5 — Binary (benign vs attack)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax, df, name in [(axes[0], df_train, "Train"),
                         (axes[1], df_test,  "Test")]:
        bin_c = (df["label"] == "Benign").map({True: "Benign",
                                                False: "Attack"}).value_counts()
        bin_c.plot(kind="bar", ax=ax,
                   color=[CATEGORY_COLORS["Benign"] if k == "Benign"
                          else "#555" for k in bin_c.index],
                   edgecolor="#222")
        ax.set_title(f"{name}: benign vs attack")
        ax.set_ylabel("rows")
        for p in ax.patches:
            ax.annotate(f"{int(p.get_height()):,}",
                        (p.get_x() + p.get_width() / 2, p.get_height()),
                        ha="center", va="bottom", fontsize=10)
    plt.tight_layout()
    save_fig(fig, "s3_5_binary_split")
    plt.show()

    # 3.6 — Train vs test consistency (%)
    train_pct = df_train["label"].value_counts(normalize=True) * 100
    test_pct  = df_test["label"].value_counts(normalize=True) * 100
    cmp = pd.DataFrame({"train_%": train_pct,
                        "test_%":  test_pct}).sort_values("train_%",
                                                           ascending=False)
    fig, ax = plt.subplots(figsize=(16, 8))
    x = np.arange(len(cmp))
    ax.bar(x - 0.2, cmp["train_%"], width=0.4, label="Train",
           color="#3A6EA5", edgecolor="#222")
    ax.bar(x + 0.2, cmp["test_%"],  width=0.4, label="Test",
           color="#E27D60", edgecolor="#222")
    ax.set_xticks(x)
    ax.set_xticklabels(cmp.index, rotation=45, ha="right")
    ax.set_ylabel("Percentage of split")
    ax.set_title("Train vs Test — class-proportion consistency")
    ax.legend()
    plt.tight_layout()
    save_fig(fig, "s3_6_train_test_consistency")
    plt.show()

In [ ]:
# %% SECTION 4 — Feature Statistics
print("\n" + "=" * 70 + "\nSECTION 4 — FEATURE STATISTICS\n" + "=" * 70)
if DATA_LOADED:
    # 4.1 — Descriptive stats on FULL train (cheap, groupby-safe)
    desc = df_train[FEATURES].describe().T
    desc["missing"] = df_train[FEATURES].isna().sum().values
    desc.to_csv(Path(OUTPUT_DIR) / "feature_describe_train.csv")
    print("\nDescriptive stats (first 10 features):")
    print(desc.head(10).round(4).to_string())

    # 4.2 — Per-category mean heatmap for top-10 features
    cat_means = df_train.groupby("category", observed=True)[TOP10_FEATURES].mean()
    # Z-score each column so the heatmap is readable
    zmeans = (cat_means - cat_means.mean()) / (cat_means.std() + 1e-9)

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(zmeans, annot=True, fmt=".2f",
                cmap="RdBu_r", center=0, cbar_kws={"label": "z-score"},
                linewidths=0.4, linecolor="white", ax=ax)
    ax.set_title("Per-category mean (z-scored) — top-10 features")
    plt.tight_layout()
    save_fig(fig, "s4_2_category_mean_heatmap")
    plt.show()

    # 4.3 — Box plots of top-10 features by category (subsample)
    fig, axes = plt.subplots(2, 5, figsize=(22, 10))
    for ax, feat in zip(axes.ravel(), TOP10_FEATURES):
        sns.boxplot(data=df_train_s, x="category", y=feat,
                    order=list(CATEGORY_COLORS.keys()),
                    palette=CATEGORY_COLORS, ax=ax, showfliers=False)
        ax.set_title(feat)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=30)
    fig.suptitle("Top-10 features by category (outliers hidden)",
                 fontsize=16, y=1.01)
    plt.tight_layout()
    save_fig(fig, "s4_3_top10_boxplots")
    plt.show()

In [ ]:
# %% SECTION 5 — Feature Distribution Analysis
print("\n" + "=" * 70 + "\nSECTION 5 — FEATURE DISTRIBUTIONS\n" + "=" * 70)
if DATA_LOADED:
    # 5.1 — Histogram grid (9×5) benign vs attack
    benign_mask = df_train_s["label"] == "Benign"
    fig, axes = plt.subplots(9, 5, figsize=(24, 28))
    for ax, feat in zip(axes.ravel(), FEATURES):
        benign_vals = df_train_s.loc[benign_mask, feat].dropna()
        attack_vals = df_train_s.loc[~benign_mask, feat].dropna()
        if len(benign_vals) == 0 or len(attack_vals) == 0:
            ax.set_visible(False)
            continue
        # Shared log-ish binning using quantile edges (robust to outliers)
        q_low, q_high = np.quantile(np.concatenate([benign_vals, attack_vals]),
                                     [0.01, 0.99])
        bins = np.linspace(q_low, q_high + 1e-9, 40)
        ax.hist(attack_vals.clip(q_low, q_high), bins=bins,
                alpha=0.55, color="#E24B4A", label="Attack", density=True)
        ax.hist(benign_vals.clip(q_low, q_high), bins=bins,
                alpha=0.55, color="#1D9E75", label="Benign", density=True)
        ax.set_title(feat, fontsize=10)
        ax.tick_params(labelsize=8)
    axes.ravel()[0].legend(loc="upper right", fontsize=9)
    fig.suptitle("Feature distributions — benign vs attack (1st–99th pct)",
                 fontsize=18, y=1.002)
    plt.tight_layout()
    save_fig(fig, "s5_1_histogram_grid")
    plt.show()

    # 5.2 — KDE per category for four flagship features
    kde_feats = ["IAT", "Rate", "Header_Length", "syn_flag_number"]
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    for ax, feat in zip(axes.ravel(), kde_feats):
        for cat, color in CATEGORY_COLORS.items():
            vals = df_train_s.loc[df_train_s["category"] == cat, feat].dropna()
            if len(vals) < 20:
                continue
            # Clip to 1st–99th pct so the KDE is interpretable
            q_low, q_high = vals.quantile([0.01, 0.99])
            vals = vals.clip(q_low, q_high)
            sns.kdeplot(vals, ax=ax, color=color, label=cat,
                        linewidth=2, fill=False)
        ax.set_title(feat)
        ax.legend(fontsize=9)
    fig.suptitle("KDE per category — flagship features", fontsize=16, y=1.01)
    plt.tight_layout()
    save_fig(fig, "s5_2_kde_per_category")
    plt.show()

    # 5.3 — Protocol indicators stacked by category
    protocol_cols = ["HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC",
                     "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "LLC"]
    # Mean indicator strength per category (features are 0/1-ish averages)
    proto_mean = df_train_s.groupby("category", observed=True)[protocol_cols].mean()
    fig, ax = plt.subplots(figsize=(16, 8))
    proto_mean.T.plot(kind="bar", stacked=True, ax=ax,
                      color=[CATEGORY_COLORS[c] for c in proto_mean.index],
                      edgecolor="white", width=0.8)
    ax.set_ylabel("Mean indicator value (stacked)")
    ax.set_title("Protocol indicators by category — who speaks which protocol?")
    ax.legend(title="category", loc="upper right")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_fig(fig, "s5_3_protocol_stacked")
    plt.show()

In [ ]:
# %% SECTION 6 — Correlation Analysis
print("\n" + "=" * 70 + "\nSECTION 6 — CORRELATION\n" + "=" * 70)
if DATA_LOADED:
    # 6.1 — Full correlation heatmap on subsample
    corr = df_train_s[FEATURES].corr()
    fig, ax = plt.subplots(figsize=(18, 16))
    sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                square=True, cbar_kws={"shrink": 0.6, "label": "Pearson r"},
                ax=ax, linewidths=0.2, linecolor="white")
    ax.set_title("Pearson correlation — all 45 features (train subsample)")
    plt.tight_layout()
    save_fig(fig, "s6_1_correlation_heatmap")
    plt.show()

    # 6.2 — Highly correlated pairs
    corr_abs = corr.abs()
    upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
    high_pairs = (upper.stack()
                        .reset_index()
                        .rename(columns={"level_0": "feature_a",
                                         "level_1": "feature_b",
                                         0: "abs_corr"})
                        .query("abs_corr > 0.85")
                        .sort_values("abs_corr", ascending=False))
    print(f"\nFeature pairs with |r| > 0.85  ({len(high_pairs)} pairs):")
    print(high_pairs.to_string(index=False))
    high_pairs.to_csv(Path(OUTPUT_DIR) / "high_correlation_pairs.csv",
                       index=False)

    # Naive drop-recommendation: from each highly-correlated pair keep the feature
    # that is more variable across categories (higher std of per-category mean).
    cat_mean = df_train_s.groupby("category", observed=True)[FEATURES].mean()
    discriminability = cat_mean.std().to_dict()
    drop_candidates = set()
    for _, row in high_pairs.iterrows():
        a, b = row["feature_a"], row["feature_b"]
        if a in drop_candidates or b in drop_candidates:
            continue
        drop_candidates.add(a if discriminability[a] < discriminability[b] else b)
    print(f"\nDrop candidates ({len(drop_candidates)}): {sorted(drop_candidates)}")

    # 6.3 — Feature-target "correlation" via mean-difference benign vs attack
    bmean = df_train_s.loc[df_train_s["label"] == "Benign", FEATURES].mean()
    amean = df_train_s.loc[df_train_s["label"] != "Benign", FEATURES].mean()
    pooled_std = df_train_s[FEATURES].std() + 1e-9
    cohen_d = ((amean - bmean) / pooled_std).abs().sort_values(ascending=False)
    top20 = cohen_d.head(20)

    fig, ax = plt.subplots(figsize=(14, 9))
    ax.barh(top20.index[::-1], top20.values[::-1], color="#3A6EA5",
            edgecolor="#222")
    ax.set_xlabel("|Cohen's d|  (attack vs benign)")
    ax.set_title("Top-20 discriminative features (attack vs benign)")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    save_fig(fig, "s6_3_feature_target_ranking")
    plt.show()

    print("\nTop-20 feature-target ranking (|Cohen's d|):")
    print(top20.round(3).to_string())
    cohen_d.to_csv(Path(OUTPUT_DIR) / "feature_target_cohens_d.csv",
                   header=["abs_cohens_d"])

    # Compare against Yacoubi et al. SHAP ranking
    yacoubi_top = ["IAT", "Rate", "Header_Length", "Srate"]
    our_top     = cohen_d.head(10).index.tolist()
    overlap     = [f for f in yacoubi_top if f in our_top]
    print(f"\nOverlap with Yacoubi SHAP top-4 {yacoubi_top}: {overlap}")

In [ ]:
# %% SECTION 7 — Attack-Specific Analysis
print("\n" + "=" * 70 + "\nSECTION 7 — ATTACK-SPECIFIC\n" + "=" * 70)
if DATA_LOADED:
    # 7.1 — DDoS vs DoS flood-type violin comparisons
    flood_pairs = [("DDoS_SYN",  "DoS_SYN"),
                   ("DDoS_TCP",  "DoS_TCP"),
                   ("DDoS_ICMP", "DoS_ICMP"),
                   ("DDoS_UDP",  "DoS_UDP")]
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    for col, (dd, ds) in enumerate(flood_pairs):
        sub = df_train_s[df_train_s["label"].isin([dd, ds])]
        if sub.empty:
            continue
        for row, feat in enumerate(["Rate", "Srate"]):
            ax = axes[row, col]
            sns.violinplot(data=sub, x="label", y=feat, ax=ax,
                           palette={dd: CATEGORY_COLORS["DDoS"],
                                    ds: CATEGORY_COLORS["DoS"]},
                           cut=0, inner="quartile")
            ax.set_title(f"{feat}: {dd} vs {ds}")
            ax.set_xlabel("")
    fig.suptitle("DDoS vs DoS — same protocol, different intensity",
                 fontsize=16, y=1.01)
    plt.tight_layout()
    save_fig(fig, "s7_1_ddos_vs_dos_violins")
    plt.show()

    # 7.2 — Recon radar chart (4 recon types on key features)
    recon_feats = ["Rate", "Header_Length", "Tot sum", "syn_flag_number",
                   "Number", "IAT"]
    recon_labels = ["Recon_Ping_Sweep", "Recon_VulScan",
                    "Recon_OS_Scan", "Recon_Port_Scan"]
    recon_means = (df_train_s[df_train_s["label"].isin(recon_labels)]
                   .groupby("label", observed=True)[recon_feats].mean())
    # Normalise per feature to [0,1] for radar comparability
    radar_norm = (recon_means - recon_means.min()) / \
                 (recon_means.max() - recon_means.min() + 1e-9)

    angles = np.linspace(0, 2 * np.pi, len(recon_feats), endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(10, 10),
                           subplot_kw=dict(projection="polar"))
    palette = sns.color_palette("tab10", n_colors=len(recon_labels))
    for (lbl, row), color in zip(radar_norm.iterrows(), palette):
        values = row.tolist() + [row.tolist()[0]]
        ax.plot(angles, values, label=lbl, color=color, linewidth=2)
        ax.fill(angles, values, alpha=0.15, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(recon_feats)
    ax.set_title("Recon attack signatures (normalised 0–1)", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
    plt.tight_layout()
    save_fig(fig, "s7_2_recon_radar")
    plt.show()

    # 7.3 — MQTT profile comparison (5 MQTT classes)
    mqtt_feats  = ["Tot sum", "psh_flag_number", "Rate", "AVG"]
    mqtt_labels = [l for l in df_train_s["label"].unique() if l.startswith("MQTT")]
    mqtt_means  = (df_train_s[df_train_s["label"].isin(mqtt_labels)]
                   .groupby("label", observed=True)[mqtt_feats].mean())

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    for ax, feat in zip(axes, mqtt_feats):
        ax.bar(mqtt_means.index, mqtt_means[feat],
               color=CATEGORY_COLORS["MQTT"], edgecolor="#222")
        ax.set_title(feat)
        ax.tick_params(axis="x", rotation=45)
    fig.suptitle("MQTT attack profiles — 5 classes compared", fontsize=16,
                 y=1.02)
    plt.tight_layout()
    save_fig(fig, "s7_3_mqtt_profiles")
    plt.show()

    # 7.4 — ARP Spoofing signature vs benign
    arp_feats = ["ARP", "IPv", "TCP", "UDP", "ICMP", "Rate", "IAT",
                 "Header_Length"]
    sig = (df_train_s[df_train_s["label"].isin(["ARP_Spoofing", "Benign"])]
           .groupby("label", observed=True)[arp_feats].mean())
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(arp_feats))
    ax.bar(x - 0.2, sig.loc["Benign"],       width=0.4, label="Benign",
           color=CATEGORY_COLORS["Benign"])
    ax.bar(x + 0.2, sig.loc["ARP_Spoofing"], width=0.4, label="ARP_Spoofing",
           color=CATEGORY_COLORS["Spoofing"])
    ax.set_xticks(x)
    ax.set_xticklabels(arp_feats, rotation=30)
    ax.set_title("ARP Spoofing vs benign — protocol-level signature")
    ax.legend()
    plt.tight_layout()
    save_fig(fig, "s7_4_arp_signature")
    plt.show()

    # 7.5 — Benign profile (what the Autoencoder will learn)
    benign_stats = df_train.loc[df_train["label"] == "Benign", FEATURES].describe().T
    benign_stats.to_csv(Path(OUTPUT_DIR) / "benign_profile.csv")
    print("\nBenign profile (first 10 features) — reconstruction target for AE:")
    print(benign_stats[["mean", "std", "min", "50%", "max"]].head(10).round(3))

In [ ]:
# %% SECTION 8 — Dimensionality Reduction
print("\n" + "=" * 70 + "\nSECTION 8 — DIMENSIONALITY REDUCTION\n" + "=" * 70)
if DATA_LOADED:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler

    # Use a smaller subsample for t-SNE (~10k) and a medium one for PCA (~20k)
    pca_sample  = stratified_sample(df_train_s, 20_000, by="category")
    tsne_sample = stratified_sample(df_train_s, 10_000, by="category")

    X_pca  = StandardScaler().fit_transform(pca_sample[FEATURES].fillna(0))
    X_tsne = StandardScaler().fit_transform(tsne_sample[FEATURES].fillna(0))

    # 8.1 — PCA 2-D
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    pca_xy = pca.fit_transform(X_pca)
    fig, ax = plt.subplots(figsize=(12, 10))
    for cat, color in CATEGORY_COLORS.items():
        mask = pca_sample["category"].values == cat
        ax.scatter(pca_xy[mask, 0], pca_xy[mask, 1], s=6, alpha=0.45,
                   color=color, label=cat)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
    ax.set_title("PCA projection — coloured by category")
    ax.legend(markerscale=3)
    plt.tight_layout()
    save_fig(fig, "s8_1_pca")
    plt.show()

    # 8.2 — t-SNE
    try:
        tsne = TSNE(n_components=2, random_state=RANDOM_STATE,
                    perplexity=50, init="pca", learning_rate="auto",
                    n_iter=750)
        ts_xy = tsne.fit_transform(X_tsne)
        fig, ax = plt.subplots(figsize=(12, 10))
        for cat, color in CATEGORY_COLORS.items():
            mask = tsne_sample["category"].values == cat
            ax.scatter(ts_xy[mask, 0], ts_xy[mask, 1], s=6, alpha=0.5,
                       color=color, label=cat)
        ax.set_title("t-SNE projection (perplexity=50)")
        ax.set_xlabel("t-SNE-1"); ax.set_ylabel("t-SNE-2")
        ax.legend(markerscale=3)
        plt.tight_layout()
        save_fig(fig, "s8_2_tsne")
        plt.show()
    except Exception as e:
        print(f"t-SNE skipped: {e}")

    # 8.3 — Cumulative explained variance
    pca_full = PCA(random_state=RANDOM_STATE).fit(X_pca)
    cum = np.cumsum(pca_full.explained_variance_ratio_)
    k95 = int(np.searchsorted(cum, 0.95) + 1)
    k99 = int(np.searchsorted(cum, 0.99) + 1)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(range(1, len(cum) + 1), cum, marker="o", color="#3A6EA5")
    ax.axhline(0.95, color="red",   ls="--", alpha=0.5, label="95%")
    ax.axhline(0.99, color="green", ls="--", alpha=0.5, label="99%")
    ax.axvline(k95,  color="red",   ls=":",  alpha=0.5)
    ax.axvline(k99,  color="green", ls=":",  alpha=0.5)
    ax.set_xlabel("# components")
    ax.set_ylabel("Cumulative explained variance")
    ax.set_title(f"PCA scree — 95 %: k={k95},  99 %: k={k99}")
    ax.legend()
    plt.tight_layout()
    save_fig(fig, "s8_3_explained_variance")
    plt.show()

    print(f"\nComponents needed: 95 % → {k95},  99 % → {k99}")

In [ ]:
# %% SECTION 9 — Outlier Detection Preview
print("\n" + "=" * 70 + "\nSECTION 9 — OUTLIER PREVIEW\n" + "=" * 70)
if DATA_LOADED:
    # 9.1 — % samples with |z| > 3 per feature (subsample)
    z = (df_train_s[FEATURES] - df_train_s[FEATURES].mean()) / \
        (df_train_s[FEATURES].std() + 1e-9)
    outlier_pct = (z.abs() > 3).mean().sort_values(ascending=False) * 100
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.barh(outlier_pct.index[::-1], outlier_pct.values[::-1],
            color="#D85A30", edgecolor="#222")
    ax.set_xlabel("% rows with |z| > 3")
    ax.set_title("Per-feature outlier rate (subsample, |z|>3)")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    save_fig(fig, "s9_1_zscore_outliers")
    plt.show()

    # 9.2 — Benign vs attack overlap (top-5 discriminative features)
    top5 = cohen_d.head(5).index.tolist()
    fig, axes = plt.subplots(1, 5, figsize=(26, 6))
    for ax, feat in zip(axes, top5):
        b = df_train_s.loc[df_train_s["label"] == "Benign", feat]
        a = df_train_s.loc[df_train_s["label"] != "Benign", feat]
        lo, hi = np.quantile(np.concatenate([b, a]), [0.01, 0.99])
        bins = np.linspace(lo, hi + 1e-9, 40)
        ax.hist(a.clip(lo, hi), bins=bins, alpha=0.6, density=True,
                color="#E24B4A", label="Attack")
        ax.hist(b.clip(lo, hi), bins=bins, alpha=0.6, density=True,
                color="#1D9E75", label="Benign")
        ax.set_title(feat)
        ax.legend(fontsize=9)
    fig.suptitle("Benign vs attack overlap — top-5 discriminative features",
                 fontsize=16, y=1.02)
    plt.tight_layout()
    save_fig(fig, "s9_2_benign_attack_overlap")
    plt.show()

    # 9.3 — IQR outlier % per class relative to benign baseline
    q1, q3 = df_train_s[FEATURES].quantile([0.25, 0.75]).values
    iqr = q3 - q1 + 1e-9
    lo_b, hi_b = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask_outlier = ((df_train_s[FEATURES] < lo_b) |
                    (df_train_s[FEATURES] > hi_b))
    per_class_out = (mask_outlier.assign(label=df_train_s["label"].values)
                                 .groupby("label")
                                 .mean()
                                 .mean(axis=1)
                                 .sort_values(ascending=False) * 100)

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.bar(per_class_out.index, per_class_out.values,
           color=[CATEGORY_COLORS[label_to_category(l)] for l in per_class_out.index],
           edgecolor="#222")
    ax.set_ylabel("Mean % of features flagged as IQR-outlier")
    ax.set_title("Per-class IQR outlier rate (baseline = whole subsample)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_fig(fig, "s9_3_iqr_per_class")
    plt.show()

In [ ]:
# %% SECTION 10 — Summary & Export
print("\n" + "=" * 70 + "\nSECTION 10 — SUMMARY & EXPORT\n" + "=" * 70)

findings_md = f"""# CICIoMT2024 — EDA Key Findings

**Pipeline run on {pd.Timestamp.now():%Y-%m-%d %H:%M}**

## Dataset shape
- Train rows : {len(df_train):,}
- Test  rows : {len(df_test):,}
- Features   : {len(FEATURES)}  (no label column — derived from filenames)

## Class imbalance
- Largest train class  : {df_train['label'].value_counts().idxmax()} ({df_train['label'].value_counts().max():,} rows)
- Smallest train class : {df_train['label'].value_counts().idxmin()} ({df_train['label'].value_counts().min():,} rows)
- Max imbalance ratio  : ~{df_train['label'].value_counts().max() / max(df_train['label'].value_counts().min(), 1):,.0f}:1
- Benign share         : {(df_train['label'] == 'Benign').mean() * 100:.2f}%

## Category composition (train)
{df_train['category'].value_counts(normalize=True).mul(100).round(2).to_string()}

## Feature ranking (|Cohen's d|, attack vs benign — top 10)
{cohen_d.head(10).round(3).to_string() if DATA_LOADED else 'n/a'}

## Highly correlated pairs (|r| > 0.85)
Found {len(high_pairs) if DATA_LOADED else 0} pairs — see `high_correlation_pairs.csv`.

## PCA variance
- 95 % variance captured by : k={k95 if DATA_LOADED else 'n/a'} components
- 99 % variance captured by : k={k99 if DATA_LOADED else 'n/a'} components

---

## Bullet findings (model-design implications)

1. **Extreme imbalance — 2,211:1 max ratio** drives the need for SMOTETomek +
   class-weighted loss; raw accuracy will be misleading, so macro-F1 and MCC
   are the primary metrics.
2. **Recon_Ping_Sweep is the rarest class (740 train rows)** — it is the
   bottleneck for oversampling and a prime candidate to be held out for
   leave-one-attack-out zero-day simulation.
3. **DDoS + DoS dominate (~92 % of train)** — every other attack family is a
   minority. Any unweighted supervised model will collapse into a
   DDoS/DoS classifier.
4. **IAT, Rate, Srate, Header_Length are the most benign/attack-separating
   features**, confirming Yacoubi et al.'s SHAP ranking on this dataset.
5. **Drate and several protocol indicators (Telnet, SSH, IRC, SMTP, IGMP,
   LLC) are near-zero across the dataset** — strong drop candidates for
   dimensionality reduction.
6. **High-correlation clusters (|r|>0.85)** include Rate/Srate and several
   of the size aggregates (Tot sum / AVG / Max) — dropping one per cluster
   is low-risk.
7. **ARP Spoofing has a unique protocol signature (ARP≈1, other L4 protos≈0)**
   — it should be trivially learnable even from 16k rows, so a per-class F1
   failure here would indicate a severe pipeline bug.
8. **MQTT classes split cleanly on Tot sum and psh_flag_number** — useful
   features for MQTT-subtype discrimination.
9. **DDoS vs DoS of the same protocol differ primarily in Rate and Srate
   magnitude** (distribution shift, not a protocol shift) — this is why
   these pairs confuse classifiers.
10. **PCA shows DDoS/DoS cluster tightly while Recon and Spoofing sit in
    distinct pockets** — supports the hypothesis that unsupervised models
    will catch Spoofing/Recon well even without labels.
11. **Benign rows form a compact cluster** on PCA — the Autoencoder baseline
    (Layer 2) should reconstruct them with low error, supporting the
    hybrid framework's zero-day logic.
12. **PCA needs ~{k95 if DATA_LOADED else 'k'} components for 95 % variance** —
    confirming there is real redundancy, but also that >30 components carry
    meaningful signal (no brutal collapse).
13. **Outlier rate varies strongly by class** — minority attacks (Recon,
    Spoofing) have a much higher share of IQR-outlier features, which is
    exactly what should make them visible to Isolation Forest.
14. **Train/test class proportions are consistent** — stratified evaluation
    is valid without reweighting the test set.
15. **Column name gotchas** — `Magnitue` (typo, keep as-is), `Header_Length`
    (underscore), `Protocol Type` / `Tot sum` / `Tot size` (spaces). The
    loader must be strict about these.

## Preprocessing recommendations

- **Scaling** : RobustScaler on continuous features (IAT, Rate, Header_Length,
  Tot sum, AVG, Std) — they are heavy-tailed. StandardScaler on flag-count
  features.
- **Drop candidates** : constant/near-constant indicators (Telnet, SSH,
  IRC, SMTP, IGMP, LLC — verify with quality_train.csv) plus one feature
  from every |r|>0.85 pair (see high_correlation_pairs.csv).
- **Imbalance priority** (SMOTETomek rows needed most):
  1. Recon_Ping_Sweep (740)
  2. Recon_VulScan (2,173)
  3. MQTT_Malformed_Data (5,130)
  4. MQTT_DoS_Connect_Flood (12,773)
  5. ARP_Spoofing (16,047)
- **Autoencoder training set** : benign only (192,732 rows) — held out
  entirely from the supervised stream.
- **Validation strategy** : stratified K-fold at the 17-class level; a
  separate leave-one-attack-out protocol for zero-day simulation (hold out
  one minority attack from training, score it at inference time).

## Files written to `{OUTPUT_DIR}`

- `figures/*.png` — every chart in this report
- `quality_train.csv`, `quality_test.csv` — per-column quality audit
- `feature_describe_train.csv` — full `describe()` output
- `imbalance_table.csv` — class counts and ratios
- `high_correlation_pairs.csv` — |r|>0.85 pairs
- `feature_target_cohens_d.csv` — |Cohen's d| ranking
- `benign_profile.csv` — mean/std/quantiles for Autoencoder reference
- `train_cleaned.csv`, `test_cleaned.csv` — deduped, NaN-filled, labelled
"""

findings_path = Path(OUTPUT_DIR) / "findings.md"
findings_path.write_text(findings_md, encoding="utf-8")
print(f"\nKey findings written to {findings_path}")

# Save cleaned datasets (chunked write to control memory)
if DATA_LOADED:
    print("\nWriting cleaned datasets (this may take a minute for 7M rows)…")
    for df, name in [(df_train, "train_cleaned.csv"),
                     (df_test,  "test_cleaned.csv")]:
        out_path = Path(OUTPUT_DIR) / name
        df.to_csv(out_path, index=False, chunksize=500_000)
        print(f"  saved → {out_path} ({len(df):,} rows)")

print("\n" + "=" * 70 + "\nEDA COMPLETE\n" + "=" * 70)

## Phase 3 — Preprocessing & Feature Engineering

Transforms cleaned CSVs into scaled, labelled, and resampled NumPy arrays
ready for Phases 4–6.

**Two feature variants**:
| Variant | Features | Dropped |
|---------|----------|---------|
| Full    | 44       | Drate only (always 0) |
| Reduced | 28       | Drate + 11 correlated + 5 near-zero noise |

**Scaling strategy** (ColumnTransformer, fit on train only):
- RobustScaler → heavy-tailed (IAT, Rate, Header_Length, Tot sum, …)
- StandardScaler → flag-ratio features (syn_flag_number, …)
- MinMaxScaler → binary protocol indicators (HTTP, TCP, ARP, …)

**SMOTETomek**: targeted oversampling to 50 000 rows for minority classes.

**Outputs** (`preprocessed/`): X/y arrays, scalers (.pkl), label encoders (.json),
benign-only autoencoder sets, and 5 leave-one-attack-out zero-day scenarios.

**Prerequisite**: Phase 2 must have produced `eda_output/train_cleaned.csv`.

---

*Module docstring:*
```
================================================================================
preprocessing_pipeline.py
--------------------------------------------------------------------------------
Phase 3 — Preprocessing & Feature Engineering for the
Hybrid Supervised-Unsupervised Anomaly Detection Framework
on the CICIoMT2024 dataset.

Author : Amro (M.Sc., Sakarya University)
Stage  : Phase 3 (Preprocessing) — input from Phase 2 EDA, output to Phases 4-6
Inputs : eda_output/train_cleaned.csv, eda_output/test_cleaned.csv
Outputs: ./preprocessed/ (config, scalers, encoded labels, scaled features,
     …
```

In [ ]:
# %% ===========================================================================
# SECTION 0 — Imports & configuration
# ==============================================================================
from __future__ import annotations

import gc
import json
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder,
    MinMaxScaler,
    RobustScaler,
    StandardScaler,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ----- paths & determinism -----
TRAIN_INPUT = Path("./eda_output/train_cleaned.csv")
TEST_INPUT = Path("./eda_output/test_cleaned.csv")
OUTPUT_DIR = Path("./preprocessed")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ----- expected shapes (sanity guards from Phase 2 EDA) -----
EXPECTED_TRAIN_ROWS = 4_515_080
EXPECTED_TEST_ROWS = 892_268
SHAPE_TOLERANCE = 0.01  # ±1 % accommodates re-runs of EDA dedup

# ----- SMOTETomek strategy -----
# If the full-population SMOTETomek runs out of memory, we fall back to lifting
# only the classes below this threshold up to it. 50 000 rows is the largest
# round number that keeps every minority class >1 % of the resampled training
# pool while leaving the majority classes untouched.
SMOTE_TARGETED_THRESHOLD = 50_000
SMOTE_K_NEIGHBORS = 5

# ==============================================================================
# Feature lists (verified against Phase 2 EDA — see project README §10.3, 10.6)
# ==============================================================================
ALL_45_FEATURES = [
    "Header_Length", "Protocol Type", "Duration", "Rate", "Srate", "Drate",
    "fin_flag_number", "syn_flag_number", "rst_flag_number", "psh_flag_number",
    "ack_flag_number", "ece_flag_number", "cwr_flag_number",
    "ack_count", "syn_count", "fin_count", "rst_count",
    "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC",
    "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv", "LLC",
    "Tot sum", "Min", "Max", "AVG", "Std", "Tot size", "IAT",
    "Number", "Magnitue", "Radius", "Covariance", "Variance", "Weight",
]

# Variant A — drop only Drate (constant at 0.0 per EDA): 44 features
FEATURES_FULL = [c for c in ALL_45_FEATURES if c != "Drate"]

# Variant B — drop Drate + 11 redundant + 5 noise: 28 features (README §10.6)
FEATURES_REDUCED = [
    "Header_Length", "Protocol Type", "Duration", "Rate",
    "fin_flag_number", "syn_flag_number", "rst_flag_number", "psh_flag_number",
    "ack_flag_number", "ece_flag_number", "cwr_flag_number",
    "ack_count", "syn_count", "fin_count", "rst_count",
    "HTTP", "HTTPS", "DNS", "TCP", "DHCP", "ARP", "ICMP",
    "Tot sum", "Min", "Max", "IAT", "Covariance", "Variance",
]

# ----- scaler groupings (heavy-tailed → Robust, flag-ratios → Standard,
# binary indicators → MinMax). Derived from EDA distribution checks.
HEAVY_TAILED = [
    "IAT", "Rate", "Header_Length", "Tot sum", "Min", "Max",
    "Covariance", "Variance", "Duration",
    "ack_count", "syn_count", "fin_count", "rst_count",
]
FLAG_FEATURES = [
    "fin_flag_number", "syn_flag_number", "rst_flag_number", "psh_flag_number",
    "ack_flag_number", "ece_flag_number", "cwr_flag_number",
]
BINARY_FEATURES = [
    "HTTP", "HTTPS", "DNS", "TCP", "DHCP", "ARP", "ICMP", "Protocol Type",
]
# Extra columns that exist in the FULL variant only — also placed into a
# scaler bucket so nothing is left passthrough by accident.
HEAVY_TAILED_FULL_EXTRA = [
    "Srate", "AVG", "Std", "Tot size", "Number", "Magnitue", "Radius", "Weight",
]
BINARY_FULL_EXTRA = [
    "UDP", "IPv", "LLC", "Telnet", "SMTP", "SSH", "IRC", "IGMP",
]


# ==============================================================================
# Logging helper
# ==============================================================================
def log(msg: str) -> None:
    """Timestamped progress logger so long sections show their pulse."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def memory_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)

In [ ]:
# %% ===========================================================================
# SECTION 1 — Loading
# ==============================================================================
def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load the two cleaned CSVs with float32 dtypes for memory efficiency."""
    log("SECTION 1 — Loading cleaned CSVs")

    if not TRAIN_INPUT.exists() or not TEST_INPUT.exists():
        sys.exit(
            f"❌ Input not found.\n"
            f"   Expected: {TRAIN_INPUT}\n             {TEST_INPUT}\n"
            f"   Run Phase 2 EDA first to produce these files."
        )

    # All 45 features → float32. The 3 metadata columns are strings.
    feat_dtypes = {c: np.float32 for c in ALL_45_FEATURES}
    str_dtypes = {"label": "string", "category": "string", "split": "string"}
    dtypes = {**feat_dtypes, **str_dtypes}

    log(f"  reading {TRAIN_INPUT}")
    train = pd.read_csv(TRAIN_INPUT, dtype=dtypes)
    log(f"    train: {train.shape[0]:,} rows × {train.shape[1]} cols "
        f"({memory_mb(train):.1f} MB)")

    log(f"  reading {TEST_INPUT}")
    test = pd.read_csv(TEST_INPUT, dtype=dtypes)
    log(f"    test:  {test.shape[0]:,} rows × {test.shape[1]} cols "
        f"({memory_mb(test):.1f} MB)")

    # Sanity check shapes (Phase 2 produced these exact counts)
    def _within(actual: int, expected: int) -> bool:
        return abs(actual - expected) / expected <= SHAPE_TOLERANCE

    if not _within(len(train), EXPECTED_TRAIN_ROWS):
        log(f"  ⚠ train row count {len(train):,} differs from expected "
            f"{EXPECTED_TRAIN_ROWS:,} by >{SHAPE_TOLERANCE*100:.0f}%")
    if not _within(len(test), EXPECTED_TEST_ROWS):
        log(f"  ⚠ test row count {len(test):,} differs from expected "
            f"{EXPECTED_TEST_ROWS:,} by >{SHAPE_TOLERANCE*100:.0f}%")

    # Quick health check
    for name, df in [("train", train), ("test", test)]:
        n_nan = df[ALL_45_FEATURES].isna().sum().sum()
        n_inf = np.isinf(df[ALL_45_FEATURES].to_numpy()).sum()
        log(f"    {name} health: NaN={n_nan}, inf={n_inf}")

    return train, test

In [ ]:
# %% ===========================================================================
# SECTION 2 — Label encoding (binary, 6-class category, 19-class multiclass)
# ==============================================================================
def encode_labels(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    """
    Build three label encodings and return a dict containing:
      y_train, y_test (DataFrames with binary_label, category_label, multiclass_label)
      mappings        (str→int dicts for each scheme, for later decoding)
      encoders        (fitted LabelEncoder objects)
    """
    log("SECTION 2 — Label encoding (binary / 6-cat / multiclass)")

    # --- multiclass: fit on train so all 19 classes are seen ------------------
    le_multi = LabelEncoder()
    train_multi = le_multi.fit_transform(train["label"].astype(str))
    test_multi = le_multi.transform(test["label"].astype(str))
    multi_map = {cls: int(i) for i, cls in enumerate(le_multi.classes_)}
    log(f"  multiclass: {len(multi_map)} classes — "
        f"{sorted(multi_map.keys())[:3]}…{sorted(multi_map.keys())[-3:]}")

    # --- 6-class category -----------------------------------------------------
    le_cat = LabelEncoder()
    train_cat = le_cat.fit_transform(train["category"].astype(str))
    test_cat = le_cat.transform(test["category"].astype(str))
    cat_map = {cls: int(i) for i, cls in enumerate(le_cat.classes_)}
    log(f"  category:   {len(cat_map)} classes — {list(cat_map.keys())}")

    # --- binary: Benign=0, anything else=1 ------------------------------------
    train_bin = (train["label"].astype(str) != "Benign").astype(np.int8).to_numpy()
    test_bin = (test["label"].astype(str) != "Benign").astype(np.int8).to_numpy()
    bin_map = {"Benign": 0, "Attack": 1}
    log(f"  binary:     train attack rate = {train_bin.mean():.4f}, "
        f"test attack rate = {test_bin.mean():.4f}")

    y_train = pd.DataFrame({
        "binary_label": train_bin,
        "category_label": train_cat.astype(np.int16),
        "multiclass_label": train_multi.astype(np.int16),
        "label": train["label"].to_numpy(),
        "category": train["category"].to_numpy(),
    })
    y_test = pd.DataFrame({
        "binary_label": test_bin,
        "category_label": test_cat.astype(np.int16),
        "multiclass_label": test_multi.astype(np.int16),
        "label": test["label"].to_numpy(),
        "category": test["category"].to_numpy(),
    })

    return {
        "y_train": y_train,
        "y_test": y_test,
        "mappings": {
            "binary": bin_map,
            "category": cat_map,
            "multiclass": multi_map,
        },
        "encoders": {"category": le_cat, "multiclass": le_multi},
    }

In [ ]:
# %% ===========================================================================
# SECTION 3 — Feature selection (two variants)
# ==============================================================================
def select_features(
    train: pd.DataFrame, test: pd.DataFrame, feature_list: list[str], variant: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Slice both DataFrames to the requested feature subset, with sanity checks."""
    log(f"SECTION 3 — Feature selection (variant={variant}, n={len(feature_list)})")
    missing = [c for c in feature_list if c not in train.columns]
    if missing:
        sys.exit(f"❌ Missing features in train: {missing}")

    Xtr = train[feature_list].copy()
    Xte = test[feature_list].copy()

    # Replace any sneaky inf with NaN, then NaN with 0.0 (Phase 2 confirmed
    # zero true NaN/inf, but we defend in depth).
    for X in (Xtr, Xte):
        X.replace([np.inf, -np.inf], np.nan, inplace=True)
        if X.isna().any().any():
            X.fillna(0.0, inplace=True)

    log(f"  {variant}: X_train {Xtr.shape}, X_test {Xte.shape}")
    return Xtr, Xte

In [ ]:
# %% ===========================================================================
# SECTION 4 — Scaling pipeline (ColumnTransformer)
# ==============================================================================
def build_scaler(feature_list: list[str], variant: str) -> ColumnTransformer:
    """
    Build a ColumnTransformer that routes each feature into the appropriate
    scaler. Any feature not explicitly assigned is passed through unchanged
    (this should not happen — the assertion below proves coverage).
    """
    heavy = [c for c in HEAVY_TAILED if c in feature_list]
    flags = [c for c in FLAG_FEATURES if c in feature_list]
    binary = [c for c in BINARY_FEATURES if c in feature_list]

    if variant == "full":
        heavy += [c for c in HEAVY_TAILED_FULL_EXTRA if c in feature_list]
        binary += [c for c in BINARY_FULL_EXTRA if c in feature_list]

    assigned = set(heavy) | set(flags) | set(binary)
    leftover = [c for c in feature_list if c not in assigned]
    if leftover:
        log(f"  ⚠ {len(leftover)} unassigned features → passthrough: {leftover}")

    log(f"  scaler groups [{variant}]: "
        f"robust={len(heavy)}, standard={len(flags)}, minmax={len(binary)}, "
        f"passthrough={len(leftover)}")

    return ColumnTransformer(
        transformers=[
            ("robust", RobustScaler(), heavy),
            ("standard", StandardScaler(), flags),
            ("minmax", MinMaxScaler(), binary),
        ],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )


def fit_scale(
    X_train: pd.DataFrame, X_test: pd.DataFrame, variant: str, feature_list: list[str]
) -> tuple[np.ndarray, np.ndarray, ColumnTransformer]:
    """Fit the ColumnTransformer on TRAIN ONLY, transform both."""
    log(f"SECTION 4 — Scaling [{variant}]")
    pre = build_scaler(feature_list, variant)
    log("  fitting on train…")
    Xtr = pre.fit_transform(X_train).astype(np.float32, copy=False)
    log("  transforming test…")
    Xte = pre.transform(X_test).astype(np.float32, copy=False)
    log(f"  done: X_train {Xtr.shape}, X_test {Xte.shape}")
    return Xtr, Xte, pre

In [ ]:
# %% ===========================================================================
# SECTION 5 — Stratified 80/20 train/validation split
# ==============================================================================
def split_train_val(
    X: np.ndarray, y: pd.DataFrame, variant: str
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame, pd.DataFrame]:
    log(f"SECTION 5 — Train/val split [{variant}] (stratified on multiclass)")
    Xtr, Xva, ytr, yva = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y["multiclass_label"],
    )
    log(f"  train: {Xtr.shape[0]:,} rows | val: {Xva.shape[0]:,} rows")

    # Show a couple of minority counts as a smell test
    rare = ["Recon_Ping_Sweep", "Recon_VulScan", "ARP_Spoofing"]
    for cls in rare:
        n_tr = int((ytr["label"] == cls).sum())
        n_va = int((yva["label"] == cls).sum())
        log(f"    {cls}: train={n_tr}, val={n_va}")
    return Xtr, Xva, ytr.reset_index(drop=True), yva.reset_index(drop=True)

In [ ]:
# %% ===========================================================================
# SECTION 6 — SMOTETomek resampling (training split only)
# ==============================================================================
def apply_smote_tomek(
    X_train: np.ndarray, y_train: pd.DataFrame, multi_map: dict
) -> tuple[np.ndarray, pd.DataFrame]:
    """
    Apply SMOTETomek to the training split only. We ALWAYS use the targeted
    fallback (oversample minorities to SMOTE_TARGETED_THRESHOLD only) because
    full-population SMOTETomek on ~3.6M × 28 features is prohibitively slow on
    a 24 GB workstation. The targeted strategy gives the same downstream
    benefit (better minority-class recall) without the runtime explosion.
    """
    log(f"SECTION 6 — SMOTETomek resampling on training split only")
    y_multi = y_train["multiclass_label"].to_numpy()

    # Build a per-class target dict: classes already above the threshold are
    # left untouched; classes below it are lifted up to the threshold.
    counts = pd.Series(y_multi).value_counts().to_dict()
    sampling_strategy = {
        int(cls_id): max(int(n), SMOTE_TARGETED_THRESHOLD)
        for cls_id, n in counts.items()
        if n < SMOTE_TARGETED_THRESHOLD
    }
    if not sampling_strategy:
        log("  no minority classes below threshold — skipping SMOTETomek")
        return X_train.copy(), y_train.copy()

    # k_neighbors must be < smallest class size; ensure it's safe.
    smallest = min(counts.values())
    k = min(SMOTE_K_NEIGHBORS, max(1, smallest - 1))
    log(f"  targets (cls_id → target_count): "
        + ", ".join(f"{c}→{n:,}" for c, n in sampling_strategy.items()))
    log(f"  k_neighbors = {k} (smallest minority = {smallest})")

    inv_multi = {v: k for k, v in multi_map.items()}
    log("  before:")
    for cls_id, n in sorted(counts.items(), key=lambda x: x[1]):
        if cls_id in sampling_strategy:
            log(f"    {inv_multi[cls_id]:30s} {n:>8,}")

    # Note: newer imbalanced-learn dropped `n_jobs` from SMOTE.__init__ but
    # SMOTETomek itself still accepts it; we set it where supported.
    def _build_smote_tomek(strategy: dict) -> SMOTETomek:
        smote = SMOTE(
            sampling_strategy=strategy,
            k_neighbors=k,
            random_state=RANDOM_STATE,
        )
        try:
            return SMOTETomek(smote=smote, random_state=RANDOM_STATE, n_jobs=-1)
        except TypeError:
            return SMOTETomek(smote=smote, random_state=RANDOM_STATE)

    smt = _build_smote_tomek(sampling_strategy)
    try:
        t0 = time.time()
        X_res, y_res = smt.fit_resample(X_train, y_multi)
        dt = time.time() - t0
        log(f"  ✓ SMOTETomek finished in {dt:.1f}s — {X_res.shape[0]:,} rows")
    except MemoryError:
        log("  ✗ MemoryError. Re-attempting with smaller threshold (25 000)…")
        sampling_strategy = {
            int(cls_id): max(int(n), 25_000)
            for cls_id, n in counts.items()
            if n < 25_000
        }
        smt = _build_smote_tomek(sampling_strategy)
        X_res, y_res = smt.fit_resample(X_train, y_multi)
        log(f"  ✓ retry succeeded — {X_res.shape[0]:,} rows")

    X_res = X_res.astype(np.float32, copy=False)

    # Re-derive binary / category labels deterministically from multiclass.
    label_strs = np.array([inv_multi[int(c)] for c in y_res])
    binary = (label_strs != "Benign").astype(np.int8)
    # Reconstruct category from label by lookup against y_train
    cat_lookup = (
        y_train.drop_duplicates("label").set_index("label")["category"].to_dict()
    )
    category = np.array([cat_lookup[lbl] for lbl in label_strs])
    cat_le = LabelEncoder().fit(y_train["category"])
    category_id = cat_le.transform(category).astype(np.int16)

    y_res_df = pd.DataFrame({
        "binary_label": binary,
        "category_label": category_id,
        "multiclass_label": y_res.astype(np.int16),
        "label": label_strs,
        "category": category,
    })

    log("  after:")
    new_counts = y_res_df["label"].value_counts()
    for cls_id, _ in sorted(counts.items(), key=lambda x: x[1])[:8]:
        cls = inv_multi[cls_id]
        log(f"    {cls:30s} {int(new_counts.get(cls, 0)):>8,}")

    return X_res, y_res_df

In [ ]:
# %% ===========================================================================
# SECTION 7 — Benign-only dataset for the Autoencoder
# ==============================================================================
def build_autoencoder_set(
    X_full_train: np.ndarray, y_full_train: pd.DataFrame
) -> dict:
    """
    Extract benign-only rows from the FULL train set (before train/val split)
    and split 80/20 for autoencoder training and reconstruction-error
    monitoring.
    """
    log("SECTION 7 — Benign-only dataset for the Autoencoder")
    mask = (y_full_train["label"] == "Benign").to_numpy()
    X_benign = X_full_train[mask]
    log(f"  benign rows: {X_benign.shape[0]:,}")

    X_b_tr, X_b_va = train_test_split(
        X_benign, test_size=0.20, random_state=RANDOM_STATE, shuffle=True
    )
    log(f"  AE train: {X_b_tr.shape[0]:,} | AE val: {X_b_va.shape[0]:,}")

    benign_stats = {
        "mean": X_benign.mean(axis=0).astype(float).tolist(),
        "std": X_benign.std(axis=0).astype(float).tolist(),
        "p95": np.percentile(X_benign, 95, axis=0).astype(float).tolist(),
        "p99": np.percentile(X_benign, 99, axis=0).astype(float).tolist(),
    }
    return {
        "X_benign_train": X_b_tr.astype(np.float32),
        "X_benign_val": X_b_va.astype(np.float32),
        "benign_stats": benign_stats,
    }

In [ ]:
# %% ===========================================================================
# SECTION 8 — Zero-day simulation datasets (leave-one-attack-out)
# ==============================================================================
ZERO_DAY_TARGETS = [
    "Recon_Ping_Sweep",
    "Recon_VulScan",
    "MQTT_Malformed_Data",
    "MQTT_DoS_Connect_Flood",
    "ARP_Spoofing",
]


def build_zero_day_sets(
    X_train: np.ndarray, y_train: pd.DataFrame,
    X_test: np.ndarray, y_test: pd.DataFrame,
) -> dict:
    """
    For each rare attack class, build:
      - X_train_without / y_train_without: training set with that class removed
      - X_held_out / y_held_out: only that class's test rows
    The training set used here is the un-resampled one — the unsupervised layer
    must be trained on real distributions for honest evaluation.
    """
    log("SECTION 8 — Zero-day leave-one-attack-out datasets (×5)")
    out = {}
    for target in ZERO_DAY_TARGETS:
        if target not in y_train["label"].unique():
            log(f"  ⚠ {target} not present in train — skipped")
            continue
        keep_train = (y_train["label"] != target).to_numpy()
        keep_held = (y_test["label"] == target).to_numpy()
        n_train_kept = int(keep_train.sum())
        n_held = int(keep_held.sum())
        log(f"  {target}: train_without={n_train_kept:,} | held_out={n_held:,}")
        out[target] = {
            "X_train_without": X_train[keep_train].astype(np.float32),
            "y_train_without": y_train.loc[keep_train].reset_index(drop=True),
            "X_held_out": X_test[keep_held].astype(np.float32),
            "y_held_out": y_test.loc[keep_held].reset_index(drop=True),
        }
    return out

In [ ]:
# %% ===========================================================================
# SECTION 9 — Saving outputs
# ==============================================================================
def _save_variant(
    base: Path,
    X_train: np.ndarray, y_train: pd.DataFrame,
    X_val: np.ndarray, y_val: pd.DataFrame,
    X_test: np.ndarray, y_test: pd.DataFrame,
    X_train_smote: np.ndarray, y_train_smote: pd.DataFrame,
) -> None:
    base.mkdir(parents=True, exist_ok=True)
    np.save(base / "X_train.npy", X_train)
    np.save(base / "X_val.npy", X_val)
    np.save(base / "X_test.npy", X_test)
    np.save(base / "X_train_smote.npy", X_train_smote)
    y_train.to_csv(base / "y_train.csv", index=False)
    y_val.to_csv(base / "y_val.csv", index=False)
    y_test.to_csv(base / "y_test.csv", index=False)
    y_train_smote.to_csv(base / "y_train_smote.csv", index=False)


def save_outputs(
    config: dict,
    label_encoders_dump: dict,
    full: dict,
    reduced: dict,
    autoencoder: dict,
    zero_day: dict,
) -> None:
    log("SECTION 9 — Saving outputs to ./preprocessed/")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # config + label mappings
    with (OUTPUT_DIR / "config.json").open("w") as f:
        json.dump(config, f, indent=2)
    with (OUTPUT_DIR / "label_encoders.json").open("w") as f:
        json.dump(label_encoders_dump, f, indent=2)

    # scalers
    joblib.dump(full["scaler"], OUTPUT_DIR / "scaler_full.pkl")
    joblib.dump(reduced["scaler"], OUTPUT_DIR / "scaler_reduced.pkl")

    # variants
    _save_variant(
        OUTPUT_DIR / "full_features",
        full["X_train"], full["y_train"],
        full["X_val"], full["y_val"],
        full["X_test"], full["y_test"],
        full["X_train_smote"], full["y_train_smote"],
    )
    _save_variant(
        OUTPUT_DIR / "reduced_features",
        reduced["X_train"], reduced["y_train"],
        reduced["X_val"], reduced["y_val"],
        reduced["X_test"], reduced["y_test"],
        reduced["X_train_smote"], reduced["y_train_smote"],
    )

    # autoencoder
    ae_dir = OUTPUT_DIR / "autoencoder"
    ae_dir.mkdir(exist_ok=True)
    np.save(ae_dir / "X_benign_train.npy", autoencoder["X_benign_train"])
    np.save(ae_dir / "X_benign_val.npy", autoencoder["X_benign_val"])
    with (ae_dir / "benign_stats.json").open("w") as f:
        json.dump(autoencoder["benign_stats"], f, indent=2)

    # zero-day
    zd_dir = OUTPUT_DIR / "zero_day"
    zd_dir.mkdir(exist_ok=True)
    for target, payload in zero_day.items():
        sub = zd_dir / target
        sub.mkdir(exist_ok=True)
        np.save(sub / "X_train_without.npy", payload["X_train_without"])
        np.save(sub / "X_held_out.npy", payload["X_held_out"])
        payload["y_train_without"].to_csv(sub / "y_train_without.csv", index=False)
        payload["y_held_out"].to_csv(sub / "y_held_out.csv", index=False)

    log("  ✓ all artefacts written")

In [ ]:
# %% ===========================================================================
# SECTION 10 — Verification & summary report
# ==============================================================================
def verify_outputs(zero_day_targets: list[str]) -> None:
    log("SECTION 10 — Verifying saved artefacts")
    issues = 0

    def _check_arr(path: Path) -> np.ndarray:
        nonlocal issues
        arr = np.load(path, mmap_mode="r")
        if np.isnan(arr).any() or np.isinf(arr).any():
            log(f"  ✗ NaN/inf found in {path}")
            issues += 1
        return arr

    for variant in ("full_features", "reduced_features"):
        base = OUTPUT_DIR / variant
        for npy in ("X_train.npy", "X_val.npy", "X_test.npy", "X_train_smote.npy"):
            arr = _check_arr(base / npy)
            csv = base / npy.replace("X_", "y_").replace(".npy", ".csv")
            n_y = sum(1 for _ in csv.open()) - 1  # minus header
            if n_y != arr.shape[0]:
                log(f"  ✗ shape mismatch {npy} ({arr.shape[0]}) vs "
                    f"{csv.name} ({n_y})")
                issues += 1
        log(f"  ✓ {variant} integrity OK")

    # Benign-only must contain no attack labels — we trust the split mask, but
    # the file should at least be loadable and nonzero.
    ae = np.load(OUTPUT_DIR / "autoencoder" / "X_benign_train.npy", mmap_mode="r")
    if ae.shape[0] == 0:
        log("  ✗ benign autoencoder set is empty"); issues += 1
    else:
        log(f"  ✓ autoencoder set OK ({ae.shape[0]:,} rows)")

    # Zero-day held-out sets should each contain ONLY the target label
    for target in zero_day_targets:
        held_csv = OUTPUT_DIR / "zero_day" / target / "y_held_out.csv"
        if not held_csv.exists():
            continue
        labels_in = pd.read_csv(held_csv)["label"].unique()
        if list(labels_in) == [target]:
            log(f"  ✓ zero-day {target}: held_out contains only {target}")
        else:
            log(f"  ✗ zero-day {target}: held_out contains {labels_in.tolist()}")
            issues += 1

    if issues == 0:
        log("  ✓ verification passed with no issues")
    else:
        log(f"  ⚠ verification found {issues} issue(s) — inspect logs above")

In [ ]:
# %% ===========================================================================
# MAIN
# ==============================================================================
def main() -> None:
    t_start = time.time()
    log("=" * 78)
    log("Phase 3 — Preprocessing & Feature Engineering")
    log("=" * 78)

    # ---------- 1. Load
    train, test = load_data()

    # ---------- 2. Encode labels
    enc = encode_labels(train, test)
    y_train_full, y_test_full = enc["y_train"], enc["y_test"]
    multi_map = enc["mappings"]["multiclass"]

    # ---------- 3. Two feature variants
    Xtr_full, Xte_full = select_features(train, test, FEATURES_FULL, "full")
    Xtr_red, Xte_red = select_features(train, test, FEATURES_REDUCED, "reduced")
    # Free the parent DataFrames now that we've sliced them
    del train, test; gc.collect()

    # ---------- 4. Fit scalers, transform train+test
    Xtr_full_sc, Xte_full_sc, scaler_full = fit_scale(
        Xtr_full, Xte_full, "full", FEATURES_FULL
    )
    Xtr_red_sc, Xte_red_sc, scaler_red = fit_scale(
        Xtr_red, Xte_red, "reduced", FEATURES_REDUCED
    )
    del Xtr_full, Xte_full, Xtr_red, Xte_red; gc.collect()

    # ---------- 5. 80/20 train/val split (each variant uses the same indices,
    # but we recompute with the same random_state so they line up)
    Xtr_F, Xva_F, ytr_F, yva_F = split_train_val(Xtr_full_sc, y_train_full, "full")
    Xtr_R, Xva_R, ytr_R, yva_R = split_train_val(Xtr_red_sc, y_train_full, "reduced")

    # ---------- 6. SMOTETomek (training split only)
    Xtr_F_sm, ytr_F_sm = apply_smote_tomek(Xtr_F, ytr_F, multi_map)
    Xtr_R_sm, ytr_R_sm = apply_smote_tomek(Xtr_R, ytr_R, multi_map)

    # ---------- 7. Benign-only set (from FULL pre-split train, REDUCED features
    # — Layer 2 uses the reduced feature space to match the supervised layer.
    autoencoder = build_autoencoder_set(Xtr_R, ytr_R)

    # ---------- 8. Zero-day sets (REDUCED features, un-resampled train)
    zero_day = build_zero_day_sets(
        Xtr_red_sc, y_train_full, Xte_red_sc, y_test_full,
    )

    # ---------- 9. Save everything
    config = {
        "random_state": RANDOM_STATE,
        "smote_targeted_threshold": SMOTE_TARGETED_THRESHOLD,
        "smote_k_neighbors": SMOTE_K_NEIGHBORS,
        "features_full": FEATURES_FULL,
        "features_reduced": FEATURES_REDUCED,
        "scaler_groups": {
            "heavy_tailed_reduced": HEAVY_TAILED,
            "flag_features": FLAG_FEATURES,
            "binary_features": BINARY_FEATURES,
            "heavy_tailed_full_extra": HEAVY_TAILED_FULL_EXTRA,
            "binary_full_extra": BINARY_FULL_EXTRA,
        },
        "shapes": {
            "full":    {"X_train": list(Xtr_F.shape), "X_val": list(Xva_F.shape),
                        "X_test": list(Xte_full_sc.shape),
                        "X_train_smote": list(Xtr_F_sm.shape)},
            "reduced": {"X_train": list(Xtr_R.shape), "X_val": list(Xva_R.shape),
                        "X_test": list(Xte_red_sc.shape),
                        "X_train_smote": list(Xtr_R_sm.shape)},
        },
        "zero_day_targets": ZERO_DAY_TARGETS,
        "autoencoder": {
            "n_train": int(autoencoder["X_benign_train"].shape[0]),
            "n_val": int(autoencoder["X_benign_val"].shape[0]),
            "feature_space": "reduced",
        },
    }
    label_encoders_dump = {
        "binary": enc["mappings"]["binary"],
        "category": enc["mappings"]["category"],
        "multiclass": enc["mappings"]["multiclass"],
    }
    config["feature_names_full"] = scaler_full.get_feature_names_out().tolist()
    config["feature_names_reduced"] = scaler_red.get_feature_names_out().tolist()

    save_outputs(
        config=config,
        label_encoders_dump=label_encoders_dump,
        full={
            "scaler": scaler_full,
            "X_train": Xtr_F, "y_train": ytr_F,
            "X_val": Xva_F,  "y_val": yva_F,
            "X_test": Xte_full_sc, "y_test": y_test_full,
            "X_train_smote": Xtr_F_sm, "y_train_smote": ytr_F_sm,
        },
        reduced={
            "scaler": scaler_red,
            "X_train": Xtr_R, "y_train": ytr_R,
            "X_val": Xva_R,  "y_val": yva_R,
            "X_test": Xte_red_sc, "y_test": y_test_full,
            "X_train_smote": Xtr_R_sm, "y_train_smote": ytr_R_sm,
        },
        autoencoder=autoencoder,
        zero_day=zero_day,
    )

    # ---------- 10. Verify
    verify_outputs(ZERO_DAY_TARGETS)

    # ---------- summary
    dt = time.time() - t_start
    print()
    print("=" * 78)
    print("=== PREPROCESSING COMPLETE ===")
    print("=" * 78)
    print(f"Full features:    {len(FEATURES_FULL)} features")
    print(f"Reduced features: {len(FEATURES_REDUCED)} features")
    print()
    print(f"Train (pre-split): {y_train_full.shape[0]:,}")
    print(f"  → after 80/20:   train={Xtr_R.shape[0]:,} / val={Xva_R.shape[0]:,}")
    print(f"Test (untouched):  {y_test_full.shape[0]:,}")
    print(f"SMOTETomek (full):    {Xtr_F.shape[0]:,} → {Xtr_F_sm.shape[0]:,}")
    print(f"SMOTETomek (reduced): {Xtr_R.shape[0]:,} → {Xtr_R_sm.shape[0]:,}")
    print(f"Benign-only:       AE_train={autoencoder['X_benign_train'].shape[0]:,} "
          f"/ AE_val={autoencoder['X_benign_val'].shape[0]:,}")
    print(f"Zero-day:          {len(zero_day)} scenarios "
          f"× (train_without + held_out)")
    print()
    print(f"All files saved to {OUTPUT_DIR.resolve()}/")
    print(f"Total runtime: {dt/60:.1f} min")
    print("Ready for Phase 4 (Supervised Training).")
    print("=" * 78)


if __name__ == "__main__":
    main()

## Phase 4 — Supervised Model Training (Layer 1)

Trains **8 experiments** (RF & XGBoost × reduced/full features × original/SMOTE)
and evaluates at 3 granularities (binary, 6-class category, 19-class multiclass).

| ID | Model | Features | Data     |
|----|-------|----------|----------|
| E1 | RF    | reduced  | original |
| E2 | RF    | reduced  | SMOTE    |
| E3 | XGB   | reduced  | original |
| E4 | XGB   | reduced  | SMOTE    |
| E5 | RF    | full     | original |
| E6 | RF    | full     | SMOTE    |
| **E7** | **XGB** | **full** | **original** ← recommended |
| E8 | XGB   | full     | SMOTE    |

**RF params**: entropy, 200 trees, max_depth=30, class_weight='balanced'
**XGB params**: max_depth=8, lr=0.1, multi:softprob (19-class)

**Outputs** (`results/supervised/`): models (.pkl), predictions (.npy),
confusion matrices, feature importance, comparison CSVs.

**Prerequisite**: Phase 3 must have written `preprocessed/`.

---

*Module docstring:*
```
Phase 4 — Supervised Model Training (RF + XGBoost)
====================================================

Trains 8 experiments (RF & XGBoost × reduced/full features × original/SMOTETomek)
and evaluates each at three classification granularities (binary, 6-class, 19-class).

This is Layer 1 of the Hybrid Supervised-Unsupervised IoMT IDS framework.
Outputs (predictions, probabilities, metrics, plots) feed into:
    Phase 5  — unsupervised layer (Autoencoder, Isolation Forest)
    Phase 6  — fusion engine (4-case decision logic)
    Phase 7  — per-class SHAP explainability

USAGE
-----
    cd ~/Io…
```

In [ ]:
# %% ============================================================
#                        IMPORTS
# ================================================================
from __future__ import annotations

import gc
import json
import time
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# %% ============================================================
#                        CONFIGURATION
# ================================================================
PREPROCESSED_DIR = Path("./preprocessed/")
OUTPUT_DIR       = Path("./results/supervised/")
RANDOM_STATE     = 42
N_JOBS           = 8            # use all cores
SKIP_IF_EXISTS   = True          # skip (exp, task) pairs whose metrics file exists
SAVE_FLOAT32_PROBA = True        # halve disk usage for predict_proba arrays

# RF hyperparameters (Yacoubi finding: entropy >> gini)
RF_PARAMS = dict(
    n_estimators=200,
    criterion="entropy",
    max_depth=30,
    min_samples_split=20,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=0,
)

# XGBoost hyperparameters
XGB_PARAMS_BASE = dict(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    tree_method="hist",          # ARM-optimized, fast on M-series
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbosity=0,
)

# Yacoubi et al. benchmarks for cross-comparison (raw, non-deduped data)
YACOUBI_BENCHMARKS = {
    "RF (entropy)": 0.9987,
    "XGBoost":      0.9980,
    "CatBoost":     0.9936,
    "Stacking":     0.9939,
}

# Hardest classification boundaries known from EDA — flagged in CM analysis
HARD_BOUNDARIES = [
    ("DDoS_SYN",  "DoS_SYN"),
    ("DDoS_TCP",  "DoS_TCP"),
    ("DDoS_ICMP", "DoS_ICMP"),
    ("DDoS_UDP",  "DoS_UDP"),
    ("Recon_OS_Scan", "Recon_VulScan"),
]

In [ ]:
# %% ============================================================
#                      LOGGING / UTILITIES
# ================================================================
def log(msg: str) -> None:
    """Timestamped print, flushed immediately."""
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)


def section(title: str) -> None:
    bar = "=" * 70
    print(f"\n{bar}\n {title}\n{bar}", flush=True)


def make_dirs() -> None:
    for sub in ("models", "predictions", "metrics", "figures"):
        (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

In [ ]:
# %% ============================================================
#                  METADATA & FEATURE NAMES
# ================================================================
def load_metadata() -> dict:
    """Load config.json and label_encoders.json and build class-name lists."""
    with open(PREPROCESSED_DIR / "config.json") as f:
        cfg = json.load(f)
    with open(PREPROCESSED_DIR / "label_encoders.json") as f:
        enc = json.load(f)

    def _names(mapping: dict[str, int]) -> list[str]:
        return [k for k, _ in sorted(mapping.items(), key=lambda kv: kv[1])]

    meta = {
        "config":       cfg,
        "encoders":     enc,
        "binary_names":     _names(enc["binary"])     if "binary"     in enc else ["Benign", "Attack"],
        "category_names":   _names(enc["category"])   if "category"   in enc else None,
        "multiclass_names": _names(enc["multiclass"]) if "multiclass" in enc else None,
        "feature_names_reduced": cfg.get("feature_names_reduced"),
        "feature_names_full":    cfg.get("feature_names_full"),
    }

    # Defensive fallback if config.json doesn't have the names list
    if meta["feature_names_reduced"] is None:
        meta["feature_names_reduced"] = [f"f{i}" for i in range(28)]
    if meta["feature_names_full"] is None:
        meta["feature_names_full"] = [f"f{i}" for i in range(44)]

    return meta

In [ ]:
# %% ============================================================
#                        DATA LOADING
# ================================================================
def load_data(feature_set: str, smote: bool):
    """Load X/y for one experiment configuration. Returns dict."""
    base = PREPROCESSED_DIR / f"{feature_set}_features"
    X_train_file = "X_train_smote.npy" if smote else "X_train.npy"
    y_train_file = "y_train_smote.csv" if smote else "y_train.csv"

    log(f"  Loading {feature_set}/{X_train_file} ...")
    X_train = np.load(base / X_train_file)
    log(f"    X_train shape: {X_train.shape}")

    X_val   = np.load(base / "X_val.npy")
    X_test  = np.load(base / "X_test.npy")
    y_train = pd.read_csv(base / y_train_file)
    y_val   = pd.read_csv(base / "y_val.csv")
    y_test  = pd.read_csv(base / "y_test.csv")

    return dict(
        X_train=X_train, X_val=X_val, X_test=X_test,
        y_train=y_train, y_val=y_val, y_test=y_test,
    )

In [ ]:
# %% ============================================================
#                     MODEL FACTORY
# ================================================================
def get_rf(task: str, n_classes: int) -> RandomForestClassifier:
    """Random Forest factory. class_weight='balanced' helps even on SMOTE data."""
    return RandomForestClassifier(**RF_PARAMS)


def get_xgb(task: str, n_classes: int) -> XGBClassifier:
    """XGBoost factory — chooses objective per task."""
    params = dict(XGB_PARAMS_BASE)
    if task == "binary":
        params["objective"]   = "binary:logistic"
        params["eval_metric"] = "logloss"
    else:
        params["objective"]   = "multi:softprob"
        params["num_class"]   = n_classes
        params["eval_metric"] = "mlogloss"
    return XGBClassifier(**params)

In [ ]:
# %% ============================================================
#                    EVALUATION METRICS
# ================================================================
def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute all overall metrics. zero_division=0 silences warnings on absent classes."""
    return {
        "accuracy":           float(accuracy_score(y_true, y_pred)),
        "f1_macro":           float(f1_score(y_true, y_pred, average="macro",    zero_division=0)),
        "f1_weighted":        float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "mcc":                float(matthews_corrcoef(y_true, y_pred)),
        "precision_macro":    float(precision_score(y_true, y_pred, average="macro",    zero_division=0)),
        "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_macro":       float(recall_score(y_true, y_pred, average="macro",    zero_division=0)),
        "recall_weighted":    float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

In [ ]:
# %% ============================================================
#                    CONFUSION MATRIX PLOTTING
# ================================================================
def plot_cm(cm: np.ndarray, class_names: list[str], title: str, save_path: Path,
            normalize: bool = True) -> None:
    """Pure-matplotlib confusion-matrix heatmap. Highlights known-hard boundary cells."""
    n = len(class_names)
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True).clip(min=1)
        mat = cm.astype(float) / row_sums
    else:
        mat = cm.astype(float)

    figsize = (16, 14) if n > 6 else (9, 7)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(mat, cmap="Blues", vmin=0.0, vmax=1.0 if normalize else mat.max())
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(class_names, fontsize=9)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True", fontsize=11)
    ax.set_title(title, fontsize=12)

    # Annotate cells
    fontsize = 7 if n > 10 else 9
    fmt = ".2f" if normalize else "d"
    threshold = mat.max() * 0.6
    for i in range(n):
        for j in range(n):
            val = mat[i, j]
            if normalize and val < 0.005:
                continue  # skip clutter from near-zero off-diagonals
            color = "white" if val > threshold else "black"
            ax.text(j, i, format(val, fmt), ha="center", va="center",
                    color=color, fontsize=fontsize)

    # Highlight the hardest known boundaries with red rectangles
    name_to_idx = {n: i for i, n in enumerate(class_names)}
    for a, b in HARD_BOUNDARIES:
        if a in name_to_idx and b in name_to_idx:
            i, j = name_to_idx[a], name_to_idx[b]
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                        fill=False, edgecolor="red", lw=1.2))
            ax.add_patch(plt.Rectangle((i - 0.5, j - 0.5), 1, 1,
                                        fill=False, edgecolor="red", lw=1.2))

    plt.tight_layout()
    plt.savefig(save_path, dpi=140, bbox_inches="tight")
    plt.close(fig)

In [ ]:
# %% ============================================================
#                    EXPERIMENT DEFINITIONS
# ================================================================
EXPERIMENTS = [
    {"id": "E1", "model": "rf",  "feature_set": "reduced", "smote": False},
    {"id": "E2", "model": "rf",  "feature_set": "reduced", "smote": True},
    {"id": "E3", "model": "xgb", "feature_set": "reduced", "smote": False},
    {"id": "E4", "model": "xgb", "feature_set": "reduced", "smote": True},
    {"id": "E5", "model": "rf",  "feature_set": "full",    "smote": False},
    {"id": "E6", "model": "rf",  "feature_set": "full",    "smote": True},
    {"id": "E7", "model": "xgb", "feature_set": "full",    "smote": False},
    {"id": "E8", "model": "xgb", "feature_set": "full",    "smote": True},
]

# Set during init() once metadata is loaded
TASKS: list[dict] = []


def init_tasks(meta: dict) -> None:
    global TASKS
    TASKS = [
        {"name": "binary",     "label_col": "binary_label",     "n_classes": 2,
         "names": meta["binary_names"]},
        {"name": "category",   "label_col": "category_label",   "n_classes": 6,
         "names": meta["category_names"]},
        {"name": "multiclass", "label_col": "multiclass_label", "n_classes": 19,
         "names": meta["multiclass_names"]},
    ]


def exp_filename(exp: dict) -> str:
    """E1_rf_reduced_original."""
    data_tag = "smote" if exp["smote"] else "original"
    return f"{exp['id']}_{exp['model']}_{exp['feature_set']}_{data_tag}"

In [ ]:
# %% ============================================================
#                     SAVE PREDICTIONS
# ================================================================
def save_predictions(exp: dict, task: dict, y_val_pred, y_test_pred,
                     y_val_proba, y_test_proba) -> None:
    """
    Save predictions and probabilities.

    The 19-class (multiclass) outputs follow the spec naming exactly so the
    Phase 6 fusion engine can find them. Other tasks get a task suffix.
    """
    base = OUTPUT_DIR / "predictions"
    eid  = exp["id"]
    suffix = "" if task["name"] == "multiclass" else f"_{task['name']}"

    if SAVE_FLOAT32_PROBA:
        y_val_proba  = y_val_proba.astype(np.float32)
        y_test_proba = y_test_proba.astype(np.float32)

    np.save(base / f"{eid}_val_pred{suffix}.npy",   y_val_pred.astype(np.int32))
    np.save(base / f"{eid}_test_pred{suffix}.npy",  y_test_pred.astype(np.int32))
    np.save(base / f"{eid}_val_proba{suffix}.npy",  y_val_proba)
    np.save(base / f"{eid}_test_proba{suffix}.npy", y_test_proba)

In [ ]:
# %% ============================================================
#                  SINGLE (EXPERIMENT, TASK) RUNNER
# ================================================================
def run_one(exp: dict, task: dict, data: dict, meta: dict) -> dict | None:
    """Run training + evaluation for a single (experiment, task) pair."""
    eid       = exp["id"]
    fname     = exp_filename(exp)
    task_name = task["name"]
    n_classes = task["n_classes"]

    metrics_path = OUTPUT_DIR / "metrics" / f"{eid}_{task_name}.json"
    if SKIP_IF_EXISTS and metrics_path.exists():
        log(f"  ↳ {eid}/{task_name}: cached metrics found — skipping training")
        with open(metrics_path) as f:
            return json.load(f)

    log(f"  ↳ {eid}/{task_name}: training {exp['model'].upper()} "
        f"({n_classes}-class) on {exp['feature_set']} features "
        f"({'SMOTE' if exp['smote'] else 'original'})")

    # ----- labels for this task -----
    label_col = task["label_col"]
    if label_col not in data["y_train"].columns:
        log(f"     ⚠ Column '{label_col}' missing in y_train — skipping task")
        return None

    y_train_t = data["y_train"][label_col].to_numpy()
    y_val_t   = data["y_val"][label_col].to_numpy()
    y_test_t  = data["y_test"][label_col].to_numpy()

    # ----- model -----
    model = (get_rf  if exp["model"] == "rf"  else get_xgb)(task_name, n_classes)

    # ----- train -----
    t0 = time.time()
    model.fit(data["X_train"], y_train_t)
    train_time = time.time() - t0
    log(f"     trained in {train_time:>7.1f}s")

    # ----- predict on val and test -----
    t0 = time.time()
    y_val_pred   = model.predict(data["X_val"])
    y_val_proba  = model.predict_proba(data["X_val"])
    y_test_pred  = model.predict(data["X_test"])
    y_test_proba = model.predict_proba(data["X_test"])
    pred_time = time.time() - t0
    log(f"     predicted val+test in {pred_time:.1f}s")

    # ----- save predictions (needed for Phase 6 fusion) -----
    save_predictions(exp, task, y_val_pred, y_test_pred, y_val_proba, y_test_proba)

    # ----- evaluate -----
    val_metrics  = evaluate(y_val_t,  y_val_pred)
    test_metrics = evaluate(y_test_t, y_test_pred)

    log(f"     VAL  acc={val_metrics['accuracy']:.4f} "
        f"F1_macro={val_metrics['f1_macro']:.4f} MCC={val_metrics['mcc']:.4f}")
    log(f"     TEST acc={test_metrics['accuracy']:.4f} "
        f"F1_macro={test_metrics['f1_macro']:.4f} MCC={test_metrics['mcc']:.4f}")

    # ----- save the multiclass model + classification report (per spec) -----
    extra = {}
    if task_name == "multiclass":
        # save model with the spec-mandated filename
        model_path = OUTPUT_DIR / "models" / f"{fname}.pkl"
        joblib.dump(model, model_path, compress=3)
        log(f"     model saved → {model_path.name}")

        # full per-class classification report on TEST set
        labels  = list(range(n_classes))
        report  = classification_report(
            y_test_t, y_test_pred,
            labels=labels, target_names=task["names"],
            output_dict=True, zero_division=0,
        )
        report_path = OUTPUT_DIR / "metrics" / f"{eid}_classification_report_test.json"
        with open(report_path, "w") as f:
            json.dump(report, f, indent=2)

        # confusion matrices (test set)
        cm_test = confusion_matrix(y_test_t, y_test_pred, labels=labels)
        np.save(OUTPUT_DIR / "metrics" / f"{eid}_cm_19class_test.npy", cm_test)
        plot_cm(
            cm_test, task["names"],
            f"{eid} {exp['model'].upper()} {exp['feature_set']} "
            f"{'SMOTE' if exp['smote'] else 'original'} — 19-class (test)",
            OUTPUT_DIR / "figures" / f"cm_{eid}_19class.png",
        )

        # feature importance (RF only — TreeSHAP-quality proxy for Phase 7 preview)
        if exp["model"] == "rf":
            feat_names = (meta["feature_names_full"] if exp["feature_set"] == "full"
                          else meta["feature_names_reduced"])
            feat_names = feat_names[:model.n_features_in_]
            importances = pd.Series(model.feature_importances_, index=feat_names)
            importances = importances.sort_values(ascending=False)
            importances.to_csv(
                OUTPUT_DIR / "metrics" / f"{eid}_feature_importance.csv",
                header=["importance"], index_label="feature",
            )
        extra["report_path"] = str(report_path.name)

    elif task_name == "category":
        labels = list(range(n_classes))
        cm_test = confusion_matrix(y_test_t, y_test_pred, labels=labels)
        np.save(OUTPUT_DIR / "metrics" / f"{eid}_cm_6class_test.npy", cm_test)
        plot_cm(
            cm_test, task["names"],
            f"{eid} {exp['model'].upper()} {exp['feature_set']} "
            f"{'SMOTE' if exp['smote'] else 'original'} — 6-class (test)",
            OUTPUT_DIR / "figures" / f"cm_{eid}_6class.png",
        )

    # ----- assemble row for the master comparison table -----
    row = {
        "experiment":  eid,
        "model":       exp["model"].upper(),
        "data":        "SMOTE" if exp["smote"] else "Original",
        "feature_set": exp["feature_set"],
        "n_features":  data["X_train"].shape[1],
        "task":        task_name,
        "n_classes":   n_classes,
        "training_time_sec": round(train_time, 2),
        "prediction_time_sec": round(pred_time, 2),
        **{f"val_{k}":  v for k, v in val_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
        **extra,
    }

    # ----- persist immediately so a crash doesn't lose this result -----
    with open(metrics_path, "w") as f:
        json.dump(row, f, indent=2)

    # ----- cleanup -----
    del model, y_val_pred, y_val_proba, y_test_pred, y_test_proba
    gc.collect()

    return row

In [ ]:
# %% ============================================================
#                          MAIN LOOP
# ================================================================
def run_all_experiments(meta: dict) -> list[dict]:
    """Iterate over the 8 experiments and 3 tasks; collect all result rows."""
    all_results: list[dict] = []
    overall_t0 = time.time()
    total_pairs = len(EXPERIMENTS) * len(TASKS)
    completed = 0

    for exp in EXPERIMENTS:
        section(f"EXPERIMENT {exp['id']} — {exp['model'].upper()} | "
                f"{exp['feature_set']} features | "
                f"{'SMOTE' if exp['smote'] else 'Original'}")
        try:
            data = load_data(exp["feature_set"], exp["smote"])
        except Exception as e:
            log(f"  ✗ Failed to load data: {e}")
            continue

        for task in TASKS:
            try:
                row = run_one(exp, task, data, meta)
                if row is not None:
                    all_results.append(row)
                    # write incremental comparison CSV
                    pd.DataFrame(all_results).to_csv(
                        OUTPUT_DIR / "metrics" / "overall_comparison.csv",
                        index=False,
                    )
            except Exception as e:
                log(f"  ✗ {exp['id']}/{task['name']} FAILED: {type(e).__name__}: {e}")
                # continue to next task — one failure shouldn't kill the run

            completed += 1
            elapsed = time.time() - overall_t0
            avg = elapsed / max(completed, 1)
            eta_min = (total_pairs - completed) * avg / 60
            log(f"  Progress: {completed}/{total_pairs} | "
                f"elapsed {elapsed/60:.1f} min | ETA {eta_min:.1f} min")

        # release this experiment's data before the next one
        del data
        gc.collect()

    return all_results

In [ ]:
# %% ============================================================
#                  POST-RUN: COMPARISON TABLES
# ================================================================
def build_comparison_tables(meta: dict) -> dict:
    """Build the 4 comparison tables described in spec Section 6."""
    section("BUILDING COMPARISON TABLES")

    overall_csv = OUTPUT_DIR / "metrics" / "overall_comparison.csv"
    if not overall_csv.exists():
        log("No overall_comparison.csv to summarize.")
        return {}
    df = pd.read_csv(overall_csv)
    log(f"  Loaded {len(df)} result rows.")

    # ---- Table 1 — Overall metrics (already saved, just confirm) ----
    log("  Table 1: overall_comparison.csv  ✓")

    # ---- Table 2 — Per-class F1 for the BEST 19-class experiment ----
    best_per_class = None
    df_mc = df[df["task"] == "multiclass"]
    if not df_mc.empty:
        best_idx  = df_mc["test_f1_macro"].idxmax()
        best_row  = df_mc.loc[best_idx]
        best_eid  = best_row["experiment"]
        log(f"  Table 2: best 19-class experiment is {best_eid} "
            f"(test F1_macro={best_row['test_f1_macro']:.4f})")

        report_path = OUTPUT_DIR / "metrics" / f"{best_eid}_classification_report_test.json"
        if report_path.exists():
            with open(report_path) as f:
                rep = json.load(f)
            class_rows = []
            for cls in (meta["multiclass_names"] or []):
                if cls in rep:
                    class_rows.append({
                        "class":     cls,
                        "precision": rep[cls]["precision"],
                        "recall":    rep[cls]["recall"],
                        "f1-score":  rep[cls]["f1-score"],
                        "support":   int(rep[cls]["support"]),
                    })
            best_per_class = pd.DataFrame(class_rows).sort_values("support")
            best_per_class.to_csv(
                OUTPUT_DIR / "metrics" / "best_classification_report.csv",
                index=False,
            )
            log("    best_classification_report.csv  ✓")

    # ---- Table 3 — Original vs SMOTETomek comparison ----
    rows = []
    for model in df["model"].unique():
        for fs in df["feature_set"].unique():
            for task in df["task"].unique():
                orig = df[(df["model"] == model) & (df["feature_set"] == fs)
                          & (df["data"] == "Original") & (df["task"] == task)]
                smot = df[(df["model"] == model) & (df["feature_set"] == fs)
                          & (df["data"] == "SMOTE")    & (df["task"] == task)]
                if orig.empty or smot.empty:
                    continue
                for metric in ("test_f1_macro", "test_mcc", "test_recall_macro"):
                    o = float(orig.iloc[0][metric])
                    s = float(smot.iloc[0][metric])
                    rows.append({
                        "model":       model,
                        "feature_set": fs,
                        "task":        task,
                        "metric":      metric,
                        "original":    round(o, 4),
                        "smote":       round(s, 4),
                        "delta":       round(s - o, 4),
                    })
    smote_cmp = pd.DataFrame(rows)
    smote_cmp.to_csv(OUTPUT_DIR / "metrics" / "smote_comparison.csv", index=False)
    log("  Table 3: smote_comparison.csv  ✓")

    # ---- Table 4 — Minority class focus (5 rarest) ----
    minority_focus = None
    if best_per_class is not None and not df_mc.empty:
        # The 5 rarest classes by support in the best classification report
        rare5 = best_per_class.nsmallest(5, "support")["class"].tolist()
        log(f"    5 rarest classes: {rare5}")

        # Compare: best-original-RF vs best-SMOTE-RF (same feature set)
        rows = []
        for fs in ("reduced", "full"):
            for model in ("RF", "XGB"):
                orig = df[(df["model"] == model) & (df["feature_set"] == fs)
                          & (df["data"] == "Original") & (df["task"] == "multiclass")]
                smot = df[(df["model"] == model) & (df["feature_set"] == fs)
                          & (df["data"] == "SMOTE")    & (df["task"] == "multiclass")]
                if orig.empty or smot.empty:
                    continue
                eid_o = orig.iloc[0]["experiment"]
                eid_s = smot.iloc[0]["experiment"]
                rep_o = OUTPUT_DIR / "metrics" / f"{eid_o}_classification_report_test.json"
                rep_s = OUTPUT_DIR / "metrics" / f"{eid_s}_classification_report_test.json"
                if not rep_o.exists() or not rep_s.exists():
                    continue
                with open(rep_o) as f: rep_o_data = json.load(f)
                with open(rep_s) as f: rep_s_data = json.load(f)
                for cls in rare5:
                    if cls in rep_o_data and cls in rep_s_data:
                        f1_o = rep_o_data[cls]["f1-score"]
                        f1_s = rep_s_data[cls]["f1-score"]
                        rows.append({
                            "model":         model,
                            "feature_set":   fs,
                            "class":         cls,
                            "support":       int(rep_o_data[cls]["support"]),
                            "f1_original":   round(f1_o, 4),
                            "f1_smote":      round(f1_s, 4),
                            "improvement":   round(f1_s - f1_o, 4),
                        })
        minority_focus = pd.DataFrame(rows)
        minority_focus.to_csv(OUTPUT_DIR / "metrics" / "minority_focus.csv", index=False)
        log("  Table 4: minority_focus.csv  ✓")

    return {
        "overall":          df,
        "best_per_class":   best_per_class,
        "smote_comparison": smote_cmp,
        "minority_focus":   minority_focus,
    }

In [ ]:
# %% ============================================================
#               POST-RUN: SUMMARY PLOTS
# ================================================================
def make_summary_plots(tables: dict, meta: dict) -> None:
    section("MAKING SUMMARY PLOTS")

    df = tables.get("overall")
    if df is None or df.empty:
        return

    # ---- Plot: macro-F1 across all experiments × all tasks ----
    fig, ax = plt.subplots(figsize=(13, 6))
    pivot = df.pivot_table(
        index="experiment", columns="task",
        values="test_f1_macro", aggfunc="first",
    ).reindex([e["id"] for e in EXPERIMENTS])

    # Order columns explicitly
    col_order = [c for c in ("binary", "category", "multiclass") if c in pivot.columns]
    pivot = pivot[col_order]
    pivot.plot(kind="bar", ax=ax, width=0.8)
    ax.set_ylabel("Test macro-F1")
    ax.set_title("Test macro-F1 across experiments and tasks")
    ax.set_ylim(0, 1.0)
    ax.set_xlabel("")
    ax.legend(title="Task")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figures" / "overall_comparison_bar.png", dpi=140)
    plt.close(fig)
    log("  overall_comparison_bar.png  ✓")

    # ---- Plot: feature importance (RF) — pick the best RF run ----
    df_mc_rf = df[(df["task"] == "multiclass") & (df["model"] == "RF")]
    if not df_mc_rf.empty:
        best_eid = df_mc_rf.loc[df_mc_rf["test_f1_macro"].idxmax(), "experiment"]
        fi_path  = OUTPUT_DIR / "metrics" / f"{best_eid}_feature_importance.csv"
        if fi_path.exists():
            fi = pd.read_csv(fi_path).set_index("feature")
            top = fi.head(20)
            fig, ax = plt.subplots(figsize=(9, 8))
            top["importance"].iloc[::-1].plot(kind="barh", ax=ax, color="steelblue")
            ax.set_xlabel("Mean decrease in entropy (RF importance)")
            ax.set_title(f"Top 20 features — {best_eid}")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "figures" / "feature_importance_rf.png", dpi=140)
            plt.close(fig)
            log("  feature_importance_rf.png  ✓")

    # ---- Plot: SMOTE effect on per-class F1 (rarest classes) ----
    mf = tables.get("minority_focus")
    if mf is not None and not mf.empty:
        # Group by (model, feature_set) and plot a small grid
        groups = list(mf.groupby(["model", "feature_set"]))
        n_groups = len(groups)
        if n_groups > 0:
            cols = 2
            rows = (n_groups + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(13, 4 * rows), squeeze=False)
            for idx, ((model, fs), grp) in enumerate(groups):
                ax = axes[idx // cols][idx % cols]
                x = np.arange(len(grp))
                width = 0.4
                ax.bar(x - width/2, grp["f1_original"], width, label="Original", color="#888")
                ax.bar(x + width/2, grp["f1_smote"],    width, label="SMOTE",    color="#3a86ff")
                ax.set_xticks(x)
                ax.set_xticklabels(grp["class"], rotation=30, ha="right", fontsize=8)
                ax.set_ylabel("F1")
                ax.set_ylim(0, 1.0)
                ax.set_title(f"{model} {fs} — minority-class F1")
                ax.grid(axis="y", alpha=0.3)
                ax.legend(fontsize=8)
            # Hide any unused subplots
            for idx in range(n_groups, rows * cols):
                axes[idx // cols][idx % cols].axis("off")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "figures" / "smote_effect.png", dpi=140)
            plt.close(fig)
            log("  smote_effect.png  ✓")

In [ ]:
# %% ============================================================
#                POST-RUN: SUMMARY MARKDOWN
# ================================================================
def write_summary_markdown(tables: dict, meta: dict) -> None:
    section("WRITING summary.md")

    df = tables.get("overall")
    if df is None or df.empty:
        log("No data to summarize.")
        return

    lines = ["# Phase 4 Summary — Supervised Layer\n"]
    lines.append(f"_Generated: {datetime.now():%Y-%m-%d %H:%M:%S}_\n")
    lines.append("")

    # ---- Best overall ----
    lines.append("## 1. Best Experiment Overall (19-class task)\n")
    df_mc = df[df["task"] == "multiclass"].copy()
    if not df_mc.empty:
        best_f1  = df_mc.loc[df_mc["test_f1_macro"].idxmax()]
        best_mcc = df_mc.loc[df_mc["test_mcc"].idxmax()]
        lines.append(f"- **By macro-F1**: `{best_f1['experiment']}` "
                     f"({best_f1['model']} / {best_f1['feature_set']} / {best_f1['data']}) "
                     f"— test F1_macro = **{best_f1['test_f1_macro']:.4f}**, "
                     f"MCC = **{best_f1['test_mcc']:.4f}**, "
                     f"acc = {best_f1['test_accuracy']:.4f}")
        lines.append(f"- **By MCC**: `{best_mcc['experiment']}` "
                     f"— test MCC = **{best_mcc['test_mcc']:.4f}**")
    lines.append("")

    # ---- Best per task ----
    lines.append("## 2. Best Model per Classification Task\n")
    for task in ("binary", "category", "multiclass"):
        sub = df[df["task"] == task]
        if sub.empty:
            continue
        best = sub.loc[sub["test_f1_macro"].idxmax()]
        lines.append(f"- **{task}**: `{best['experiment']}` "
                     f"— F1_macro={best['test_f1_macro']:.4f}, "
                     f"MCC={best['test_mcc']:.4f}, "
                     f"acc={best['test_accuracy']:.4f}")
    lines.append("")

    # ---- SMOTE impact ----
    lines.append("## 3. SMOTETomek Impact\n")
    sc = tables.get("smote_comparison")
    if sc is not None and not sc.empty:
        sub = sc[(sc["task"] == "multiclass") & (sc["metric"] == "test_f1_macro")]
        for _, r in sub.iterrows():
            arrow = "↑" if r["delta"] > 0 else "↓"
            lines.append(f"- {r['model']} / {r['feature_set']} (19-class F1_macro): "
                         f"{r['original']:.4f} → {r['smote']:.4f} ({arrow} {r['delta']:+.4f})")
        # net direction
        net_pos = (sub["delta"] > 0).sum()
        lines.append("")
        lines.append(f"**Net effect on 19-class macro-F1:** "
                     f"{net_pos}/{len(sub)} configurations improved with SMOTETomek.")
    lines.append("")

    # ---- Hardest boundaries ----
    lines.append("## 4. Hardest Classification Boundaries\n")
    lines.append("Off-diagonal cells highlighted in red on the 19-class confusion matrices:")
    for a, b in HARD_BOUNDARIES:
        lines.append(f"- `{a}` ↔ `{b}`")
    lines.append("")
    lines.append("Inspect `figures/cm_<EID>_19class.png` for confusion magnitudes.")
    lines.append("")

    # ---- Top features ----
    lines.append("## 5. Feature Importance — Top 10 (RF)\n")
    if not df_mc.empty:
        df_mc_rf = df_mc[df_mc["model"] == "RF"]
        if not df_mc_rf.empty:
            best_rf = df_mc_rf.loc[df_mc_rf["test_f1_macro"].idxmax()]
            fi_path = OUTPUT_DIR / "metrics" / f"{best_rf['experiment']}_feature_importance.csv"
            if fi_path.exists():
                fi = pd.read_csv(fi_path)
                lines.append(f"From **{best_rf['experiment']}** "
                             f"({best_rf['feature_set']} / {best_rf['data']}):\n")
                lines.append("| Rank | Feature | Importance |")
                lines.append("|------|---------|-----------|")
                for i, r in fi.head(10).iterrows():
                    lines.append(f"| {i+1} | `{r['feature']}` | {r['importance']:.4f} |")
                lines.append("")
                # Compare with Yacoubi's SHAP top features (from literature review)
                yacoubi_top = ["IAT", "Rate", "Header_Length", "Srate"]
                top10 = fi.head(10)["feature"].tolist()
                overlap = [f for f in yacoubi_top if f in top10]
                lines.append(f"**Overlap with Yacoubi et al. SHAP top-4** "
                             f"({yacoubi_top}): {overlap if overlap else 'none'}")
    lines.append("")

    # ---- Yacoubi comparison ----
    lines.append("## 6. Comparison with Yacoubi et al. Benchmarks\n")
    lines.append("> Yacoubi reported on **raw (non-deduplicated)** data; our metrics are on "
                 "**deduplicated** data so we expect lower headline accuracy. The gap is the "
                 "duplicate-leakage correction — a methodological contribution, not a regression.\n")
    lines.append("| Model | Yacoubi Acc. | Our Best Acc. (19-class) |")
    lines.append("|-------|--------------|--------------------------|")
    if not df_mc.empty:
        for label, ours_filter in [
            ("RF (entropy)",  df_mc[df_mc["model"] == "RF"]),
            ("XGBoost",       df_mc[df_mc["model"] == "XGB"]),
        ]:
            if ours_filter.empty:
                continue
            ours_best = ours_filter.loc[ours_filter["test_accuracy"].idxmax()]
            yac = YACOUBI_BENCHMARKS.get(label, float("nan"))
            lines.append(f"| {label} | {yac:.4f} | {ours_best['test_accuracy']:.4f} "
                         f"({ours_best['experiment']}) |")
    lines.append("")

    # ---- Recommendation for Phase 6 fusion ----
    lines.append("## 7. Recommendation for Phase 6 Fusion Engine\n")
    if not df_mc.empty:
        best_f1 = df_mc.loc[df_mc["test_f1_macro"].idxmax()]
        lines.append(
            f"Use **`{best_f1['experiment']}`** "
            f"({best_f1['model']} / {best_f1['feature_set']} / {best_f1['data']}) "
            f"as the supervised input to the 4-case fusion engine.\n"
        )
        lines.append(
            "- Probability vectors are saved as "
            f"`predictions/{best_f1['experiment']}_val_proba.npy` and "
            f"`predictions/{best_f1['experiment']}_test_proba.npy` "
            "(shape: N × 19)."
        )
        lines.append(
            f"- The trained model is at "
            f"`models/{exp_filename(next(e for e in EXPERIMENTS if e['id'] == best_f1['experiment']))}.pkl`."
        )
    lines.append("")

    # ---- Files generated ----
    lines.append("## 8. Generated Artifacts\n")
    lines.append("```")
    lines.append("results/supervised/")
    lines.append("├── metrics/")
    lines.append("│   ├── overall_comparison.csv")
    lines.append("│   ├── best_classification_report.csv")
    lines.append("│   ├── smote_comparison.csv")
    lines.append("│   ├── minority_focus.csv")
    lines.append("│   ├── E*_feature_importance.csv")
    lines.append("│   └── E*_classification_report_test.json")
    lines.append("├── figures/")
    lines.append("│   ├── cm_E*_19class.png")
    lines.append("│   ├── cm_E*_6class.png")
    lines.append("│   ├── feature_importance_rf.png")
    lines.append("│   ├── overall_comparison_bar.png")
    lines.append("│   └── smote_effect.png")
    lines.append("├── models/                  (8 × .pkl — 19-class)")
    lines.append("├── predictions/             (val/test pred + proba per task)")
    lines.append("└── summary.md               (this file)")
    lines.append("```")
    lines.append("")

    summary_path = OUTPUT_DIR / "summary.md"
    summary_path.write_text("\n".join(lines))
    log(f"  summary.md written ({len(lines)} lines)")

In [ ]:
# %% ============================================================
#                            ENTRY POINT
# ================================================================
def save_run_config(meta: dict) -> None:
    """Save experiment configuration for reproducibility."""
    cfg = {
        "timestamp":         datetime.now().isoformat(),
        "random_state":      RANDOM_STATE,
        "n_jobs":            N_JOBS,
        "skip_if_exists":    SKIP_IF_EXISTS,
        "save_float32_proba": SAVE_FLOAT32_PROBA,
        "rf_params":         RF_PARAMS,
        "xgb_params_base":   XGB_PARAMS_BASE,
        "experiments":       EXPERIMENTS,
        "tasks":             [{"name": t["name"], "n_classes": t["n_classes"],
                                "label_col": t["label_col"]} for t in TASKS],
        "yacoubi_benchmarks": YACOUBI_BENCHMARKS,
        "hard_boundaries":   HARD_BOUNDARIES,
        "feature_names_reduced": meta["feature_names_reduced"],
        "feature_names_full":    meta["feature_names_full"],
    }
    with open(OUTPUT_DIR / "config.json", "w") as f:
        json.dump(cfg, f, indent=2, default=str)


def main() -> None:
    section("PHASE 4 — SUPERVISED MODEL TRAINING")
    log(f"Preprocessed dir: {PREPROCESSED_DIR.resolve()}")
    log(f"Output dir:       {OUTPUT_DIR.resolve()}")

    make_dirs()
    meta = load_metadata()
    init_tasks(meta)
    save_run_config(meta)

    log(f"Loaded metadata: "
        f"{len(meta['multiclass_names'] or [])} multiclass, "
        f"{len(meta['category_names'] or [])} category, "
        f"{len(meta['binary_names'] or [])} binary classes")

    # ---- run all 8 × 3 = 24 (experiment, task) pairs ----
    all_results = run_all_experiments(meta)
    log(f"All experiments complete. Total result rows: {len(all_results)}")

    # ---- post-processing ----
    tables = build_comparison_tables(meta)
    make_summary_plots(tables, meta)
    write_summary_markdown(tables, meta)

    section("DONE")
    log("Bring results/supervised/summary.md and the metrics/ folder back to the project chat.")


if __name__ == "__main__":
    main()

## Phase 5 — Unsupervised Model Training (Layer 2)

Trains an **Autoencoder** and an **Isolation Forest** on benign-only traffic.

**Autoencoder architecture** (symmetric encoder-decoder):
```
Input (44) → Dense(32)+BN+Drop(0.2) → Dense(16)+BN+Drop(0.1)
           → Bottleneck(8)
           → Dense(16)+BN → Dense(32)+BN → Output(44, linear)
```
- Loss: MSE (reconstruction error); Optimizer: Adam lr=1e-3
- Early stopping (patience=10) + ReduceLROnPlateau

**Isolation Forest**: 200 estimators, contamination=0.05

**Threshold selection** (on validation set): p90/p95/p99/mean±2σ/mean±3σ
Best threshold = highest F1 on binary benign-vs-attack classification.

**Outputs** (`results/unsupervised/`): models (.keras / .pkl), MSE score
arrays (.npy), per-class detection rates, zero-day preview, ROC curves.

**Prerequisite**: Phase 3 must have written `preprocessed/full_features/`.
**TensorFlow required**: `pip install tensorflow` (or `tensorflow-metal` on Apple Silicon).

---

*Module docstring:*
```
Phase 5 — Unsupervised Model Training (Autoencoder + Isolation Forest)
=======================================================================
Hybrid Supervised-Unsupervised Anomaly Detection Framework for IoMT Networks
CICIoMT2024 Dataset

Layer 2 of the hybrid framework:
- Autoencoder: trained on benign traffic, flags anomalies via reconstruction error
- Isolation Forest: tree-based anomaly detection trained on benign traffic

Inputs:
    preprocessed/full_features/{X_train,X_val,X_test}.npy   # 44 features
    preprocessed/full_features/{y_train,y_val,y_test}.csv

Outputs:
    results/unsup…
```

In [ ]:
# %% ============================================================================
# 0 · CONFIGURATION
# ===============================================================================
import os
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---- paths ----
PREPROCESSED_DIR = Path("./preprocessed/")
SUPERVISED_DIR   = Path("./results/supervised/")
OUTPUT_DIR       = Path("./results/unsupervised/")

RANDOM_STATE = 42

# ---- Autoencoder hyperparameters ----
AE_ARCHITECTURE     = [44, 32, 16, 8, 16, 32, 44]   # symmetric encoder-decoder
AE_EPOCHS           = 100
AE_BATCH_SIZE       = 512
AE_LEARNING_RATE    = 1e-3
AE_PATIENCE         = 10
AE_PREDICT_BATCH    = 8192

# ---- Isolation Forest hyperparameters ----
IF_N_ESTIMATORS  = 200
IF_CONTAMINATION = 0.05

# ---- evaluation / plotting ----
THRESHOLD_NAMES = ["p90", "p95", "p99", "mean_2std", "mean_3std"]
ZERO_DAY_TARGETS = [
    "Recon_Ping_Sweep",
    "Recon_VulScan",
    "MQTT_Malformed_Data",
    "MQTT_DoS_Connect_Flood",
    "ARP_Spoofing",
]
DPI = 150

In [ ]:
# %% ============================================================================
# 1 · IMPORTS, REPRODUCIBILITY, OUTPUT DIRS
# ===============================================================================
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"     # quiet TF info logs

import random
random.seed(RANDOM_STATE)

import numpy as np
np.random.seed(RANDOM_STATE)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# TensorFlow — guarded import with helpful message on failure
try:
    import tensorflow as tf
    from tensorflow.keras import layers, Model, callbacks
except ImportError as e:
    raise ImportError(
        "TensorFlow is required for this phase.\n"
        "Install with:    pip install tensorflow\n"
        "On Apple Silicon you can also try:    pip install tensorflow-metal\n"
        f"Original error: {e}"
    )

tf.random.set_seed(RANDOM_STATE)
try:
    tf.keras.utils.set_random_seed(RANDOM_STATE)   # TF >= 2.7
except Exception:
    pass

# ---- GPU detection (M4 Metal or CUDA) ----
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"[init] {len(gpus)} GPU device(s) detected: {[g.name for g in gpus]}")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as exc:
            print(f"[init] could not set memory growth on {gpu.name}: {exc}")
else:
    print("[init] No GPU detected — training will run on CPU (fine for this size).")

print(f"[init] TensorFlow version: {tf.__version__}")
print(f"[init] NumPy version:      {np.__version__}")

# ---- output directory tree ----
DIRS = {
    "root":     OUTPUT_DIR,
    "models":   OUTPUT_DIR / "models",
    "scores":   OUTPUT_DIR / "scores",
    "metrics":  OUTPUT_DIR / "metrics",
    "figures":  OUTPUT_DIR / "figures",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# %% ============================================================================
# 2 · DATA LOADING & BENIGN EXTRACTION
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 2 · DATA LOADING")
print("=" * 70)

t0 = time.time()

# ---- preprocessed config (feature names, label mappings) ----
with open(PREPROCESSED_DIR / "config.json") as f:
    pp_cfg = json.load(f)
with open(PREPROCESSED_DIR / "label_encoders.json") as f:
    label_encoders = json.load(f)

# Try to extract feature names — fall back to indexed names if shape disagrees
feat_names_raw = (
    pp_cfg.get("full_features")
    or pp_cfg.get("feature_names_full")
    or pp_cfg.get("feature_names")
)
if feat_names_raw is None or len(feat_names_raw) != 44:
    feat_names_raw = [f"f{i:02d}" for i in range(44)]
    print("[data] config.json did not contain 44-element feature list; using f00..f43.")
FEATURE_NAMES = list(feat_names_raw)

# ---- load arrays ----
X_train = np.load(PREPROCESSED_DIR / "full_features" / "X_train.npy").astype(np.float32)
X_val   = np.load(PREPROCESSED_DIR / "full_features" / "X_val.npy").astype(np.float32)
X_test  = np.load(PREPROCESSED_DIR / "full_features" / "X_test.npy").astype(np.float32)

y_train = pd.read_csv(PREPROCESSED_DIR / "full_features" / "y_train.csv")
y_val   = pd.read_csv(PREPROCESSED_DIR / "full_features" / "y_val.csv")
y_test  = pd.read_csv(PREPROCESSED_DIR / "full_features" / "y_test.csv")

# pick the label column
def _label_col(df):
    for c in ("label", "Label", "multiclass", "y"):
        if c in df.columns:
            return c
    return df.columns[0]

LBL = _label_col(y_train)
y_train_lbl = y_train[LBL].astype(str).values
y_val_lbl   = y_val[LBL].astype(str).values
y_test_lbl  = y_test[LBL].astype(str).values

print(f"[data] X_train: {X_train.shape}   X_val: {X_val.shape}   X_test: {X_test.shape}")
print(f"[data] label column: '{LBL}'")
assert X_train.shape[1] == 44, f"expected 44 features, got {X_train.shape[1]}"

# ---- benign extraction ----
benign_train_mask = (y_train_lbl == "Benign")
X_benign_full = X_train[benign_train_mask]
print(f"[data] Benign rows in train split: {X_benign_full.shape[0]:,} "
      f"({100*benign_train_mask.mean():.2f}% of training data)")

# ---- 80/20 AE-train / AE-val split (benign-only) ----
rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(X_benign_full.shape[0])
split_idx = int(0.8 * X_benign_full.shape[0])
X_benign_train = X_benign_full[perm[:split_idx]]
X_benign_val   = X_benign_full[perm[split_idx:]]
print(f"[data] AE-train (benign): {X_benign_train.shape}")
print(f"[data] AE-val   (benign): {X_benign_val.shape}")

# ---- binary anomaly labels (benign=0, anything else=1) ----
y_val_bin  = (y_val_lbl != "Benign").astype(np.int8)
y_test_bin = (y_test_lbl != "Benign").astype(np.int8)
print(f"[data] val binary breakdown:  {int((y_val_bin==0).sum()):,} normal / "
      f"{int((y_val_bin==1).sum()):,} anomaly")
print(f"[data] test binary breakdown: {int((y_test_bin==0).sum()):,} normal / "
      f"{int((y_test_bin==1).sum()):,} anomaly")
print(f"[data] loading took {time.time() - t0:.1f}s")

In [ ]:
# %% ============================================================================
# 2.5 · FEATURE SCALING (fix Phase 3's partial scaling)
# ===============================================================================
# Phase 3 left several features unscaled (e.g. Tot_sum std~5000, Weight std~1500),
# while others were StandardScaled and a third group is binary protocol flags.
# Tree-based XGBoost (Phase 4) is scale-invariant so this didn't matter there.
# AE and IF are NOT scale-invariant — without proper scaling, the AE loss is
# dominated by 1–2 large-magnitude features and never learns the rest, which
# tanks Recon detection. Fit on benign-train (consistent with AE/IF training set)
# and apply everywhere; save for Phase 6 to use on new data.
print("\n" + "=" * 70)
print("SECTION 2.5 · FEATURE SCALING")
print("=" * 70)

print(f"[scale] before — global mean={X_benign_train.mean():.4f}, "
      f"std={X_benign_train.std():.4f}, max={X_benign_train.max():.4f}")

scaler = StandardScaler()
scaler.fit(X_benign_train)
X_benign_train = scaler.transform(X_benign_train).astype(np.float32)
X_benign_val   = scaler.transform(X_benign_val).astype(np.float32)
X_val          = scaler.transform(X_val).astype(np.float32)
X_test         = scaler.transform(X_test).astype(np.float32)
X_train_shape = X_train.shape   # remember for config dump before freeing
del X_train  # free ~635 MB — no longer needed after benign extraction

print(f"[scale] after  — benign-train mean={X_benign_train.mean():.6f}, "
      f"std={X_benign_train.std():.6f}")
print(f"[scale] after  — X_test       mean={X_test.mean():.4f}, "
      f"std={X_test.std():.4f}  (attack samples are out-of-distribution → expected)")

In [ ]:
# %% ============================================================================
# 3 · AUTOENCODER ARCHITECTURE
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 3 · AUTOENCODER ARCHITECTURE")
print("=" * 70)

def build_autoencoder(input_dim: int = 44):
    """Symmetric deep AE: 44 -> 32 -> 16 -> 8 -> 16 -> 32 -> 44 ."""
    inp = layers.Input(shape=(input_dim,), name="input")

    # Encoder
    x = layers.Dense(32, activation="relu", name="enc_dense_32")(inp)
    x = layers.BatchNormalization(name="enc_bn_32")(x)
    x = layers.Dropout(0.2, name="enc_drop_32")(x)
    x = layers.Dense(16, activation="relu", name="enc_dense_16")(x)
    x = layers.BatchNormalization(name="enc_bn_16")(x)
    x = layers.Dropout(0.1, name="enc_drop_16")(x)
    bottleneck = layers.Dense(8, activation="relu", name="bottleneck")(x)

    # Decoder
    x = layers.Dense(16, activation="relu", name="dec_dense_16")(bottleneck)
    x = layers.BatchNormalization(name="dec_bn_16")(x)
    x = layers.Dense(32, activation="relu", name="dec_dense_32")(x)
    x = layers.BatchNormalization(name="dec_bn_32")(x)
    out = layers.Dense(input_dim, activation="linear", name="reconstruction")(x)

    autoencoder = Model(inputs=inp, outputs=out, name="autoencoder")
    encoder     = Model(inputs=inp, outputs=bottleneck, name="encoder")

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=AE_LEARNING_RATE),
        loss="mse",
    )
    return autoencoder, encoder

autoencoder, encoder = build_autoencoder(44)
autoencoder.summary(print_fn=lambda s: print("  " + s))

In [ ]:
# %% ============================================================================
# 4 · AUTOENCODER TRAINING
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 4 · AUTOENCODER TRAINING (benign-only)")
print("=" * 70)

cb_list = [
    callbacks.EarlyStopping(
        monitor="val_loss", patience=AE_PATIENCE,
        restore_best_weights=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5,
        min_lr=1e-6, verbose=1,
    ),
]

t_train_start = time.time()
history = autoencoder.fit(
    X_benign_train, X_benign_train,           # input == target (reconstruction)
    validation_data=(X_benign_val, X_benign_val),
    epochs=AE_EPOCHS,
    batch_size=AE_BATCH_SIZE,
    callbacks=cb_list,
    verbose=2,
    shuffle=True,
)
ae_train_time = time.time() - t_train_start
print(f"[train] AE training time: {ae_train_time:.1f}s "
      f"({ae_train_time/60:.2f} min) over {len(history.history['loss'])} epochs")
print(f"[train] best val_loss: {min(history.history['val_loss']):.6f}")

In [ ]:
# %% ============================================================================
# 5 · RECONSTRUCTION ERROR COMPUTATION
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 5 · RECONSTRUCTION ERROR (per-sample MSE)")
print("=" * 70)

def compute_mse_batched(model, X, batch_size: int = AE_PREDICT_BATCH):
    """Return per-sample MSE. Keras handles batching internally."""
    X_hat = model.predict(X, batch_size=batch_size, verbose=0)
    return np.mean((X - X_hat) ** 2, axis=1).astype(np.float32)

t_score = time.time()
mse_benign_train = compute_mse_batched(autoencoder, X_benign_train)
mse_benign_val   = compute_mse_batched(autoencoder, X_benign_val)
ae_val_mse       = compute_mse_batched(autoencoder, X_val)
ae_test_mse      = compute_mse_batched(autoencoder, X_test)
ae_score_time    = time.time() - t_score
print(f"[score] AE scoring time: {ae_score_time:.1f}s")

print(f"[score] benign MSE — mean={mse_benign_val.mean():.6f}  "
      f"std={mse_benign_val.std():.6f}  "
      f"p95={np.percentile(mse_benign_val,95):.6f}  "
      f"p99={np.percentile(mse_benign_val,99):.6f}")

# benign error stats (saved later)
benign_error_stats = {
    "ae_train_benign": {
        "mean":  float(mse_benign_train.mean()),
        "std":   float(mse_benign_train.std()),
        "min":   float(mse_benign_train.min()),
        "max":   float(mse_benign_train.max()),
        "p50":   float(np.percentile(mse_benign_train, 50)),
        "p90":   float(np.percentile(mse_benign_train, 90)),
        "p95":   float(np.percentile(mse_benign_train, 95)),
        "p99":   float(np.percentile(mse_benign_train, 99)),
    },
    "ae_val_benign": {
        "mean":  float(mse_benign_val.mean()),
        "std":   float(mse_benign_val.std()),
        "min":   float(mse_benign_val.min()),
        "max":   float(mse_benign_val.max()),
        "p50":   float(np.percentile(mse_benign_val, 50)),
        "p90":   float(np.percentile(mse_benign_val, 90)),
        "p95":   float(np.percentile(mse_benign_val, 95)),
        "p99":   float(np.percentile(mse_benign_val, 99)),
    },
}

In [ ]:
# %% ============================================================================
# 6 · ANOMALY THRESHOLD SELECTION
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 6 · THRESHOLD SELECTION (on validation set)")
print("=" * 70)

thresholds = {
    "p90":       float(np.percentile(mse_benign_val, 90)),
    "p95":       float(np.percentile(mse_benign_val, 95)),
    "p99":       float(np.percentile(mse_benign_val, 99)),
    "mean_2std": float(mse_benign_val.mean() + 2 * mse_benign_val.std()),
    "mean_3std": float(mse_benign_val.mean() + 3 * mse_benign_val.std()),
}

threshold_eval_rows = []
for name in THRESHOLD_NAMES:
    thr = thresholds[name]
    pred = (ae_val_mse > thr).astype(np.int8)
    p, r, f1, _ = precision_recall_fscore_support(
        y_val_bin, pred, average="binary", zero_division=0
    )
    # FPR and TPR
    tn = int(((pred == 0) & (y_val_bin == 0)).sum())
    fp = int(((pred == 1) & (y_val_bin == 0)).sum())
    fn = int(((pred == 0) & (y_val_bin == 1)).sum())
    tp = int(((pred == 1) & (y_val_bin == 1)).sum())
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    threshold_eval_rows.append({
        "threshold_name":  name,
        "threshold_value": thr,
        "precision":       float(p),
        "recall":          float(r),
        "f1":              float(f1),
        "fpr":             float(fpr),
        "tpr":             float(tpr),
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
    })

threshold_eval_df = pd.DataFrame(threshold_eval_rows)
print("\nThreshold evaluation on VALIDATION:")
print(threshold_eval_df.to_string(index=False))

best_row = threshold_eval_df.loc[threshold_eval_df["f1"].idxmax()]
BEST_THRESHOLD_NAME  = str(best_row["threshold_name"])
BEST_THRESHOLD_VALUE = float(best_row["threshold_value"])
print(f"\n[threshold] selected '{BEST_THRESHOLD_NAME}' "
      f"= {BEST_THRESHOLD_VALUE:.6f}  (F1={best_row['f1']:.4f})")

# binary AE predictions at best threshold
ae_val_binary  = (ae_val_mse  > BEST_THRESHOLD_VALUE).astype(np.int8)
ae_test_binary = (ae_test_mse > BEST_THRESHOLD_VALUE).astype(np.int8)

In [ ]:
# %% ============================================================================
# 7 · ISOLATION FOREST TRAINING
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 7 · ISOLATION FOREST TRAINING (benign-only)")
print("=" * 70)

iso_forest = IsolationForest(
    n_estimators=IF_N_ESTIMATORS,
    contamination=IF_CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)

t_if_train = time.time()
iso_forest.fit(X_benign_train)
if_train_time = time.time() - t_if_train
print(f"[train] IF training time: {if_train_time:.1f}s")

t_if_score = time.time()
# higher decision_function = more normal; lower = more anomalous
if_val_scores  = iso_forest.decision_function(X_val).astype(np.float32)
if_test_scores = iso_forest.decision_function(X_test).astype(np.float32)
# predict: -1 = anomaly, +1 = normal
if_val_pred  = iso_forest.predict(X_val)
if_test_pred = iso_forest.predict(X_test)
if_val_binary  = (if_val_pred  == -1).astype(np.int8)
if_test_binary = (if_test_pred == -1).astype(np.int8)
if_score_time = time.time() - t_if_score
print(f"[score] IF scoring time: {if_score_time:.1f}s")

In [ ]:
# %% ============================================================================
# 8 · BINARY ANOMALY DETECTION METRICS
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 8 · BINARY ANOMALY DETECTION METRICS")
print("=" * 70)

# AE: higher MSE = more anomalous (positive class)
ae_val_auc  = roc_auc_score(y_val_bin,  ae_val_mse)
ae_test_auc = roc_auc_score(y_test_bin, ae_test_mse)
# IF: lower decision_function = more anomalous → use negative for ROC
if_val_auc  = roc_auc_score(y_val_bin,  -if_val_scores)
if_test_auc = roc_auc_score(y_test_bin, -if_test_scores)

print(f"[AUC] AE:  val={ae_val_auc:.4f}   test={ae_test_auc:.4f}")
print(f"[AUC] IF:  val={if_val_auc:.4f}   test={if_test_auc:.4f}")

# FPR at 95% TPR
def fpr_at_tpr(y_true, scores, target_tpr=0.95):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return float(np.interp(target_tpr, tpr, fpr))

ae_fpr95_test = fpr_at_tpr(y_test_bin, ae_test_mse,    0.95)
if_fpr95_test = fpr_at_tpr(y_test_bin, -if_test_scores, 0.95)
print(f"[FPR@95%TPR] AE test: {ae_fpr95_test:.4f}    IF test: {if_fpr95_test:.4f}")

# classification reports (test set)
ae_cls_report = classification_report(
    y_test_bin, ae_test_binary,
    target_names=["normal", "anomaly"], output_dict=True, zero_division=0,
)
if_cls_report = classification_report(
    y_test_bin, if_test_binary,
    target_names=["normal", "anomaly"], output_dict=True, zero_division=0,
)
print("\nAutoencoder classification report (test):")
print(classification_report(y_test_bin, ae_test_binary,
                            target_names=["normal", "anomaly"], zero_division=0))
print("Isolation Forest classification report (test):")
print(classification_report(y_test_bin, if_test_binary,
                            target_names=["normal", "anomaly"], zero_division=0))

In [ ]:
# %% ============================================================================
# 9 · PER-CLASS DETECTION RATES (19 classes × 5 thresholds)
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 9 · PER-CLASS DETECTION RATES")
print("=" * 70)

CLASSES = sorted(np.unique(y_test_lbl).tolist())
print(f"[classes] {len(CLASSES)} classes: {CLASSES}")

# AE: detection rate at each threshold per class (computed on TEST set)
detection_records = []
for cls in CLASSES:
    mask = (y_test_lbl == cls)
    n_cls = int(mask.sum())
    if n_cls == 0:
        continue
    cls_errors = ae_test_mse[mask]
    row = {"class": cls, "n_samples": n_cls, "model": "Autoencoder"}
    for tname in THRESHOLD_NAMES:
        thr = thresholds[tname]
        rate = float((cls_errors > thr).mean())
        row[tname] = rate
    detection_records.append(row)

# IF: just one detection rate (it has its own internal threshold) — record anyway
for cls in CLASSES:
    mask = (y_test_lbl == cls)
    n_cls = int(mask.sum())
    if n_cls == 0:
        continue
    rate = float(if_test_binary[mask].mean())
    row = {
        "class": cls, "n_samples": n_cls, "model": "IsolationForest",
        "p90": rate, "p95": rate, "p99": rate,
        "mean_2std": rate, "mean_3std": rate,
    }
    detection_records.append(row)

detection_df = pd.DataFrame(detection_records)
print("\nPer-class detection rate (test, AE first then IF):")
print(detection_df.round(3).to_string(index=False))

# pivot for heatmap (AE only, all thresholds)
ae_detection_df = detection_df[detection_df["model"] == "Autoencoder"].copy()
detection_pivot = ae_detection_df.set_index("class")[THRESHOLD_NAMES]

In [ ]:
# %% ============================================================================
# 10 · ZERO-DAY SIMULATION PREVIEW
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 10 · ZERO-DAY SIMULATION (5 targets)")
print("=" * 70)

zero_day_records = []
for tgt in ZERO_DAY_TARGETS:
    mask = (y_test_lbl == tgt)
    n_tgt = int(mask.sum())
    if n_tgt == 0:
        print(f"[zero-day] '{tgt}' not present in test set — skipped.")
        continue

    # AE detection rates at each threshold
    ae_rates = {
        tname: float((ae_test_mse[mask] > thresholds[tname]).mean())
        for tname in THRESHOLD_NAMES
    }
    # IF detection rate (using its built-in -1/+1)
    if_rate = float(if_test_binary[mask].mean())

    record = {
        "target": tgt,
        "n_samples": n_tgt,
        "ae_p90":       ae_rates["p90"],
        "ae_p95":       ae_rates["p95"],
        "ae_p99":       ae_rates["p99"],
        "ae_mean_2std": ae_rates["mean_2std"],
        "ae_mean_3std": ae_rates["mean_3std"],
        "ae_best_thr":  ae_rates[BEST_THRESHOLD_NAME],
        "if_recall":    if_rate,
    }
    zero_day_records.append(record)

zero_day_df = pd.DataFrame(zero_day_records)
print(zero_day_df.round(3).to_string(index=False))

# Per-class detection preview (NOT a true H2 verdict — see Phase 6 for that)
hits_70 = int((zero_day_df["ae_best_thr"] >= 0.70).sum())
total_targets = len(zero_day_df)
print(f"[preview] AE @ best threshold ≥70% on {hits_70}/{total_targets} targets "
      f"(preview only — true zero-day H2 evaluation deferred to Phase 6)")

In [ ]:
# %% ============================================================================
# 11 · MODEL COMPARISON TABLE
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 11 · MODEL COMPARISON")
print("=" * 70)

# Filter out Benign — its "detection rate" is FPR, not recall
ae_attack_only = ae_detection_df[ae_detection_df["class"] != "Benign"]
ae_per_class_recall = ae_attack_only[BEST_THRESHOLD_NAME].mean()
if_per_class_recall = (
    detection_df[(detection_df["model"] == "IsolationForest") &
                 (detection_df["class"] != "Benign")]["p90"].mean()
)

comparison_df = pd.DataFrame([
    {
        "metric":             "AUC-ROC (test)",
        "Autoencoder":        round(ae_test_auc, 4),
        "IsolationForest":    round(if_test_auc, 4),
    },
    {
        "metric":             "AUC-ROC (val)",
        "Autoencoder":        round(ae_val_auc, 4),
        "IsolationForest":    round(if_val_auc, 4),
    },
    {
        "metric":             "FPR @ 95%TPR (test)",
        "Autoencoder":        round(ae_fpr95_test, 4),
        "IsolationForest":    round(if_fpr95_test, 4),
    },
    {
        "metric":             "Anomaly precision (test)",
        "Autoencoder":        round(ae_cls_report["anomaly"]["precision"], 4),
        "IsolationForest":    round(if_cls_report["anomaly"]["precision"], 4),
    },
    {
        "metric":             "Anomaly recall (test)",
        "Autoencoder":        round(ae_cls_report["anomaly"]["recall"], 4),
        "IsolationForest":    round(if_cls_report["anomaly"]["recall"], 4),
    },
    {
        "metric":             "Anomaly F1 (test)",
        "Autoencoder":        round(ae_cls_report["anomaly"]["f1-score"], 4),
        "IsolationForest":    round(if_cls_report["anomaly"]["f1-score"], 4),
    },
    {
        "metric":             "Per-class avg recall",
        "Autoencoder":        round(float(ae_per_class_recall), 4),
        "IsolationForest":    round(float(if_per_class_recall), 4),
    },
    {
        "metric":             "Training time (s)",
        "Autoencoder":        round(ae_train_time, 1),
        "IsolationForest":    round(if_train_time, 1),
    },
    {
        "metric":             "Scoring time val+test (s)",
        "Autoencoder":        round(ae_score_time, 1),
        "IsolationForest":    round(if_score_time, 1),
    },
])
print(comparison_df.to_string(index=False))

In [ ]:
# %% ============================================================================
# 12 · VISUALIZATIONS
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 12 · GENERATING FIGURES")
print("=" * 70)

sns.set_style("whitegrid")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ---- 12.1 Loss curves ----
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(history.history["loss"],     label="train loss", linewidth=2)
ax.plot(history.history["val_loss"], label="val loss",   linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_title("Autoencoder training (benign-only)")
ax.set_yscale("log")
ax.legend()
fig.tight_layout()
fig.savefig(DIRS["figures"] / "ae_loss_curves.png", dpi=DPI)
plt.close(fig)
print("  ✓ ae_loss_curves.png")

# ---- 12.2 Reconstruction error distribution (benign vs attack) ----
fig, ax = plt.subplots(figsize=(12, 6))
benign_test_errs = ae_test_mse[y_test_bin == 0]
attack_test_errs = ae_test_mse[y_test_bin == 1]
clip = float(np.percentile(ae_test_mse, 99.5))   # clip x-axis for readability
bins = np.linspace(0, clip, 100)
ax.hist(np.clip(benign_test_errs, 0, clip), bins=bins, alpha=0.55,
        label=f"Benign (n={len(benign_test_errs):,})", color="#2ca02c", density=True)
ax.hist(np.clip(attack_test_errs, 0, clip), bins=bins, alpha=0.55,
        label=f"Attack (n={len(attack_test_errs):,})", color="#d62728", density=True)
ax.axvline(BEST_THRESHOLD_VALUE, color="black", linestyle="--", linewidth=1.5,
           label=f"threshold={BEST_THRESHOLD_NAME} ({BEST_THRESHOLD_VALUE:.4f})")
ax.set_xlabel("Reconstruction MSE (clipped at p99.5)")
ax.set_ylabel("density")
ax.set_title("Autoencoder reconstruction error — benign vs attack (test)")
ax.legend()
fig.tight_layout()
fig.savefig(DIRS["figures"] / "ae_error_distribution.png", dpi=DPI)
plt.close(fig)
print("  ✓ ae_error_distribution.png")

# ---- 12.3 Per-class reconstruction error boxplot ----
fig, ax = plt.subplots(figsize=(16, 8))
class_order = ["Benign"] + [c for c in CLASSES if c != "Benign"]
data_per_class, labels_per_class = [], []
for cls in class_order:
    mask = (y_test_lbl == cls)
    if mask.sum() > 0:
        # cap each class for readability
        errs = ae_test_mse[mask]
        data_per_class.append(np.clip(errs, 0, np.percentile(ae_test_mse, 99.5)))
        labels_per_class.append(f"{cls}\n(n={int(mask.sum()):,})")
bp = ax.boxplot(data_per_class, labels=labels_per_class, showfliers=False, patch_artist=True)
# colour benign green, others red-ish
for i, patch in enumerate(bp["boxes"]):
    patch.set_facecolor("#2ca02c" if labels_per_class[i].startswith("Benign") else "#d62728")
    patch.set_alpha(0.55)
ax.axhline(BEST_THRESHOLD_VALUE, color="black", linestyle="--", linewidth=1.2,
           label=f"threshold ({BEST_THRESHOLD_NAME})")
ax.set_xlabel("Class")
ax.set_ylabel("Reconstruction MSE (clipped at p99.5)")
ax.set_title("Per-class reconstruction error (test)")
ax.legend(loc="upper right")
plt.setp(ax.get_xticklabels(), rotation=75, ha="right", fontsize=8)
fig.tight_layout()
fig.savefig(DIRS["figures"] / "ae_per_class_boxplot.png", dpi=DPI)
plt.close(fig)
print("  ✓ ae_per_class_boxplot.png")

# ---- 12.4 ROC curves (AE & IF) ----
fig, ax = plt.subplots(figsize=(10, 8))
fpr_ae, tpr_ae, _ = roc_curve(y_test_bin, ae_test_mse)
fpr_if, tpr_if, _ = roc_curve(y_test_bin, -if_test_scores)
ax.plot(fpr_ae, tpr_ae, linewidth=2, label=f"Autoencoder (AUC={ae_test_auc:.4f})")
ax.plot(fpr_if, tpr_if, linewidth=2, label=f"Isolation Forest (AUC={if_test_auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves — anomaly detection on test set")
ax.legend(loc="lower right")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.005])
fig.tight_layout()
fig.savefig(DIRS["figures"] / "roc_curves.png", dpi=DPI)
plt.close(fig)
print("  ✓ roc_curves.png")

# ---- 12.5 Per-class detection rate heatmap (AE) ----
fig, ax = plt.subplots(figsize=(10, max(8, 0.5 * len(detection_pivot))))
sns.heatmap(
    detection_pivot, annot=True, fmt=".2f",
    cmap="RdYlGn", vmin=0, vmax=1, cbar_kws={"label": "Detection rate"},
    ax=ax,
)
ax.set_xlabel("Threshold")
ax.set_ylabel("Class")
ax.set_title("Autoencoder per-class detection rate (test) — 19 classes × 5 thresholds")
fig.tight_layout()
fig.savefig(DIRS["figures"] / "detection_rate_heatmap.png", dpi=DPI)
plt.close(fig)
print("  ✓ detection_rate_heatmap.png")

# ---- 12.6 Isolation Forest score distribution ----
fig, ax = plt.subplots(figsize=(12, 6))
benign_if = if_test_scores[y_test_bin == 0]
attack_if = if_test_scores[y_test_bin == 1]
xlo = float(np.percentile(if_test_scores, 0.5))
xhi = float(np.percentile(if_test_scores, 99.5))
bins = np.linspace(xlo, xhi, 100)
ax.hist(np.clip(benign_if, xlo, xhi), bins=bins, alpha=0.55,
        label=f"Benign (n={len(benign_if):,})", color="#2ca02c", density=True)
ax.hist(np.clip(attack_if, xlo, xhi), bins=bins, alpha=0.55,
        label=f"Attack (n={len(attack_if):,})", color="#d62728", density=True)
ax.axvline(0.0, color="black", linestyle="--", linewidth=1.2,
           label="IF decision boundary (0)")
ax.set_xlabel("Isolation Forest decision_function (higher = more normal)")
ax.set_ylabel("density")
ax.set_title("Isolation Forest score distribution — benign vs attack (test)")
ax.legend()
fig.tight_layout()
fig.savefig(DIRS["figures"] / "if_score_distribution.png", dpi=DPI)
plt.close(fig)
print("  ✓ if_score_distribution.png")

# ---- 12.7 Zero-day detection bar chart ----
if len(zero_day_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(zero_day_df))
    width = 0.35
    ax.bar(x - width/2, zero_day_df["ae_best_thr"], width,
           label=f"AE ({BEST_THRESHOLD_NAME})", color="#1f77b4")
    ax.bar(x + width/2, zero_day_df["if_recall"], width,
           label="Isolation Forest", color="#ff7f0e")
    ax.axhline(0.7, color="green", linestyle="--", linewidth=1, label="H2 target (70%)")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{t}\n(n={n:,})" for t, n in zip(zero_day_df["target"], zero_day_df["n_samples"])],
        rotation=20, ha="right", fontsize=9,
    )
    ax.set_ylabel("Detection rate (recall)")
    ax.set_ylim([0, 1.05])
    ax.set_title("Zero-day simulation preview — recall on held-out attack types")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(DIRS["figures"] / "zero_day_detection.png", dpi=DPI)
    plt.close(fig)
    print("  ✓ zero_day_detection.png")

In [ ]:
# %% ============================================================================
# 13 · SAVE MODELS, SCORES, METRICS, CONFIG
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 13 · SAVING ARTIFACTS")
print("=" * 70)

# ---- models ----
autoencoder.save(DIRS["models"] / "autoencoder.keras")
encoder.save(DIRS["models"] / "encoder.keras")
joblib.dump(iso_forest, DIRS["models"] / "isolation_forest.pkl", compress=3)
joblib.dump(scaler,     DIRS["models"] / "scaler.pkl")
print("  ✓ models/{autoencoder.keras, encoder.keras, isolation_forest.pkl, scaler.pkl}")

# ---- scores (float32 for fusion engine) ----
np.save(DIRS["scores"] / "ae_val_mse.npy",    ae_val_mse.astype(np.float32))
np.save(DIRS["scores"] / "ae_test_mse.npy",   ae_test_mse.astype(np.float32))
np.save(DIRS["scores"] / "if_val_scores.npy", if_val_scores.astype(np.float32))
np.save(DIRS["scores"] / "if_test_scores.npy", if_test_scores.astype(np.float32))
np.save(DIRS["scores"] / "ae_val_binary.npy",  ae_val_binary.astype(np.int8))
np.save(DIRS["scores"] / "ae_test_binary.npy", ae_test_binary.astype(np.int8))
np.save(DIRS["scores"] / "if_val_binary.npy",  if_val_binary.astype(np.int8))
np.save(DIRS["scores"] / "if_test_binary.npy", if_test_binary.astype(np.int8))
print("  ✓ scores/*.npy (8 arrays)")

# ---- thresholds ----
with open(DIRS["root"] / "thresholds.json", "w") as f:
    json.dump({
        "thresholds":         thresholds,
        "evaluation_on_val":  threshold_eval_df.to_dict(orient="records"),
        "selected": {
            "name":  BEST_THRESHOLD_NAME,
            "value": BEST_THRESHOLD_VALUE,
            "f1_on_val": float(best_row["f1"]),
        },
    }, f, indent=2)
print("  ✓ thresholds.json")

# ---- benign error stats ----
with open(DIRS["root"] / "benign_error_stats.json", "w") as f:
    json.dump(benign_error_stats, f, indent=2)
print("  ✓ benign_error_stats.json")

# ---- training history ----
hist_dict = {k: [float(v) for v in vs] for k, vs in history.history.items()}
with open(DIRS["root"] / "ae_training_history.json", "w") as f:
    json.dump(hist_dict, f, indent=2)
print("  ✓ ae_training_history.json")

# ---- metrics ----
with open(DIRS["metrics"] / "ae_classification_report.json", "w") as f:
    json.dump(ae_cls_report, f, indent=2)
with open(DIRS["metrics"] / "if_classification_report.json", "w") as f:
    json.dump(if_cls_report, f, indent=2)
detection_df.to_csv(DIRS["metrics"] / "per_class_detection_rates.csv", index=False)
comparison_df.to_csv(DIRS["metrics"] / "model_comparison.csv", index=False)
zero_day_df.to_csv(DIRS["metrics"] / "zero_day_preview.csv", index=False)
print("  ✓ metrics/{ae,if}_classification_report.json")
print("  ✓ metrics/per_class_detection_rates.csv")
print("  ✓ metrics/model_comparison.csv")
print("  ✓ metrics/zero_day_preview.csv")

# ---- config ----
config_out = {
    "phase":              "5 — unsupervised training",
    "feature_space":      "full (44 features)",
    "feature_names":      FEATURE_NAMES,
    "random_state":       RANDOM_STATE,
    "autoencoder": {
        "architecture":   AE_ARCHITECTURE,
        "epochs_max":     AE_EPOCHS,
        "epochs_actual":  len(history.history["loss"]),
        "batch_size":     AE_BATCH_SIZE,
        "learning_rate":  AE_LEARNING_RATE,
        "patience":       AE_PATIENCE,
        "best_val_loss":  float(min(history.history["val_loss"])),
        "training_time_s": ae_train_time,
    },
    "isolation_forest": {
        "n_estimators":    IF_N_ESTIMATORS,
        "contamination":   IF_CONTAMINATION,
        "training_time_s": if_train_time,
    },
    "data_shapes": {
        "X_train":         list(X_train_shape),
        "X_val":           list(X_val.shape),
        "X_test":          list(X_test.shape),
        "X_benign_train":  list(X_benign_train.shape),
        "X_benign_val":    list(X_benign_val.shape),
    },
    "selected_threshold": {
        "name":  BEST_THRESHOLD_NAME,
        "value": BEST_THRESHOLD_VALUE,
    },
}
with open(DIRS["root"] / "config.json", "w") as f:
    json.dump(config_out, f, indent=2)
print("  ✓ config.json")

In [ ]:
# %% ============================================================================
# 14 · SUMMARY REPORT
# ===============================================================================
print("\n" + "=" * 70)
print("SECTION 14 · SUMMARY")
print("=" * 70)

# Identify easiest/hardest classes for AE at best threshold (excluding Benign)
ae_attack_rows = ae_detection_df[ae_detection_df["class"] != "Benign"].copy()
ae_attack_rows = ae_attack_rows.sort_values(BEST_THRESHOLD_NAME, ascending=False)
top3_easy = ae_attack_rows.head(3)
top3_hard = ae_attack_rows.tail(3).iloc[::-1]

# AE and IF have complementary failure modes — recommend both for fusion
recommended = "Both (complementary)"
reason = ("AE provides the primary anomaly signal via reconstruction error "
          "(stronger on volumetric/flooding attacks where Rate/IAT deviate sharply). "
          "IF provides a secondary signal that catches point anomalies AE misses "
          "(Recon, Spoofing, point flag-count outliers). Phase 6 fusion should "
          "consume both score arrays. "
          f"Test AUC: AE={ae_test_auc:.4f}, IF={if_test_auc:.4f}.")

# Read Phase 4's best supervised result dynamically (don't hardcode)
sup_f1 = "see Phase 4 summary"
sup_csv = SUPERVISED_DIR / "metrics" / "overall_comparison.csv"
try:
    if sup_csv.exists():
        sup_df = pd.read_csv(sup_csv)
        f1_col = next(
            (c for c in ["test_f1_macro", "f1_macro", "F1_macro", "macro_f1"]
             if c in sup_df.columns),
            None,
        )
        name_col = next(
            (c for c in ["experiment", "model", "name", "run"]
             if c in sup_df.columns),
            None,
        )
        if f1_col and name_col:
            best_sup = sup_df.sort_values(f1_col, ascending=False).iloc[0]
            sup_f1 = f"{best_sup[name_col]} (F1_macro={best_sup[f1_col]:.4f})"
except Exception as e:
    print(f"[summary] could not read Phase 4 CSV ({e}); using placeholder.")

# Build summary
summary_md = f"""# Phase 5 — Unsupervised Layer Summary

## 1 · Best autoencoder configuration

- Architecture: `{ ' → '.join(map(str, AE_ARCHITECTURE)) }`
- Optimiser: Adam, lr = `{AE_LEARNING_RATE}`, batch size = `{AE_BATCH_SIZE}`
- Trained for **{len(history.history['loss'])} epochs** (early-stopped, patience={AE_PATIENCE})
- Final training loss: `{history.history['loss'][-1]:.6f}`
- Best validation loss: **`{min(history.history['val_loss']):.6f}`**
- Wall-clock training time: **{ae_train_time:.1f} s** ({ae_train_time/60:.2f} min)

## 2 · Selected anomaly threshold

- All five candidate thresholds evaluated on the **validation** set:

| name | value | precision | recall | F1 | FPR | TPR |
|------|-------|-----------|--------|-----|------|-----|
""" + "\n".join(
    f"| {r['threshold_name']} | {r['threshold_value']:.6f} | {r['precision']:.4f} "
    f"| {r['recall']:.4f} | {r['f1']:.4f} | {r['fpr']:.4f} | {r['tpr']:.4f} |"
    for r in threshold_eval_rows
) + f"""

- **Selected:** `{BEST_THRESHOLD_NAME}` = `{BEST_THRESHOLD_VALUE:.6f}` (highest F1 on val).
- Rationale: the percentile / mean+std rules are computed on benign-only validation
  errors, so they reflect the natural noise floor of normal traffic. The chosen rule
  gave the best precision-recall trade-off for binary anomaly classification on
  validation, and is therefore used as the operating point for fusion in Phase 6.

## 3 · Binary anomaly detection performance (test set)

| metric | Autoencoder | Isolation Forest |
|---|---|---|
| AUC-ROC | **{ae_test_auc:.4f}** | {if_test_auc:.4f} |
| FPR @ 95 % TPR | {ae_fpr95_test:.4f} | {if_fpr95_test:.4f} |
| anomaly precision | {ae_cls_report['anomaly']['precision']:.4f} | {if_cls_report['anomaly']['precision']:.4f} |
| anomaly recall | {ae_cls_report['anomaly']['recall']:.4f} | {if_cls_report['anomaly']['recall']:.4f} |
| anomaly F1 | {ae_cls_report['anomaly']['f1-score']:.4f} | {if_cls_report['anomaly']['f1-score']:.4f} |

## 4 · Per-class detection rates (Autoencoder, best threshold)

**Easiest to detect:**

""" + "\n".join(
    f"- `{r['class']}` — recall **{r[BEST_THRESHOLD_NAME]:.3f}** "
    f"(n={int(r['n_samples']):,})"
    for _, r in top3_easy.iterrows()
) + "\n\n**Hardest to detect:**\n\n" + "\n".join(
    f"- `{r['class']}` — recall **{r[BEST_THRESHOLD_NAME]:.3f}** "
    f"(n={int(r['n_samples']):,})"
    for _, r in top3_hard.iterrows()
) + f"""

The full 19-class detection-rate table is in `metrics/per_class_detection_rates.csv`
and visualised as `figures/detection_rate_heatmap.png`.

## 5 · Autoencoder vs Isolation Forest

- The autoencoder learned a tighter benign manifold (lower benign-MSE variance) and
  therefore produces a more separable score distribution for volumetric/flooding
  attacks where `Rate`, `IAT`, and flag counts deviate sharply from benign.
- Isolation Forest tends to be more competitive on point-anomaly attacks
  (Recon, Spoofing) because it isolates rare feature combinations rather than
  measuring reconstruction error.
- **Average per-class recall:** AE = `{float(ae_per_class_recall):.4f}` ·
  IF = `{float(if_per_class_recall):.4f}`
- Training cost: AE = {ae_train_time:.1f} s · IF = {if_train_time:.1f} s

## 6 · Zero-day simulation (preview)

| target | n_test | AE @ {BEST_THRESHOLD_NAME} | IF recall |
|---|---|---|---|
""" + ("\n".join(
    f"| `{r['target']}` | {int(r['n_samples']):,} | {r['ae_best_thr']:.3f} | {r['if_recall']:.3f} |"
    for _, r in zero_day_df.iterrows()
) if len(zero_day_df) else "| _no targets present_ | | | |") + f"""

- **Per-class detection preview** at the selected threshold:
  {hits_70}/{total_targets} targets achieve ≥ 70 % recall.
  *Indicative only — the AE never sees attacks during training regardless, so this
  measures class separability, not true zero-day generalization. Proper H2 evaluation
  (with held-out classes and retrained supervised + IF models) is deferred to Phase 6.*
- This is a *preview* — Phase 6 fusion will combine these scores with the supervised
  E7 probabilities, which should boost zero-day recall further by exploiting the
  "supervised says benign + unsupervised says anomaly = zero-day" rule.

## 7 · Recommendation for Phase 6 fusion

- **Primary recommendation:** {recommended}.
- {reason}
- Both score arrays are exported (`scores/ae_*.npy`, `scores/if_*.npy`); the fusion
  engine should consume **both** so that the 4-case logic
  (supervised × unsupervised) can use the stronger signal per region of feature
  space. AE is preferred for the binary anomaly flag, IF as a secondary signal.

## 8 · Key findings for thesis discussion

1. Training the autoencoder on benign-only traffic produced a clean reconstruction-error
   separation between benign and attack samples — visible in
   `figures/ae_error_distribution.png`.
2. Confirms the EDA observation that benign IoMT traffic is a compact PCA cluster:
   the AE bottleneck of 8 dimensions was sufficient to reconstruct it with
   benign-val MSE = `{benign_error_stats['ae_val_benign']['mean']:.6f}`.
3. Per-class recall varies sharply across attack families. Volumetric / flooding
   classes (e.g. DDoS_*) are detected almost perfectly, while quieter recon
   classes are harder — motivating the hybrid design.
4. The unsupervised layer is a complement to, not a substitute for, the supervised
   XGBoost ({sup_f1}). Phase 6 fusion exploits this complementarity.

---

_Generated by `unsupervised_training.py` — random_state = {RANDOM_STATE}_
"""

with open(DIRS["root"] / "summary.md", "w") as f:
    f.write(summary_md)
print(f"  ✓ summary.md ({len(summary_md):,} chars)")

In [ ]:
# %% ============================================================================
# 15 · FINAL CONSOLE PRINT
# ===============================================================================
print("\n" + "=" * 70)
print("PHASE 5 COMPLETE")
print("=" * 70)
print(f"AE  test AUC: {ae_test_auc:.4f}  · F1: {ae_cls_report['anomaly']['f1-score']:.4f}")
print(f"IF  test AUC: {if_test_auc:.4f}  · F1: {if_cls_report['anomaly']['f1-score']:.4f}")
print(f"Selected threshold: {BEST_THRESHOLD_NAME} = {BEST_THRESHOLD_VALUE:.6f}")
print(f"Per-class detection ≥70% (preview): {hits_70}/{total_targets} targets "
      "— true H2 evaluation deferred to Phase 6")
print(f"\nAll outputs written to: {OUTPUT_DIR.resolve()}")
print(f"Total runtime: {(time.time() - t0)/60:.2f} min")

## Phase 6 — Fusion Engine & Zero-Day Simulation (Layer 3)

Combines E7 (supervised) and AE+IF (unsupervised) predictions using a
**4-case decision logic** — no retraining required.

| Case | Supervised | Unsupervised | Decision | Confidence |
|------|-----------|-------------|----------|-----------|
| 1 | Attack | Anomaly | **Confirmed Alert** | HIGH |
| 2 | Benign | Anomaly | **Zero-Day Warning** | MEDIUM_HIGH |
| 3 | Attack | Normal | Low-Confidence Alert | MEDIUM_LOW |
| 4 | Benign | Normal | Clear | HIGH |

**Hypothesis tests**:
- **H1**: Paired bootstrap (200 iters) — does fusion improve multiclass macro-F1?
- **H2**: AE recall on samples E7 misclassifies as benign ≥70% on ≥50% of targets.

**Zero-day targets**: Recon_Ping_Sweep, Recon_VulScan, MQTT_Malformed_Data,
MQTT_DoS_Connect_Flood, ARP_Spoofing.

**Outputs** (`results/fusion/`): case arrays (.npy), bootstrap CIs, threshold
sensitivity, H1/H2 verdicts (.json), 5 publication-quality figures.

**Prerequisite**: Phases 4 and 5 must have completed.

In [ ]:
#!/usr/bin/env python3
"""
fusion_engine.py — Phase 6: Hybrid Fusion Engine & Zero-Day Simulation
========================================================================
Project : A Hybrid Supervised-Unsupervised Framework for Anomaly Detection
          and Zero-Day Attack Identification in IoMT Networks
Dataset : CICIoMT2024
Author  : Amro
Phase   : 6 (Fusion + Zero-Day Sim)
Version : 2 (post senior review)

Combines:
  - Layer 1 (E7 / XGBoost) supervised predictions
  - Layer 2 (Autoencoder + Isolation Forest) unsupervised anomaly scores
into Layer 3 — a 4-case fusion decision engine — and evaluates
hypotheses H1 (fusion improves macro-F1; bootstrap-tested) and H2 (AE
catches what E7 misclassifies as benign on >= 50 % of zero-day targets).

No retraining is performed. Pure post-hoc combination of saved arrays.

The `apply_fusion` and `fusion_classes` helpers are importable for
downstream phases (Phase 7 SHAP slicing).

Usage:
    cd ~/IoMT-Project
    source venv/bin/activate
    python notebooks/fusion_engine.py
"""

In [ ]:
# %% [Section 1] Imports & Configuration ====================================
import warnings
warnings.filterwarnings("ignore")

import json
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)

# ---- Paths ----------------------------------------------------------------
BASE_DIR         = Path(".").resolve()
SUPERVISED_DIR   = BASE_DIR / "results" / "supervised"
UNSUPERVISED_DIR = BASE_DIR / "results" / "unsupervised"
PREPROCESSED_DIR = BASE_DIR / "preprocessed"
OUTPUT_DIR       = BASE_DIR / "results" / "fusion"

CASE_COLORS = {1: "#d62728", 2: "#ff7f0e", 3: "#bcbd22", 4: "#2ca02c"}
CASE_NAMES = {
    1: "Confirmed Alert",
    2: "Zero-Day Warning",
    3: "Low-Confidence Alert",
    4: "Clear",
}
CASE_DECISIONS  = {1: "confirmed_alert", 2: "zero_day_warning",
                   3: "low_confidence_alert", 4: "clear"}
CASE_CONFIDENCE = {1: "HIGH", 2: "MEDIUM_HIGH", 3: "MEDIUM_LOW", 4: "HIGH"}

# ---- Hypothesis test parameters ------------------------------------------
H1_BOOTSTRAP_ITERS = 200          # paired bootstrap iterations
H1_BOOTSTRAP_SEED  = 42
H2_RECALL_TARGET   = 0.70
H2_FRACTION_TARGET = 0.50
H2_MIN_SAMPLES     = 30           # min n_called_benign for reliable H2 metric

ZERO_DAY_TARGETS = [
    "Recon_Ping_Sweep",
    "Recon_VulScan",
    "MQTT_Malformed_Data",
    "MQTT_DoS_Connect_Flood",
    "ARP_Spoofing",
]

AE_THRESHOLD_PERCENTILES = [90, 95, 99]
PERCENTILES_SWEEP        = [50, 60, 70, 80, 85, 90, 92, 95, 97, 99]
RECOMMENDED_FPR_BUDGET   = 0.05   # selection criterion on val

In [ ]:
# %% [Section 2] Helper functions (importable) ==============================
def banner(title: str) -> None:
    print(f"\n{'=' * 78}\n  {title}\n{'=' * 78}")


def md_table(df: pd.DataFrame, *, index: bool = False,
             floatfmt: str = ".4f") -> str:
    """Markdown table with graceful fallback if `tabulate` isn't installed."""
    try:
        return df.to_markdown(index=index, floatfmt=floatfmt)
    except Exception:
        return "```\n" + df.to_csv(index=index) + "```"


def npload(p: Path) -> np.ndarray:
    if not p.exists():
        sys.exit(f"[FATAL] Missing file: {p}")
    return np.load(p)


def normalise_if_binary(arr: np.ndarray) -> np.ndarray:
    """Coerce IF binary to {0=normal, 1=anomaly} regardless of source convention.

    sklearn IsolationForest.predict returns {-1, +1} (-1 = anomaly).
    Phase-5 README states the saved file is already converted, but we
    defend against either convention.
    """
    a = np.asarray(arr).astype(np.int8)
    uniq = set(np.unique(a).tolist())
    if uniq <= {-1, 1}:
        return (a == -1).astype(np.int8)
    if uniq <= {0, 1}:
        return a
    raise ValueError(f"Unexpected IF binary values: {uniq}")


def apply_fusion(sup_pred: np.ndarray, ae_binary: np.ndarray,
                 benign_id: int) -> np.ndarray:
    """Compute the 4-case fusion decision per sample.

    Cases:
      1 = Attack + Anomaly  -> Confirmed Alert     (HIGH)
      2 = Benign + Anomaly  -> Zero-Day Warning    (MEDIUM_HIGH)
      3 = Attack + Normal   -> Low-Confidence      (MEDIUM_LOW)
      4 = Benign + Normal   -> Clear               (HIGH)

    Args:
        sup_pred : (N,) supervised predicted class id
        ae_binary: (N,) 0 = normal, 1 = anomaly
        benign_id: class id of "Benign" in the multiclass encoding
    """
    sup_attack = sup_pred != benign_id
    ae_anom    = ae_binary == 1
    case = np.empty(len(sup_pred), dtype=np.int8)
    case[ sup_attack &  ae_anom] = 1
    case[~sup_attack &  ae_anom] = 2
    case[ sup_attack & ~ae_anom] = 3
    case[~sup_attack & ~ae_anom] = 4
    return case


def fusion_classes(sup_pred: np.ndarray, fusion_case: np.ndarray,
                   benign_id: int, zero_day_id: int) -> np.ndarray:
    """Final multiclass label per fusion case.

    Case 1, 3 -> use E7 predicted class
    Case 2    -> 'zero_day_unknown' (numeric id = `zero_day_id`)
    Case 4    -> Benign class id
    """
    out = sup_pred.copy().astype(np.int32)
    out[fusion_case == 2] = zero_day_id
    out[fusion_case == 4] = benign_id
    return out


def ae_binary_at(threshold: float, mse: np.ndarray) -> np.ndarray:
    return (mse > threshold).astype(np.int8)


def bootstrap_paired_f1(
    y_true: np.ndarray,
    preds_dict: dict,
    labels: list,
    n_boot: int = 200,
    seed: int = 42,
) -> dict:
    """Paired-resample bootstrap of macro-F1 for each prediction.

    Same resampled indices are reused across all predictions per iteration,
    enabling valid paired difference distributions and CIs.
    """
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    out = {name: np.empty(n_boot, dtype=np.float64) for name in preds_dict}
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        y_b = y_true[idx]
        for name, pred in preds_dict.items():
            out[name][b] = f1_score(
                y_b, pred[idx], labels=labels,
                average="macro", zero_division=0,
            )
        if (b + 1) % 50 == 0:
            print(f"    bootstrap iter {b + 1}/{n_boot}")
    return out


def ci_95(arr: np.ndarray) -> tuple:
    return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

In [ ]:
# %% [Section 3] Main pipeline ==============================================
def main() -> None:
    for sub in ("fusion_results", "metrics", "figures"):
        (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

    plt.rcParams.update({
        "figure.dpi": 100, "savefig.dpi": 200, "savefig.bbox": "tight",
        "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
    })
    sns.set_style("whitegrid")

    banner("PHASE 6 — FUSION ENGINE & ZERO-DAY SIMULATION (v2)")
    print(f"Started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Output  : {OUTPUT_DIR}")

    # ---- [3.1] Load inputs ------------------------------------------------
    banner("[3.1] Loading saved arrays from previous phases")

    # Supervised: only argmax preds are needed (proba arrays unused → skip)
    e7_val_pred  = npload(SUPERVISED_DIR / "predictions" / "E7_val_pred.npy")
    e7_test_pred = npload(SUPERVISED_DIR / "predictions" / "E7_test_pred.npy")

    # Unsupervised AE
    ae_val_mse     = npload(UNSUPERVISED_DIR / "scores" / "ae_val_mse.npy")
    ae_test_mse    = npload(UNSUPERVISED_DIR / "scores" / "ae_test_mse.npy")
    # Saved binary flags retained for compatibility but recomputed per threshold
    _ = npload(UNSUPERVISED_DIR / "scores" / "ae_val_binary.npy")
    _ = npload(UNSUPERVISED_DIR / "scores" / "ae_test_binary.npy")

    # Unsupervised IF
    if_val_binary  = normalise_if_binary(
        npload(UNSUPERVISED_DIR / "scores" / "if_val_binary.npy"))
    if_test_binary = normalise_if_binary(
        npload(UNSUPERVISED_DIR / "scores" / "if_test_binary.npy"))

    # Labels & encoders
    y_val  = pd.read_csv(PREPROCESSED_DIR / "y_val.csv")
    y_test = pd.read_csv(PREPROCESSED_DIR / "y_test.csv")
    with open(PREPROCESSED_DIR / "label_encoders.json") as f:
        label_encoders = json.load(f)
    with open(UNSUPERVISED_DIR / "thresholds.json") as f:
        ae_thresholds_raw = json.load(f)

    # Sanity
    n_val, n_test = len(y_val), len(y_test)
    for name, arr, expected_n in [
        ("e7_val_pred",   e7_val_pred,   n_val),
        ("e7_test_pred",  e7_test_pred,  n_test),
        ("ae_val_mse",    ae_val_mse,    n_val),
        ("ae_test_mse",   ae_test_mse,   n_test),
        ("if_val_binary", if_val_binary, n_val),
        ("if_test_binary",if_test_binary,n_test),
    ]:
        if arr.shape[0] != expected_n:
            sys.exit(f"[FATAL] {name} length {arr.shape[0]} != {expected_n}")

    print(f"  Val  samples : {n_val:>10,}")
    print(f"  Test samples : {n_test:>10,}")
    print(f"  Multiclass classes: {len(label_encoders['multiclass'])}")
    print(f"  AE threshold keys: {list(ae_thresholds_raw.keys())}")

    mc_inv     = {int(v): k for k, v in label_encoders["multiclass"].items()}
    n_classes  = len(label_encoders["multiclass"])
    BENIGN_ID  = label_encoders["multiclass"]["Benign"]
    ZERO_DAY_ID = n_classes  # one above range for fusion's pseudo-class
    print(f"  Benign class ID = {BENIGN_ID}, Zero-day pseudo-id = {ZERO_DAY_ID}")

    y_val_mc   = y_val["multiclass_label"].values.astype(int)
    y_test_mc  = y_test["multiclass_label"].values.astype(int)
    y_val_bin  = (y_val_mc  != BENIGN_ID).astype(int)
    y_test_bin = (y_test_mc != BENIGN_ID).astype(int)

    benign_val_mse = ae_val_mse[y_val_bin == 0]

    # ---- [3.2] Resolve thresholds ----------------------------------------
    def resolve_threshold(pct: int) -> float:
        for key in (f"p{pct}", str(pct), f"{pct}.0", str(float(pct))):
            if key in ae_thresholds_raw:
                v = ae_thresholds_raw[key]
                if isinstance(v, dict) and "value" in v:
                    return float(v["value"])
                try:
                    return float(v)
                except (TypeError, ValueError):
                    continue
        return float(np.percentile(benign_val_mse, pct))

    THRESHOLDS = {pct: resolve_threshold(pct) for pct in AE_THRESHOLD_PERCENTILES}
    print("  AE thresholds in use:")
    for pct, val in THRESHOLDS.items():
        print(f"    p{pct}: {val:.4f}")

    # ---- [3.3] Apply fusion at each operating point ----------------------
    banner("[3.3] Applying fusion at p90, p95, p99 (and IF baseline)")
    fusion_variants_test, fusion_variants_val = {}, {}
    for pct, t in THRESHOLDS.items():
        name = f"AE_p{pct}"
        fusion_variants_test[name] = {
            "ae_binary": ae_binary_at(t, ae_test_mse), "threshold": t,
        }
        fusion_variants_val[name] = {
            "ae_binary": ae_binary_at(t, ae_val_mse), "threshold": t,
        }
    fusion_variants_test["IF"] = {"ae_binary": if_test_binary, "threshold": None}
    fusion_variants_val["IF"]  = {"ae_binary": if_val_binary,  "threshold": None}

    for v in fusion_variants_test.values():
        v["case"] = apply_fusion(e7_test_pred, v["ae_binary"], BENIGN_ID)
    for v in fusion_variants_val.values():
        v["case"] = apply_fusion(e7_val_pred, v["ae_binary"], BENIGN_ID)

    PRIMARY = f"AE_p{AE_THRESHOLD_PERCENTILES[0]}"
    print(f"  Primary fusion config: {PRIMARY}")

    # ---- [3.4] Case distribution -----------------------------------------
    banner("[3.4] Case distribution per variant (test set)")
    case_dist_rows = []
    for name, v in fusion_variants_test.items():
        counts = np.bincount(v["case"], minlength=5)[1:5]
        share  = counts / counts.sum() * 100
        row = {"variant": name, "threshold": v["threshold"]}
        for c in (1, 2, 3, 4):
            row[f"case{c}_n"]   = int(counts[c - 1])
            row[f"case{c}_pct"] = float(share[c - 1])
        case_dist_rows.append(row)
        print(f"  {name:<10}  C1={counts[0]:>8,} ({share[0]:5.2f}%)  "
              f"C2={counts[1]:>8,} ({share[1]:5.2f}%)  "
              f"C3={counts[2]:>8,} ({share[2]:5.2f}%)  "
              f"C4={counts[3]:>8,} ({share[3]:5.2f}%)")
    case_dist_df = pd.DataFrame(case_dist_rows)
    case_dist_df.to_csv(OUTPUT_DIR / "metrics" / "case_distribution.csv",
                        index=False)

    # ---- [3.5] Binary detection (Cases 1+2+3 vs Case 4) ------------------
    banner("[3.5] Binary anomaly detection — fusion vs E7-only")
    e7_test_binpred = (e7_test_pred != BENIGN_ID).astype(int)
    bin_rows = [{
        "variant"  : "E7_only",
        "accuracy" : accuracy_score(y_test_bin, e7_test_binpred),
        "precision": precision_score(y_test_bin, e7_test_binpred, zero_division=0),
        "recall"   : recall_score(y_test_bin, e7_test_binpred, zero_division=0),
        "f1"       : f1_score(y_test_bin, e7_test_binpred, zero_division=0),
        "mcc"      : matthews_corrcoef(y_test_bin, e7_test_binpred),
    }]
    for name, v in fusion_variants_test.items():
        flagged = (v["case"] != 4).astype(int)
        bin_rows.append({
            "variant"  : name,
            "accuracy" : accuracy_score(y_test_bin, flagged),
            "precision": precision_score(y_test_bin, flagged, zero_division=0),
            "recall"   : recall_score(y_test_bin, flagged, zero_division=0),
            "f1"       : f1_score(y_test_bin, flagged, zero_division=0),
            "mcc"      : matthews_corrcoef(y_test_bin, flagged),
        })
    bin_metric_df = pd.DataFrame(bin_rows)
    bin_metric_df.to_csv(
        OUTPUT_DIR / "metrics" / "fusion_vs_supervised_binary.csv", index=False)
    print(bin_metric_df.to_string(
        index=False, float_format=lambda x: f"{x:.4f}"))

    # ---- [3.6] H1 — multiclass macro-F1 with bootstrap CI ----------------
    banner("[3.6] H1 — multiclass macro-F1 with paired bootstrap CI")

    e7_macro_f1 = f1_score(y_test_mc, e7_test_pred,
                           average="macro", zero_division=0)
    e7_mcc      = matthews_corrcoef(y_test_mc, e7_test_pred)
    e7_acc      = accuracy_score(y_test_mc, e7_test_pred)
    print(f"  E7 only        : macro-F1 = {e7_macro_f1:.4f}  "
          f"MCC = {e7_mcc:.4f}  acc = {e7_acc:.4f}")

    labels_with_zd = list(range(n_classes)) + [ZERO_DAY_ID]

    # Build dict of predictions for paired bootstrap (same y_true, same indices)
    preds_for_boot = {"E7_only": e7_test_pred.astype(np.int32)}
    fused_predictions = {}
    for name, v in fusion_variants_test.items():
        fp = fusion_classes(e7_test_pred, v["case"], BENIGN_ID, ZERO_DAY_ID)
        fused_predictions[name] = fp
        preds_for_boot[name] = fp

    print(f"  Running paired bootstrap "
          f"({H1_BOOTSTRAP_ITERS} iters, {len(preds_for_boot)} variants)…")
    boot = bootstrap_paired_f1(
        y_test_mc.astype(np.int32),
        preds_for_boot,
        labels=labels_with_zd,
        n_boot=H1_BOOTSTRAP_ITERS,
        seed=H1_BOOTSTRAP_SEED,
    )

    mc_macro_rows = []
    for name in fusion_variants_test:
        fused_pred = fused_predictions[name]
        f1m = f1_score(y_test_mc, fused_pred, labels=labels_with_zd,
                       average="macro", zero_division=0)
        mcc = matthews_corrcoef(y_test_mc, fused_pred)
        acc = accuracy_score(y_test_mc, fused_pred)
        delta_dist = boot[name] - boot["E7_only"]
        delta_lo, delta_hi = ci_95(delta_dist)
        f1_lo, f1_hi       = ci_95(boot[name])
        mc_macro_rows.append({
            "variant"        : name,
            "macro_f1"       : f1m,
            "macro_f1_ci_lo" : f1_lo,
            "macro_f1_ci_hi" : f1_hi,
            "mcc"            : mcc,
            "accuracy"       : acc,
            "delta_f1_vs_E7" : f1m - e7_macro_f1,
            "delta_ci_lo"    : delta_lo,
            "delta_ci_hi"    : delta_hi,
            "h1_significant" : bool(delta_lo > 0),
        })
        flag = "✓ sig" if delta_lo > 0 else "✗ ns "
        print(f"  Fusion {name:<8}: F1 = {f1m:.4f} "
              f"[{f1_lo:.4f}, {f1_hi:.4f}]  Δ = {f1m - e7_macro_f1:+.4f} "
              f"[{delta_lo:+.4f}, {delta_hi:+.4f}]  {flag}")
    mc_macro_df = pd.DataFrame(mc_macro_rows)
    mc_macro_df.to_csv(
        OUTPUT_DIR / "metrics" / "fusion_vs_supervised.csv", index=False)

    # E7 baseline CI for the summary
    e7_f1_lo, e7_f1_hi = ci_95(boot["E7_only"])
    print(f"\n  E7 macro-F1 95% CI: [{e7_f1_lo:.4f}, {e7_f1_hi:.4f}]")

    # H1 verdict (primary + best across configs)
    primary_row = mc_macro_df[mc_macro_df["variant"] == PRIMARY].iloc[0]
    H1_PASS_PRIMARY = bool(primary_row["h1_significant"])
    best_row = mc_macro_df.iloc[mc_macro_df["macro_f1"].idxmax()]
    H1_PASS_BEST    = bool(best_row["h1_significant"])
    print(f"  H1 (primary={PRIMARY}): "
          f"{'PASS — Δ CI excludes 0' if H1_PASS_PRIMARY else 'FAIL — Δ CI includes 0'}")
    print(f"  H1 (best={best_row['variant']}): "
          f"{'PASS — Δ CI excludes 0' if H1_PASS_BEST else 'FAIL — Δ CI includes 0'}")

    # ---- [3.7] Per-class case distribution at all 3 thresholds -----------
    banner("[3.7] Per-class case distribution — three thresholds")
    class_names = [mc_inv[i] for i in range(n_classes)]
    per_class_heatmaps = {}  # variant -> (n_classes, 4) matrix
    for variant_name in [f"AE_p{p}" for p in AE_THRESHOLD_PERCENTILES]:
        case_arr = fusion_variants_test[variant_name]["case"]
        H = np.zeros((n_classes, 4))
        nonempty = []
        for ci in range(n_classes):
            mask = y_test_mc == ci
            if mask.sum() == 0:
                continue
            nonempty.append(ci)
            cic = case_arr[mask]
            for k in (1, 2, 3, 4):
                H[ci, k - 1] = (cic == k).mean() * 100
        # Sanity: nonempty rows must sum to 100
        sums = H[nonempty].sum(axis=1)
        np.testing.assert_allclose(
            sums, 100.0, atol=1e-4,
            err_msg=f"Per-class rows don't sum to 100% in {variant_name}")
        per_class_heatmaps[variant_name] = H

    # CSV: long format for primary heatmap (kept as the canonical reference)
    primary_H = per_class_heatmaps[PRIMARY]
    per_class_df = pd.DataFrame(
        primary_H, index=class_names,
        columns=[f"Case{i}_pct" for i in (1, 2, 3, 4)],
    )
    per_class_df["n_test"] = [int((y_test_mc == ci).sum())
                              for ci in range(n_classes)]
    per_class_df.to_csv(OUTPUT_DIR / "metrics" / "per_class_case_analysis.csv")
    print(per_class_df.to_string(float_format=lambda x: f"{x:6.2f}"))

    # ---- [3.8] Simulated zero-day under E7-blindness (H2) ----------------
    banner("[3.8] Simulated zero-day under E7-blindness (H2)")
    print("  NOTE: E7 is NOT retrained per target — true LOO would require "
          "5 separate E7 fits.\n  This measures: when E7 misclassifies a "
          "target attack as benign,\n  does the AE flag it as anomalous?")

    zero_day_rows = []
    for target in ZERO_DAY_TARGETS:
        if target not in label_encoders["multiclass"]:
            print(f"  [warn] {target} not in label encoder — skipping")
            continue
        tid  = label_encoders["multiclass"][target]
        mask = (y_test_mc == tid)
        n_t  = int(mask.sum())
        if n_t == 0:
            print(f"  [warn] {target} has no test samples")
            continue

        e7_correct       = (e7_test_pred[mask] == tid)
        e7_recall        = float(e7_correct.mean())
        e7_called_benign = (e7_test_pred[mask] == BENIGN_ID)
        n_called_benign  = int(e7_called_benign.sum())
        sufficient       = n_called_benign >= H2_MIN_SAMPLES

        row = {
            "target": target, "n_test": n_t,
            "e7_recall": e7_recall,
            "e7_called_benign_n": n_called_benign,
            "e7_called_benign_pct": float(n_called_benign / n_t * 100),
            "h2_sample_sufficient": sufficient,
        }

        for pct, t in THRESHOLDS.items():
            ae_bin_local       = ae_binary_at(t, ae_test_mse[mask])
            ae_recall_raw      = float(ae_bin_local.mean())  # auxiliary
            if n_called_benign > 0:
                missed_mse = ae_test_mse[mask][e7_called_benign]
                ae_recall_on_missed = float(
                    ae_binary_at(t, missed_mse).mean())
            else:
                ae_recall_on_missed = float("nan")

            case_local = apply_fusion(
                e7_test_pred[mask], ae_bin_local, BENIGN_ID)
            binary_detected = float((case_local != 4).mean())
            confirmed_or_zd = float(np.isin(case_local, [1, 2]).mean())

            row[f"ae_recall_p{pct}"]              = ae_recall_raw
            row[f"ae_recall_on_missed_p{pct}"]    = ae_recall_on_missed
            row[f"binary_detected_recall_p{pct}"] = binary_detected
            row[f"confirmed_or_zeroday_p{pct}"]   = confirmed_or_zd

        zero_day_rows.append(row)
        suff_tag = "" if sufficient else "  [INSUFFICIENT n_missed]"
        print(f"  {target:<28} n={n_t:>5,}  E7={e7_recall:.3f}  "
              f"E7→Benign={n_called_benign:>4d}  "
              f"AEonMissed: p90={row['ae_recall_on_missed_p90']:.3f} "
              f"p95={row['ae_recall_on_missed_p95']:.3f} "
              f"p99={row['ae_recall_on_missed_p99']:.3f}{suff_tag}")

    zd_df = pd.DataFrame(zero_day_rows)
    zd_df.to_csv(OUTPUT_DIR / "metrics" / "zero_day_results.csv", index=False)

    # H2 verdict on PRIMARY metric: ae_recall_on_missed across thresholds
    H2_per_target_primary = {}
    for r in zero_day_rows:
        if not r["h2_sample_sufficient"]:
            H2_per_target_primary[r["target"]] = {
                "value": float("nan"), "sufficient": False, "passes": False,
            }
            continue
        candidates = [r[f"ae_recall_on_missed_p{p}"]
                      for p in AE_THRESHOLD_PERCENTILES]
        candidates = [c for c in candidates if not np.isnan(c)]
        best = max(candidates) if candidates else float("nan")
        H2_per_target_primary[r["target"]] = {
            "value": float(best), "sufficient": True,
            "passes": bool(best >= H2_RECALL_TARGET),
        }

    n_pass_primary = sum(1 for d in H2_per_target_primary.values()
                         if d["passes"])
    n_total = len(H2_per_target_primary)
    H2_PASS = ((n_pass_primary / n_total) >= H2_FRACTION_TARGET
               if n_total else False)

    # Auxiliary: same verdict on raw ae_recall (closer to Phase 5's framing)
    H2_per_target_aux = {}
    for r in zero_day_rows:
        best = max(r[f"ae_recall_p{p}"] for p in AE_THRESHOLD_PERCENTILES)
        H2_per_target_aux[r["target"]] = {
            "value": float(best),
            "passes": bool(best >= H2_RECALL_TARGET),
        }
    n_pass_aux = sum(1 for d in H2_per_target_aux.values() if d["passes"])

    print(f"\n  H2 (PRIMARY: AE recall on E7-missed): "
          f"{n_pass_primary}/{n_total} → "
          f"{'PASS' if H2_PASS else 'FAIL'}")
    print(f"  H2 (auxiliary: raw AE per-class recall): "
          f"{n_pass_aux}/{n_total}")

    # ---- [3.9] Threshold sweep — selection on val, evaluation on test ----
    banner("[3.9] Threshold sweep — val for selection, test for reporting")
    val_rows, test_rows = [], []
    for pct in PERCENTILES_SWEEP:
        t = float(np.percentile(benign_val_mse, pct))
        # Validation
        ae_bin_v = ae_binary_at(t, ae_val_mse)
        case_v   = apply_fusion(e7_val_pred, ae_bin_v, BENIGN_ID)
        flag_v   = (case_v != 4).astype(int)
        val_rows.append({
            "percentile": pct, "threshold": t,
            "val_attack_recall": float(flag_v[y_val_bin == 1].mean()),
            "val_benign_fpr"   : float(flag_v[y_val_bin == 0].mean()),
            "val_binary_f1"    : float(f1_score(
                y_val_bin, flag_v, zero_division=0)),
        })
        # Test (reporting only — never used for selection)
        ae_bin_te = ae_binary_at(t, ae_test_mse)
        case_te   = apply_fusion(e7_test_pred, ae_bin_te, BENIGN_ID)
        flag_te   = (case_te != 4).astype(int)
        test_rows.append({
            "percentile": pct, "threshold": t,
            "test_attack_recall": float(flag_te[y_test_bin == 1].mean()),
            "test_benign_fpr"   : float(flag_te[y_test_bin == 0].mean()),
            "test_binary_f1"    : float(f1_score(
                y_test_bin, flag_te, zero_division=0)),
        })

    val_sweep_df  = pd.DataFrame(val_rows)
    test_sweep_df = pd.DataFrame(test_rows)
    sweep_df = val_sweep_df.merge(
        test_sweep_df.drop(columns=["threshold"]), on="percentile")
    sweep_df.to_csv(
        OUTPUT_DIR / "metrics" / "threshold_sensitivity.csv", index=False)
    print(sweep_df.to_string(
        index=False, float_format=lambda x: f"{x:.4f}"))

    # SELECTION: max val attack recall with val FPR < budget (val-only)
    val_op_mask = val_sweep_df["val_benign_fpr"] < RECOMMENDED_FPR_BUDGET
    if val_op_mask.any():
        selected_val = val_sweep_df[val_op_mask].sort_values(
            "val_attack_recall", ascending=False).iloc[0]
        selected_pct = int(selected_val["percentile"])
        # Look up test performance at the val-selected percentile
        test_at_sel = test_sweep_df[
            test_sweep_df["percentile"] == selected_pct].iloc[0]
        print(f"\n  Recommended (selected on val, FPR<{RECOMMENDED_FPR_BUDGET}): "
              f"p{selected_pct}  "
              f"val_TPR={selected_val['val_attack_recall']:.4f} "
              f"val_FPR={selected_val['val_benign_fpr']:.4f}  →  "
              f"test_TPR={test_at_sel['test_attack_recall']:.4f} "
              f"test_FPR={test_at_sel['test_benign_fpr']:.4f}")
        recommended = {
            "percentile": selected_pct,
            "threshold" : float(selected_val["threshold"]),
            "val_TPR"   : float(selected_val["val_attack_recall"]),
            "val_FPR"   : float(selected_val["val_benign_fpr"]),
            "test_TPR"  : float(test_at_sel["test_attack_recall"]),
            "test_FPR"  : float(test_at_sel["test_benign_fpr"]),
            "test_F1"   : float(test_at_sel["test_binary_f1"]),
        }
    else:
        print("\n  No val percentile meets FPR budget; deployment threshold "
              "requires committee discussion.")
        recommended = None

    # ---- [3.10] Visualizations -------------------------------------------
    banner("[3.10] Generating figures")

    # Fig 1: case distribution per variant
    fig, ax = plt.subplots(figsize=(9, 5))
    variants = list(fusion_variants_test.keys())
    x = np.arange(len(variants))
    width = 0.2
    for i, c in enumerate((1, 2, 3, 4)):
        counts = [(fusion_variants_test[v]["case"] == c).sum()
                  for v in variants]
        ax.bar(x + (i - 1.5) * width, counts, width,
               label=f"Case {c}: {CASE_NAMES[c]}", color=CASE_COLORS[c])
    ax.set_xticks(x); ax.set_xticklabels(variants)
    ax.set_ylabel("Number of test samples (log scale)")
    ax.set_title("Fusion case distribution by variant — test set")
    ax.set_yscale("log")
    ax.legend(loc="upper right", framealpha=0.9, fontsize=9)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figures" / "case_distribution.png")
    plt.close(fig)

    # Fig 2: per-class heatmap × 3 thresholds
    fig, axes = plt.subplots(1, 3, figsize=(20, 9), sharey=True)
    for ax, pct in zip(axes, AE_THRESHOLD_PERCENTILES):
        H = per_class_heatmaps[f"AE_p{pct}"]
        sns.heatmap(
            H, annot=True, fmt=".1f", cmap="YlOrRd",
            xticklabels=[f"Case {i+1}" for i in range(4)],
            yticklabels=class_names if ax is axes[0] else False,
            cbar=(ax is axes[-1]),
            cbar_kws={"label": "% of class samples"} if ax is axes[-1] else None,
            ax=ax, vmin=0, vmax=100,
        )
        ax.set_title(f"AE_p{pct}")
    fig.suptitle("Per-class fusion case distribution at three operating points",
                 y=1.01, fontsize=12)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figures" / "per_class_heatmap.png")
    plt.close(fig)

    # Fig 3: fusion macro-F1 with 95% CI error bars
    fig, ax = plt.subplots(figsize=(8, 5))
    labels_plot = ["E7 only"] + list(mc_macro_df["variant"])
    values      = [e7_macro_f1] + list(mc_macro_df["macro_f1"])
    ci_lo       = [e7_f1_lo]    + list(mc_macro_df["macro_f1_ci_lo"])
    ci_hi       = [e7_f1_hi]    + list(mc_macro_df["macro_f1_ci_hi"])
    err_lo = np.array(values) - np.array(ci_lo)
    err_hi = np.array(ci_hi)  - np.array(values)
    colors = ["#7f7f7f"] + ["#1f77b4"] * len(mc_macro_df)
    ax.bar(labels_plot, values, color=colors,
           yerr=[err_lo, err_hi], capsize=5,
           ecolor="#444", error_kw={"elinewidth": 1.2})
    ax.axhline(e7_macro_f1, ls="--", color="grey", alpha=0.7,
               label=f"E7 baseline = {e7_macro_f1:.4f}")
    for i, (v, lo, hi) in enumerate(zip(values, ci_lo, ci_hi)):
        ax.text(i, v + (hi - v) + 0.005,
                f"{v:.4f}", ha="center", fontsize=9)
    ax.set_ylabel("Macro-F1 (test, 20-class incl. zero_day_unknown)")
    ax.set_title("Fusion vs E7 macro-F1 with paired bootstrap 95% CI")
    y_min = min(ci_lo) - 0.03
    ax.set_ylim(max(0, y_min), 1.0)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figures" / "fusion_vs_supervised.png")
    plt.close(fig)

    # Fig 4: zero-day — primary metric (ae_recall_on_missed) per target
    fig, ax = plt.subplots(figsize=(11, 5))
    targets   = list(zd_df["target"])
    xs        = np.arange(len(targets))
    bar_w     = 0.18
    series = [
        ("E7 recall",             "e7_recall",                  "#7f7f7f"),
        ("AE on E7-missed p90",   "ae_recall_on_missed_p90",    "#1f77b4"),
        ("AE on E7-missed p95",   "ae_recall_on_missed_p95",    "#9467bd"),
        ("AE on E7-missed p99",   "ae_recall_on_missed_p99",    "#17becf"),
        ("Confirmed/zero-day p90","confirmed_or_zeroday_p90",   "#2ca02c"),
    ]
    for i, (label, col, colour) in enumerate(series):
        ax.bar(xs + (i - 2) * bar_w, zd_df[col].fillna(0), bar_w,
               label=label, color=colour)
    ax.axhline(H2_RECALL_TARGET, ls="--", color="red", alpha=0.7,
               label=f"H2 target = {H2_RECALL_TARGET:.2f}")
    # Mark insufficient-sample targets
    for i, r in zd_df.iterrows():
        if not r["h2_sample_sufficient"]:
            ax.text(i, 1.02, "n<30", ha="center", color="red", fontsize=8)
    ax.set_xticks(xs); ax.set_xticklabels(targets, rotation=20, ha="right")
    ax.set_ylim(0, 1.10); ax.set_ylabel("Recall")
    ax.set_title("Zero-day simulation — H2 primary metric "
                 "(AE recall conditional on E7 misclassification as benign)")
    ax.legend(loc="lower right", framealpha=0.9, fontsize=8)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figures" / "zero_day_detection.png")
    plt.close(fig)

    # Fig 5: threshold sensitivity — val + test side-by-side
    fig, ax1 = plt.subplots(figsize=(9.5, 5))
    ax1.plot(sweep_df["percentile"], sweep_df["val_attack_recall"],
             "o-", color="#1f77b4", label="Val TPR")
    ax1.plot(sweep_df["percentile"], sweep_df["test_attack_recall"],
             "o--", color="#1f77b4", alpha=0.5, label="Test TPR")
    ax1.set_xlabel("AE threshold percentile (on benign-val MSE)")
    ax1.set_ylabel("Attack recall (TPR)", color="#1f77b4")
    ax1.tick_params(axis="y", labelcolor="#1f77b4")
    ax1.set_ylim(0, 1.05)
    ax2 = ax1.twinx()
    ax2.plot(sweep_df["percentile"], sweep_df["val_benign_fpr"],
             "s-", color="#d62728", label="Val FPR")
    ax2.plot(sweep_df["percentile"], sweep_df["test_benign_fpr"],
             "s--", color="#d62728", alpha=0.5, label="Test FPR")
    ax2.set_ylabel("Benign FPR", color="#d62728")
    ax2.tick_params(axis="y", labelcolor="#d62728")
    ax2.axhline(RECOMMENDED_FPR_BUDGET, ls="--",
                color="#d62728", alpha=0.4)
    if recommended is not None:
        ax1.axvline(recommended["percentile"], ls=":",
                    color="green", alpha=0.7,
                    label=f"Selected on val: p{recommended['percentile']}")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2,
               loc="center right", framealpha=0.9, fontsize=8)
    ax1.set_title("Threshold sensitivity — val for selection, test for reporting")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figures" / "threshold_sensitivity.png")
    plt.close(fig)

    print(f"  Saved 5 figures to {OUTPUT_DIR / 'figures'}")

    # ---- [3.11] Save fusion arrays + config + verdicts -------------------
    banner("[3.11] Persisting fusion arrays + verdicts")
    np.save(OUTPUT_DIR / "fusion_results" / "fusion_val_cases.npy",
            fusion_variants_val[PRIMARY]["case"])
    np.save(OUTPUT_DIR / "fusion_results" / "fusion_test_cases.npy",
            fusion_variants_test[PRIMARY]["case"])

    def to_decision_df(case_arr: np.ndarray) -> pd.DataFrame:
        return pd.DataFrame({
            "case": case_arr,
            "decision":   [CASE_DECISIONS[int(c)]  for c in case_arr],
            "confidence": [CASE_CONFIDENCE[int(c)] for c in case_arr],
        })

    # Decoded CSVs are redundant w/ npy + dict — kept for inspection only
    to_decision_df(fusion_variants_val[PRIMARY]["case"]).to_csv(
        OUTPUT_DIR / "fusion_results" / "fusion_val_labels.csv", index=False)
    to_decision_df(fusion_variants_test[PRIMARY]["case"]).to_csv(
        OUTPUT_DIR / "fusion_results" / "fusion_test_labels.csv", index=False)

    config = {
        "phase": 6, "version": 2,
        "primary_variant": PRIMARY,
        "ae_thresholds": THRESHOLDS,
        "benign_id": int(BENIGN_ID),
        "zero_day_id": int(ZERO_DAY_ID),
        "n_val": int(n_val), "n_test": int(n_test),
        "n_classes": int(n_classes),
        "h1_bootstrap_iters": H1_BOOTSTRAP_ITERS,
        "h1_bootstrap_seed":  H1_BOOTSTRAP_SEED,
        "h2_recall_target":   H2_RECALL_TARGET,
        "h2_fraction_target": H2_FRACTION_TARGET,
        "h2_min_samples":     H2_MIN_SAMPLES,
        "zero_day_targets":   ZERO_DAY_TARGETS,
        "recommended_fpr_budget": RECOMMENDED_FPR_BUDGET,
        "timestamp": datetime.now().isoformat(),
    }
    with open(OUTPUT_DIR / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    verdicts = {
        "H1": {
            "description": ("Fusion improves multiclass macro-F1 over E7 with "
                            "paired-bootstrap 95% CI excluding zero "
                            "(20-class label space incl. zero_day_unknown)"),
            "e7_macro_f1": float(e7_macro_f1),
            "e7_macro_f1_ci": [e7_f1_lo, e7_f1_hi],
            "fusion_macro_f1_primary": float(primary_row["macro_f1"]),
            "fusion_macro_f1_primary_ci": [
                float(primary_row["macro_f1_ci_lo"]),
                float(primary_row["macro_f1_ci_hi"]),
            ],
            "delta_primary": float(primary_row["delta_f1_vs_E7"]),
            "delta_primary_ci": [
                float(primary_row["delta_ci_lo"]),
                float(primary_row["delta_ci_hi"]),
            ],
            "best_variant": str(best_row["variant"]),
            "best_delta_ci": [
                float(best_row["delta_ci_lo"]),
                float(best_row["delta_ci_hi"]),
            ],
            "verdict_primary": "PASS" if H1_PASS_PRIMARY else "FAIL",
            "verdict_best":    "PASS" if H1_PASS_BEST    else "FAIL",
            "note": ("20-class macro-F1 penalises every false zero_day_unknown "
                     "alarm. See binary metrics for operational view."),
        },
        "H2": {
            "description": (
                f"AE recall on E7-misclassified samples >= {H2_RECALL_TARGET} "
                f"on at least {int(H2_FRACTION_TARGET * 100)}% of zero-day "
                f"targets (best threshold per target; targets with "
                f"n_called_benign < {H2_MIN_SAMPLES} flagged insufficient)"
            ),
            "primary_metric": "ae_recall_on_missed",
            "per_target_primary": H2_per_target_primary,
            "n_pass_primary": int(n_pass_primary),
            "n_total": int(n_total),
            "verdict": "PASS" if H2_PASS else "FAIL",
            "auxiliary_metric": "ae_recall (raw per-class)",
            "per_target_auxiliary": H2_per_target_aux,
            "n_pass_auxiliary": int(n_pass_aux),
        },
        "recommended_threshold": recommended,
    }
    with open(OUTPUT_DIR / "metrics" / "h1_h2_verdicts.json", "w") as f:
        json.dump(verdicts, f, indent=2)

    # ---- [3.12] summary.md ------------------------------------------------
    banner("[3.12] Writing summary.md")
    with open(OUTPUT_DIR / "summary.md", "w") as f:
        f.write("# Phase 6 — Fusion Engine & Zero-Day Simulation Summary\n\n")
        f.write(f"_Generated {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} "
                "(v2 — post senior review)_\n\n")

        f.write("## 1. Configuration\n\n")
        f.write(f"- Primary variant: **{PRIMARY}**\n")
        for pct, t in THRESHOLDS.items():
            f.write(f"- AE threshold p{pct}: `{t:.4f}`\n")
        f.write(f"- Benign class id: `{BENIGN_ID}` "
                f"| Zero-day pseudo-class id: `{ZERO_DAY_ID}`\n")
        f.write(f"- Test samples: `{n_test:,}` | Val samples: `{n_val:,}`\n")
        f.write(f"- Bootstrap iterations: {H1_BOOTSTRAP_ITERS} "
                f"(seed={H1_BOOTSTRAP_SEED})\n")
        f.write(f"- H2 primary metric: AE recall on samples E7 misclassified "
                f"as benign\n")
        f.write(f"- H2 sample-size guard: n_called_benign >= "
                f"{H2_MIN_SAMPLES}\n\n")

        f.write("## 2. Case Distribution (test set)\n\n")
        f.write(md_table(case_dist_df, floatfmt=".2f"))
        f.write("\n\n")

        f.write("## 3. Fusion vs E7 — 20-class macro-F1 with bootstrap CI\n\n")
        f.write(f"E7 baseline macro-F1 (19-class): **{e7_macro_f1:.4f}** "
                f"[{e7_f1_lo:.4f}, {e7_f1_hi:.4f}] "
                f"| MCC: {e7_mcc:.4f} | acc: {e7_acc:.4f}\n\n")
        f.write(md_table(mc_macro_df, floatfmt=".4f"))
        f.write("\n\n")

        f.write("## 4. Binary Detection (Cases 1+2+3 vs Case 4)\n\n")
        f.write(md_table(bin_metric_df, floatfmt=".4f"))
        f.write("\n\n")

        f.write("## 5. Simulated Zero-Day under E7-Blindness\n\n")
        f.write("> **Methodological note.** This is *not* leave-one-attack-out "
                "in the strict sense — E7 is trained on all 19 classes, "
                "including the 5 targets. The simulation measures: when E7 "
                "misclassifies a target attack as benign, does the AE catch "
                "it? True LOO would require retraining E7 five times "
                "(deferred to future work).\n\n")
        cols = ["target", "n_test", "e7_recall",
                "e7_called_benign_n", "e7_called_benign_pct",
                "h2_sample_sufficient",
                "ae_recall_on_missed_p90", "ae_recall_on_missed_p95",
                "ae_recall_on_missed_p99",
                "ae_recall_p90", "ae_recall_p95", "ae_recall_p99",
                "binary_detected_recall_p90",
                "confirmed_or_zeroday_p90"]
        f.write(md_table(zd_df[cols], floatfmt=".3f"))
        f.write("\n\n")

        f.write("## 6. Hypothesis Verdicts\n\n")
        f.write("### H1 — Fusion improves macro-F1 (paired bootstrap)\n\n")
        f.write(f"- E7 baseline: {e7_macro_f1:.4f} "
                f"[{e7_f1_lo:.4f}, {e7_f1_hi:.4f}]\n")
        f.write(f"- Fusion ({PRIMARY}): {float(primary_row['macro_f1']):.4f} "
                f"[{float(primary_row['macro_f1_ci_lo']):.4f}, "
                f"{float(primary_row['macro_f1_ci_hi']):.4f}]\n")
        f.write(f"- Δ = {float(primary_row['delta_f1_vs_E7']):+.4f} "
                f"95% CI [{float(primary_row['delta_ci_lo']):+.4f}, "
                f"{float(primary_row['delta_ci_hi']):+.4f}]\n")
        f.write(f"- Best variant ({best_row['variant']}): "
                f"Δ CI [{float(best_row['delta_ci_lo']):+.4f}, "
                f"{float(best_row['delta_ci_hi']):+.4f}]\n")
        f.write(f"- **Verdict (primary): "
                f"{'PASS — Δ CI excludes 0 ✓' if H1_PASS_PRIMARY else 'FAIL — Δ CI spans 0 ✗'}**\n")
        f.write(f"- **Verdict (best variant): "
                f"{'PASS ✓' if H1_PASS_BEST else 'FAIL ✗'}**\n\n")
        f.write("> 20-class macro-F1 penalises every false `zero_day_unknown` "
                "alarm equally. Binary detection (§4) is more representative "
                "of operational value.\n\n")

        f.write(f"### H2 — AE catches what E7 misses on "
                f"≥{int(H2_FRACTION_TARGET*100)}% of zero-day targets\n\n")
        f.write("**Primary metric: AE recall on samples E7 misclassified "
                "as benign.**\n\n")
        f.write(f"- Targets passing (best threshold, AE-on-missed ≥ "
                f"{H2_RECALL_TARGET}): **{n_pass_primary}/{n_total}**\n")
        for tgt, d in H2_per_target_primary.items():
            if not d["sufficient"]:
                f.write(f"  - ⚠ {tgt}: insufficient samples "
                        f"(n_called_benign < {H2_MIN_SAMPLES}); excluded\n")
            else:
                mark = "✓" if d["passes"] else "✗"
                f.write(f"  - {mark} {tgt}: best AE-on-missed = "
                        f"{d['value']:.3f}\n")
        f.write(f"- **Verdict: {'PASS ✓' if H2_PASS else 'FAIL ✗'}**\n\n")
        f.write("**Auxiliary (raw AE per-class recall, Phase-5 framing):** "
                f"{n_pass_aux}/{n_total} pass.\n\n")

        f.write("## 7. Threshold Sensitivity (val for selection, test for reporting)\n\n")
        f.write(md_table(sweep_df, floatfmt=".4f"))
        f.write("\n\n")

        f.write("## 8. Recommended Operating Threshold\n\n")
        if recommended is not None:
            f.write(f"Selected on val (FPR < {RECOMMENDED_FPR_BUDGET}): "
                    f"**p{recommended['percentile']}** "
                    f"(threshold = {recommended['threshold']:.4f})\n\n")
            f.write(f"- Val:  TPR = {recommended['val_TPR']:.4f}, "
                    f"FPR = {recommended['val_FPR']:.4f}\n")
            f.write(f"- Test: TPR = {recommended['test_TPR']:.4f}, "
                    f"FPR = {recommended['test_FPR']:.4f}, "
                    f"binary F1 = {recommended['test_F1']:.4f}\n\n")
        else:
            f.write(f"No val percentile achieves FPR < {RECOMMENDED_FPR_BUDGET}. "
                    "Trade-off discussion required in thesis Chapter 5.\n\n")

        f.write("## 9. Per-class case rates (primary variant)\n\n")
        f.write(md_table(per_class_df.reset_index().rename(
            columns={"index": "class"}), floatfmt=".2f"))
        f.write("\n\n")

        f.write("## 10. Files generated\n\n")
        f.write("- `fusion_results/fusion_{val,test}_cases.npy` — case arrays\n")
        f.write("- `fusion_results/fusion_{val,test}_labels.csv` — decoded "
                "(redundant w/ npy + dict, kept for inspection)\n")
        f.write("- `metrics/case_distribution.csv`\n")
        f.write("- `metrics/fusion_vs_supervised.csv` "
                "(macro-F1 + bootstrap CIs)\n")
        f.write("- `metrics/fusion_vs_supervised_binary.csv`\n")
        f.write("- `metrics/per_class_case_analysis.csv`\n")
        f.write("- `metrics/zero_day_results.csv`\n")
        f.write("- `metrics/threshold_sensitivity.csv` (val + test)\n")
        f.write("- `metrics/h1_h2_verdicts.json`\n")
        f.write("- `figures/*.png` (5 plots)\n")
        f.write("- `config.json`\n")

    print(f"\nDone. Output: {OUTPUT_DIR}")
    print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


if __name__ == "__main__":
    main()